In [1]:
# ============================================================
# ZYRA V1 — COMPLETE CORE RECOMMENDATION PIPELINE
# ============================================================
#
# INPUT:
#   - Product catalog CSV
#   - User prescription JSON
#   - Existing User/Product Encoder outputs can be plugged in
#
# PIPELINE:
#   1. Load + validate catalog
#   2. Normalize product metadata
#   3. Parse product categories / attributes
#   4. User prescription
#   5. Gender + occasion candidate filtering
#   6. User ↔ Product compatibility scoring
#   7. Candidate ranking
#   8. Diversity optimization
#   9. Outfit composition
#  10. Recommendation explanations
#
# NOTE:
#   This is the V1 BASELINE.
#   We intentionally do NOT train a supervised model here yet,
#   because the catalog contains no user-product interaction labels.
# ============================================================

import os
import re
import json
import math
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. CONFIGURATION
# ------------------------------------------------------------

DATASET_PATH = "products.csv"   # <-- CHANGE THIS ONLY

TOP_CANDIDATES = 500
FINAL_PRODUCTS = 100
FINAL_OUTFITS = 10

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("=" * 70)
print("ZYRA V1 INITIALIZATION")
print("=" * 70)


# ------------------------------------------------------------
# 1. LOAD PRODUCT DATASET
# ------------------------------------------------------------

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(
        f"\nDataset not found: {DATASET_PATH}\n"
        "Change DATASET_PATH at the top of this notebook."
    )

df = pd.read_csv(DATASET_PATH)

print("\nDataset loaded successfully")
print("Shape:", df.shape)

required_columns = [
    "name",
    "sku",
    "mpn",
    "price",
    "in_stock",
    "currency",
    "brand",
    "description",
    "images",
    "gender"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Required columns: OK")


# ------------------------------------------------------------
# 2. BASIC DATA VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA VALIDATION")
print("=" * 70)

print("\nMissing values:")
print(df[required_columns].isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSKU duplicates:", df["sku"].duplicated().sum())

print("\nMPN duplicates:", df["mpn"].duplicated().sum())

print("\nGender distribution:")
print(df["gender"].value_counts())


# ------------------------------------------------------------
# 3. NORMALIZATION
# ------------------------------------------------------------

TEXT_COLUMNS = [
    "name",
    "brand",
    "description",
    "gender"
]

for col in TEXT_COLUMNS:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

df["gender_norm"] = (
    df["gender"]
    .str.lower()
    .str.strip()
)

df["name_norm"] = (
    df["name"]
    .str.lower()
)

df["description_norm"] = (
    df["description"]
    .str.lower()
)

df["brand_norm"] = (
    df["brand"]
    .str.lower()
)

# Price normalization
df["price"] = pd.to_numeric(
    df["price"],
    errors="coerce"
)

# Availability normalization
df["in_stock_norm"] = (
    df["in_stock"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)


# ------------------------------------------------------------
# 4. PRODUCT ATTRIBUTE EXTRACTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXTRACTING PRODUCT ATTRIBUTES")
print("=" * 70)


def combined_text(row):
    return (
        f"{row['name']} "
        f"{row['brand']} "
        f"{row['description']}"
    ).lower()


df["search_text"] = df.apply(
    combined_text,
    axis=1
)


# ------------------------------------------------------------
# CATEGORY DETECTION
# ------------------------------------------------------------

CATEGORY_KEYWORDS = {
    "shirt": [
        "shirt",
        "formal shirt",
        "casual shirt",
        "oxford shirt",
        "polo shirt"
    ],

    "tshirt": [
        "t-shirt",
        "tshirt",
        "t shirt"
    ],

    "trousers": [
        "trouser",
        "trousers",
        "pants",
        "formal pants"
    ],

    "jeans": [
        "jeans",
        "denim"
    ],

    "shorts": [
        "shorts"
    ],

    "blazer": [
        "blazer",
        "sport coat"
    ],

    "suit": [
        "suit",
        "suit set",
        "suit jacket"
    ],

    "jacket": [
        "jacket",
        "bomber",
        "leather jacket",
        "denim jacket"
    ],

    "kurta": [
        "kurta",
        "kurti",
        "kurta set"
    ],

    "saree": [
        "saree",
        "sari"
    ],

    "dress": [
        "dress",
        "gown",
        "maxi dress"
    ],

    "skirt": [
        "skirt"
    ],

    "leggings": [
        "leggings"
    ],

    "ethnic_set": [
        "ethnic set",
        "ethnic wear",
        "salwar",
        "churidar",
        "palazzo"
    ],

    "shoes": [
        "shoes",
        "shoe",
        "sneakers",
        "loafers",
        "boots",
        "heels",
        "sandals",
        "flats",
        "moccasins"
    ],

    "bag": [
        "bag",
        "backpack",
        "trolley",
        "handbag",
        "clutch",
        "wallet"
    ],

    "accessory": [
        "watch",
        "belt",
        "sunglasses",
        "scarf",
        "jewellery",
        "jewelry",
        "bracelet",
        "necklace"
    ],

    "innerwear": [
        "bra",
        "brief",
        "boxer",
        "innerwear"
    ],

    "fragrance": [
        "perfume",
        "fragrance",
        "eau de toilette",
        "deodorant"
    ],

    "home": [
        "lamp",
        "placemat",
        "cushion",
        "home decor",
        "decor"
    ]
}


def detect_category(text):
    text = text.lower()

    # More specific categories first
    priority = [
        "innerwear",
        "fragrance",
        "tshirt",
        "trousers",
        "ethnic_set",
        "accessory",
        "blazer",
        "suit",
        "jacket",
        "shirt",
        "jeans",
        "shorts",
        "kurta",
        "saree",
        "dress",
        "skirt",
        "leggings",
        "shoes",
        "bag",
        "home"
    ]

    for category in priority:
        for keyword in CATEGORY_KEYWORDS[category]:
            if keyword in text:
                return category

    return "other"


df["category"] = df["search_text"].apply(
    detect_category
)

print("\nCategory distribution:")
print(df["category"].value_counts().head(30))


# ------------------------------------------------------------
# 5. COLOR EXTRACTION
# ------------------------------------------------------------

COLOR_KEYWORDS = [
    "black",
    "white",
    "grey",
    "gray",
    "charcoal",
    "navy",
    "blue",
    "royal blue",
    "sky blue",
    "red",
    "maroon",
    "burgundy",
    "wine",
    "pink",
    "peach",
    "orange",
    "yellow",
    "mustard",
    "green",
    "olive",
    "emerald",
    "teal",
    "brown",
    "camel",
    "beige",
    "cream",
    "ivory",
    "purple",
    "lavender",
    "magenta",
    "gold",
    "silver"
]


def extract_colors(text):
    text = text.lower()

    found = []

    for color in COLOR_KEYWORDS:
        if color in text:
            found.append(color)

    return list(dict.fromkeys(found))


df["colors"] = df["search_text"].apply(
    extract_colors
)


# ------------------------------------------------------------
# 6. STYLE ATTRIBUTE EXTRACTION
# ------------------------------------------------------------

STYLE_KEYWORDS = {
    "minimalist": [
        "minimal",
        "minimalist",
        "solid",
        "clean look"
    ],

    "formal": [
        "formal",
        "office",
        "business",
        "bandhgala",
        "blazer",
        "suit"
    ],

    "casual": [
        "casual",
        "everyday",
        "relaxed"
    ],

    "streetwear": [
        "street",
        "oversized",
        "graphic",
        "cargo"
    ],

    "traditional": [
        "ethnic",
        "traditional",
        "kurta",
        "saree",
        "churidar",
        "salwar"
    ],

    "luxury": [
        "premium",
        "luxury",
        "silk",
        "wool",
        "designer"
    ],

    "sporty": [
        "sport",
        "sports",
        "athletic",
        "gym",
        "training"
    ],

    "party": [
        "party",
        "sequin",
        "shimmer",
        "metallic"
    ]
}


def extract_styles(text):
    text = text.lower()

    found = []

    for style, keywords in STYLE_KEYWORDS.items():
        for keyword in keywords:
            if keyword in text:
                found.append(style)
                break

    return list(dict.fromkeys(found))


df["styles"] = df["search_text"].apply(
    extract_styles
)


# ------------------------------------------------------------
# 7. OCCASION ATTRIBUTE EXTRACTION
# ------------------------------------------------------------

OCCASION_KEYWORDS = {
    "wedding": [
        "wedding",
        "wedding wear",
        "wedding guest",
        "ceremony",
        "bandhgala",
        "ethnic"
    ],

    "party": [
        "party",
        "party wear",
        "club",
        "night out",
        "sequin",
        "shimmer"
    ],

    "formal": [
        "formal",
        "office",
        "business",
        "workwear",
        "corporate"
    ],

    "casual": [
        "casual",
        "everyday",
        "daily wear"
    ],

    "festive": [
        "festive",
        "festival",
        "ethnic"
    ],

    "date": [
        "date",
        "evening",
        "romantic"
    ],

    "travel": [
        "travel",
        "holiday",
        "vacation"
    ],

    "sports": [
        "sports",
        "gym",
        "running",
        "training"
    ]
}


def extract_occasions(text):
    text = text.lower()

    found = []

    for occasion, keywords in OCCASION_KEYWORDS.items():
        for keyword in keywords:
            if keyword in text:
                found.append(occasion)
                break

    return list(dict.fromkeys(found))


df["occasions"] = df["search_text"].apply(
    extract_occasions
)


# ------------------------------------------------------------
# 8. USER PRESCRIPTION
# ------------------------------------------------------------

USER = {
    "userId": "U-ZERA-8941",

    "occasion": "wedding",

    "gender": "male",

    "limit": 10,

    "forceRefresh": False,

    "userProfile": {

        "fashionIdentity": {
            "primaryStyle": "Minimalist Luxury",
            "styleArchetype": "Minimalist",
            "gender": "male",
            "ageGroup": "adult",
            "confidence": 0.94
        },

        "colorInsights": {
            "melaninUndertone": "Warm Olive",

            "seasonalPalette": "Deep Autumn",

            "recommendedColors": [
                "Washed Black",
                "Deep Navy",
                "Burgundy",
                "Charcoal Grey",
                "Emerald Green",
                "Camel"
            ],

            "avoidColors": [
                "Neon Yellow",
                "Bright Orange",
                "Pastel Lavender"
            ],

            "contrastPreference": "High Contrast"
        },

        "fitInsights": {
            "bodyType": "Trapezoid / Athletic",

            "preferredFit": "Tailored Slim",

            "heightCm": 182.0,

            "weightKg": 76.5,

            "chestCm": 102.0,

            "waistCm": 82.0,

            "shoulderCm": 46.5,

            "inseamCm": 81.0,

            "torsoToLegRatio": 0.95
        },

        "styleInsights": {

            "formalityAffinity": 0.85,

            "favoredSilhouettes": [
                "Structured Blazer",
                "Bandhgala",
                "Tapered Chino",
                "Double Breasted Jacket"
            ],

            "preferredFabrics": [
                "Linen",
                "Merino Wool",
                "Mulberry Silk",
                "Structured Cotton"
            ]
        },

        "behaviourInsights": {

            "budgetTier": "Premium",

            "brandAffinities": [
                "LUXZERA Classic",
                "Zera Minimal"
            ]
        }
    }
}


# ------------------------------------------------------------
# 9. NORMALIZE USER PRESCRIPTION
# ------------------------------------------------------------

profile = USER["userProfile"]

user_gender = USER["gender"].lower()

occasion = USER["occasion"].lower()

primary_style = (
    profile["fashionIdentity"]["primaryStyle"]
    .lower()
)

style_archetype = (
    profile["fashionIdentity"]["styleArchetype"]
    .lower()
)

recommended_colors = [
    x.lower()
    for x in profile["colorInsights"]["recommendedColors"]
]

avoid_colors = [
    x.lower()
    for x in profile["colorInsights"]["avoidColors"]
]

preferred_fit = (
    profile["fitInsights"]["preferredFit"]
    .lower()
)

favored_silhouettes = [
    x.lower()
    for x in profile["styleInsights"]["favoredSilhouettes"]
]

preferred_fabrics = [
    x.lower()
    for x in profile["styleInsights"]["preferredFabrics"]
]

formality_affinity = float(
    profile["styleInsights"]["formalityAffinity"]
)


# ------------------------------------------------------------
# 10. GENDER COMPATIBILITY
# ------------------------------------------------------------

def gender_score(product_gender, user_gender):

    pg = str(product_gender).lower()

    if user_gender == "male":

        if pg == "men":
            return 1.0

        if pg == "unisex":
            return 0.85

        if pg in ["boys", "unisex kids"]:
            return 0.0

        if pg == "women":
            return 0.0

        if pg == "girls":
            return 0.0

    if user_gender == "female":

        if pg == "women":
            return 1.0

        if pg == "unisex":
            return 0.85

        if pg in ["girls", "unisex kids"]:
            return 0.0

        if pg == "men":
            return 0.0

        if pg == "boys":
            return 0.0

    return 0.5


# ------------------------------------------------------------
# 11. COLOR COMPATIBILITY
# ------------------------------------------------------------

def color_score(product_colors):

    if not product_colors:
        return 0.50

    product_colors = [
        x.lower()
        for x in product_colors
    ]

    # Avoid colors = strong negative
    for color in product_colors:

        for bad in avoid_colors:

            # Flexible matching
            if (
                bad in color
                or color in bad
            ):
                return 0.05

    matches = 0

    for color in product_colors:

        for good in recommended_colors:

            if (
                good in color
                or color in good
            ):
                matches += 1

    if matches > 0:
        return min(
            1.0,
            0.70 + (0.10 * matches)
        )

    return 0.50


# ------------------------------------------------------------
# 12. STYLE COMPATIBILITY
# ------------------------------------------------------------

def style_score(row):

    text = row["search_text"]

    score = 0.50

    # Primary style
    if "minimal" in primary_style:

        if (
            "solid" in text
            or "minimal" in text
            or "clean" in text
        ):
            score += 0.20

    # Luxury
    if "luxury" in primary_style:

        luxury_words = [
            "premium",
            "luxury",
            "silk",
            "wool",
            "designer",
            "structured"
        ]

        if any(
            word in text
            for word in luxury_words
        ):
            score += 0.20

    # Silhouettes
    for silhouette in favored_silhouettes:

        key_words = silhouette.split()

        if all(
            word in text
            for word in key_words
            if len(word) > 3
        ):
            score += 0.15
            break

    return min(score, 1.0)


# ------------------------------------------------------------
# 13. FORMALITY SCORE
# ------------------------------------------------------------

FORMAL_WORDS = [
    "formal",
    "blazer",
    "suit",
    "bandhgala",
    "trouser",
    "formal shirt",
    "dress shirt",
    "loafer",
    "structured"
]

CASUAL_WORDS = [
    "casual",
    "shorts",
    "graphic",
    "oversized",
    "flip flop",
    "jogger"
]


def formality_score(row):

    text = row["search_text"]

    formal_hits = sum(
        1
        for word in FORMAL_WORDS
        if word in text
    )

    casual_hits = sum(
        1
        for word in CASUAL_WORDS
        if word in text
    )

    raw = 0.50

    raw += formal_hits * 0.08
    raw -= casual_hits * 0.08

    raw = max(
        0.0,
        min(1.0, raw)
    )

    # Match user's affinity
    return 1.0 - abs(
        raw - formality_affinity
    )


# ------------------------------------------------------------
# 14. OCCASION SCORE
# ------------------------------------------------------------

def occasion_score(row, occasion):

    text = row["search_text"]

    product_occasions = row["occasions"]

    # Direct metadata match
    if occasion in product_occasions:
        return 1.0

    # Wedding heuristics
    if occasion == "wedding":

        wedding_words = [
            "bandhgala",
            "blazer",
            "suit",
            "kurta",
            "ethnic",
            "formal",
            "silk",
            "wedding"
        ]

        hits = sum(
            word in text
            for word in wedding_words
        )

        if hits >= 3:
            return 0.90

        if hits == 2:
            return 0.80

        if hits == 1:
            return 0.60

    # Party
    if occasion == "party":

        party_words = [
            "party",
            "sequin",
            "shimmer",
            "metallic",
            "evening",
            "night"
        ]

        hits = sum(
            word in text
            for word in party_words
        )

        return min(
            1.0,
            0.50 + hits * 0.15
        )

    # Formal
    if occasion == "formal":

        if any(
            word in text
            for word in FORMAL_WORDS
        ):
            return 0.90

    # Casual
    if occasion == "casual":

        if any(
            word in text
            for word in CASUAL_WORDS
        ):
            return 0.90

    return 0.50


# ------------------------------------------------------------
# 15. FIT COMPATIBILITY
# ------------------------------------------------------------

def fit_score(row):

    text = row["search_text"]

    score = 0.50

    if "tailored" in preferred_fit:

        if (
            "slim fit" in text
            or "tailored fit" in text
            or "structured" in text
        ):
            score += 0.30

        if "regular fit" in text:
            score += 0.05

        if "oversized" in text:
            score -= 0.20

        if "loose fit" in text:
            score -= 0.15

    return max(
        0.0,
        min(1.0, score)
    )


# ------------------------------------------------------------
# 16. FABRIC COMPATIBILITY
# ------------------------------------------------------------

def fabric_score(row):

    text = row["search_text"]

    if not preferred_fabrics:
        return 0.50

    hits = 0

    for fabric in preferred_fabrics:

        fabric_words = fabric.split()

        if any(
            word in text
            for word in fabric_words
            if len(word) > 3
        ):
            hits += 1

    if hits == 0:
        return 0.50

    return min(
        1.0,
        0.65 + hits * 0.10
    )


# ------------------------------------------------------------
# 17. PREMIUM / BRAND SCORE
# ------------------------------------------------------------

def brand_score(row):

    brand = row["brand_norm"]

    affinities = [
        x.lower()
        for x in profile[
            "behaviourInsights"
        ]["brandAffinities"]
    ]

    if brand in affinities:
        return 1.0

    # Since real catalog brands are different,
    # do not heavily penalize unknown brands.
    return 0.50


# ------------------------------------------------------------
# 18. AVAILABILITY SCORE
# ------------------------------------------------------------

def availability_score(row):

    return 1.0 if row["in_stock_norm"] else 0.0


# ------------------------------------------------------------
# 19. COMPATIBILITY MODEL — V1 BASELINE
# ------------------------------------------------------------

def calculate_compatibility(row):

    gender = gender_score(
        row["gender"],
        user_gender
    )

    color = color_score(
        row["colors"]
    )

    style = style_score(
        row
    )

    occasion_s = occasion_score(
        row,
        occasion
    )

    fit = fit_score(
        row
    )

    formality = formality_score(
        row
    )

    fabric = fabric_score(
        row
    )

    brand = brand_score(
        row
    )

    availability = availability_score(
        row
    )

    # --------------------------------------------------------
    # HARD GENDER PROTECTION
    # --------------------------------------------------------

    if gender <= 0:
        return 0.0, {
            "gender": gender,
            "color": color,
            "style": style,
            "occasion": occasion_s,
            "fit": fit,
            "formality": formality,
            "fabric": fabric,
            "brand": brand,
            "availability": availability
        }

    # --------------------------------------------------------
    # WEIGHTED V1 SCORE
    # --------------------------------------------------------

    score = (

        gender * 0.20

        + occasion_s * 0.18

        + color * 0.17

        + style * 0.15

        + fit * 0.10

        + formality * 0.10

        + fabric * 0.05

        + brand * 0.02

        + availability * 0.03

    )

    return float(score), {

        "gender": gender,

        "color": color,

        "style": style,

        "occasion": occasion_s,

        "fit": fit,

        "formality": formality,

        "fabric": fabric,

        "brand": brand,

        "availability": availability
    }


# ------------------------------------------------------------
# 20. CANDIDATE FILTERING
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CANDIDATE GENERATION")
print("=" * 70)

candidate_df = df.copy()

# Gender filtering
if user_gender == "male":

    candidate_df = candidate_df[
        candidate_df["gender_norm"].isin(
            ["men", "unisex"]
        )
    ]

elif user_gender == "female":

    candidate_df = candidate_df[
        candidate_df["gender_norm"].isin(
            ["women", "unisex"]
        )
    ]

# Availability
candidate_df = candidate_df[
    candidate_df["in_stock_norm"] == True
]

print(
    "Candidates after gender + availability:",
    len(candidate_df)
)


# ------------------------------------------------------------
# 21. SCORE ALL CANDIDATES
# ------------------------------------------------------------

print("\nScoring candidates...")

scores = []
components = []

for _, row in candidate_df.iterrows():

    score, breakdown = calculate_compatibility(
        row
    )

    scores.append(score)
    components.append(breakdown)


candidate_df = candidate_df.copy()

candidate_df["compatibility_score"] = scores

candidate_df["score_components"] = components


# ------------------------------------------------------------
# 22. RANK
# ------------------------------------------------------------

candidate_df = candidate_df.sort_values(
    "compatibility_score",
    ascending=False
).reset_index(drop=True)


candidate_df = candidate_df.head(
    TOP_CANDIDATES
)


print(
    f"\nTop {TOP_CANDIDATES} candidates selected."
)

print("\nTop 20:")
display(
    candidate_df[
        [
            "name",
            "brand",
            "gender",
            "category",
            "price",
            "compatibility_score"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 23. DIVERSITY ENGINE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DIVERSITY OPTIMIZATION")
print("=" * 70)


def text_similarity(row_a, row_b):

    words_a = set(
        re.findall(
            r"[a-z]+",
            row_a["search_text"]
        )
    )

    words_b = set(
        re.findall(
            r"[a-z]+",
            row_b["search_text"]
        )
    )

    if not words_a or not words_b:
        return 0.0

    intersection = len(
        words_a.intersection(words_b)
    )

    union = len(
        words_a.union(words_b)
    )

    if union == 0:
        return 0.0

    return intersection / union


def diversify(
    candidates,
    limit=100,
    diversity_strength=0.35
):

    if len(candidates) <= limit:
        return candidates.copy()

    selected = []

    remaining = list(
        candidates.index
    )

    # First = highest relevance
    first = candidates.index[0]

    selected.append(first)

    remaining.remove(first)

    while (
        remaining
        and len(selected) < limit
    ):

        best_idx = None
        best_value = -float("inf")

        for idx in remaining:

            relevance = candidates.loc[
                idx,
                "compatibility_score"
            ]

            max_similarity = 0.0

            for selected_idx in selected:

                sim = text_similarity(
                    candidates.loc[idx],
                    candidates.loc[selected_idx]
                )

                max_similarity = max(
                    max_similarity,
                    sim
                )

            # MMR-style objective
            value = (
                (1 - diversity_strength)
                * relevance
                -
                diversity_strength
                * max_similarity
            )

            if value > best_value:

                best_value = value
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return candidates.loc[
        selected
    ].reset_index(drop=True)


diverse_products = diversify(
    candidate_df,
    limit=FINAL_PRODUCTS,
    diversity_strength=0.30
)

print(
    "Diverse recommendations:",
    len(diverse_products)
)

display(
    diverse_products[
        [
            "name",
            "brand",
            "category",
            "price",
            "gender",
            "compatibility_score"
        ]
    ].head(30)
)


# ------------------------------------------------------------
# 24. OUTFIT COMPOSITION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("OUTFIT COMPOSITION")
print("=" * 70)


TOP_CATEGORIES = [
    "shirt",
    "tshirt",
    "kurta",
    "dress",
    "saree",
    "ethnic_set"
]

BOTTOM_CATEGORIES = [
    "trousers",
    "jeans",
    "shorts",
    "skirt",
    "leggings"
]

LAYER_CATEGORIES = [
    "blazer",
    "jacket",
    "suit"
]

SHOE_CATEGORIES = [
    "shoes"
]


def category_products(
    products,
    categories
):

    return products[
        products["category"].isin(
            categories
        )
    ]


tops = category_products(
    diverse_products,
    TOP_CATEGORIES
)

bottoms = category_products(
    diverse_products,
    BOTTOM_CATEGORIES
)

layers = category_products(
    diverse_products,
    LAYER_CATEGORIES
)

shoes = category_products(
    diverse_products,
    SHOE_CATEGORIES
)


print("Tops:", len(tops))
print("Bottoms:", len(bottoms))
print("Layers:", len(layers))
print("Shoes:", len(shoes))


# ------------------------------------------------------------
# OUTFIT COLOR COMPATIBILITY
# ------------------------------------------------------------

def outfit_color_score(items):

    colors = []

    for _, item in items:

        colors.extend(
            item["colors"]
        )

    colors = list(
        set(colors)
    )

    if not colors:
        return 0.50

    good = 0
    bad = 0

    for color in colors:

        for recommended in recommended_colors:

            if (
                recommended in color
                or color in recommended
            ):
                good += 1

        for avoided in avoid_colors:

            if (
                avoided in color
                or color in avoided
            ):
                bad += 1

    score = (
        0.50
        + good * 0.10
        - bad * 0.30
    )

    return max(
        0.0,
        min(1.0, score)
    )


# ------------------------------------------------------------
# OUTFIT SCORE
# ------------------------------------------------------------

def outfit_score(items):

    if not items:
        return 0.0

    product_scores = [
        item["compatibility_score"]
        for _, item in items
    ]

    product_score = np.mean(
        product_scores
    )

    color_score_value = outfit_color_score(
        items
    )

    # Formality coherence
    formality_values = []

    for _, item in items:

        breakdown = item[
            "score_components"
        ]

        formality_values.append(
            breakdown["formality"]
        )

    if formality_values:

        formality_coherence = (
            1.0
            - np.std(formality_values)
        )

    else:

        formality_coherence = 0.5

    return float(
        0.60 * product_score
        + 0.25 * color_score_value
        + 0.15 * formality_coherence
    )


# ------------------------------------------------------------
# GENERATE OUTFITS
# ------------------------------------------------------------

outfits = []

# Limit combinations so notebook remains fast
top_pool = tops.head(10)
bottom_pool = bottoms.head(10)
layer_pool = layers.head(8)
shoe_pool = shoes.head(8)

# Complete outfits
for _, top in top_pool.iterrows():

    for _, bottom in bottom_pool.iterrows():

        base_items = [
            ("top", top),
            ("bottom", bottom)
        ]

        # Add shoes if available
        if len(shoe_pool) > 0:

            for _, shoe in shoe_pool.head(5).iterrows():

                items = base_items + [
                    ("shoes", shoe)
                ]

                score = outfit_score(
                    items
                )

                outfits.append({
                    "items": items,
                    "score": score
                })

        else:

            score = outfit_score(
                base_items
            )

            outfits.append({
                "items": base_items,
                "score": score
            })


# Add formal layer outfits
for _, layer in layer_pool.iterrows():

    for _, top in top_pool.head(5).iterrows():

        for _, bottom in bottom_pool.head(5).iterrows():

            items = [
                ("layer", layer),
                ("top", top),
                ("bottom", bottom)
            ]

            score = outfit_score(
                items
            )

            outfits.append({
                "items": items,
                "score": score
            })


# ------------------------------------------------------------
# SORT OUTFITS
# ------------------------------------------------------------

outfits = sorted(
    outfits,
    key=lambda x: x["score"],
    reverse=True
)


# ------------------------------------------------------------
# REMOVE DUPLICATE OUTFITS
# ------------------------------------------------------------

unique_outfits = []

seen = set()

for outfit in outfits:

    sku_tuple = tuple(
        sorted(
            item["sku"]
            for _, item in outfit["items"]
        )
    )

    if sku_tuple in seen:
        continue

    seen.add(
        sku_tuple
    )

    unique_outfits.append(
        outfit
    )

    if len(unique_outfits) >= FINAL_OUTFITS:
        break


# ------------------------------------------------------------
# 25. EXPLANATION ENGINE
# ------------------------------------------------------------

def explain_product(row):

    breakdown = row[
        "score_components"
    ]

    reasons = []

    if breakdown["gender"] >= 0.85:
        reasons.append(
            "matches the user's gender profile"
        )

    if breakdown["occasion"] >= 0.80:
        reasons.append(
            f"is well suited for a {occasion} occasion"
        )

    if breakdown["color"] >= 0.75:
        reasons.append(
            "works well with the user's recommended color palette"
        )

    if breakdown["style"] >= 0.75:
        reasons.append(
            "aligns with the user's minimalist/luxury style"
        )

    if breakdown["fit"] >= 0.70:
        reasons.append(
            "supports the user's tailored fit preference"
        )

    if breakdown["formality"] >= 0.75:
        reasons.append(
            "matches the user's preferred level of formality"
        )

    if not reasons:
        reasons.append(
            "has reasonable overall compatibility with the user's profile"
        )

    return (
        "This product suits you because "
        + ", ".join(reasons)
        + "."
    )


# ------------------------------------------------------------
# 26. DISPLAY FINAL PRODUCT RECOMMENDATIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL PRODUCT RECOMMENDATIONS")
print("=" * 70)

final_products = diverse_products.head(
    USER["limit"]
)

for i, (_, row) in enumerate(
    final_products.iterrows(),
    start=1
):

    print("\n" + "-" * 70)

    print(
        f"{i}. {row['name']}"
    )

    print(
        f"Brand: {row['brand']}"
    )

    print(
        f"Category: {row['category']}"
    )

    print(
        f"Price: {row['price']}"
    )

    print(
        f"Gender: {row['gender']}"
    )

    print(
        f"Compatibility: "
        f"{row['compatibility_score']:.3f}"
    )

    print(
        "Why:",
        explain_product(row)
    )


# ------------------------------------------------------------
# 27. DISPLAY FINAL OUTFITS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ZYRA V1 — FINAL OUTFITS")
print("=" * 70)


for i, outfit in enumerate(
    unique_outfits,
    start=1
):

    print("\n" + "=" * 70)

    print(
        f"OUTFIT {i}"
    )

    print(
        f"Outfit Score: "
        f"{outfit['score']:.3f}"
    )

    for role, item in outfit["items"]:

        print(
            f"  {role.upper():10} → "
            f"{item['name']}"
        )

    # Explanation
    reasons = []

    if occasion == "wedding":
        reasons.append(
            "appropriate for a wedding setting"
        )

    if any(
        item["score_components"]["color"] >= 0.75
        for _, item in outfit["items"]
    ):
        reasons.append(
            "uses colors compatible with the user's Deep Autumn palette"
        )

    if any(
        item["score_components"]["style"] >= 0.75
        for _, item in outfit["items"]
    ):
        reasons.append(
            "matches the user's minimalist-luxury style"
        )

    if any(
        item["score_components"]["fit"] >= 0.70
        for _, item in outfit["items"]
    ):
        reasons.append(
            "supports the user's tailored preference"
        )

    print(
        "  WHY → "
        + ", ".join(reasons)
        + "."
    )


# ------------------------------------------------------------
# 28. SAVE V1 RESULTS
# ------------------------------------------------------------

final_product_output = final_products[
    [
        "name",
        "sku",
        "brand",
        "price",
        "gender",
        "category",
        "compatibility_score"
    ]
].copy()

final_product_output.to_csv(
    "zyra_v1_product_recommendations.csv",
    index=False
)


outfit_rows = []

for i, outfit in enumerate(
    unique_outfits,
    start=1
):

    row = {
        "outfit_id": i,
        "outfit_score": outfit["score"]
    }

    for role, item in outfit["items"]:

        row[
            f"{role}_name"
        ] = item["name"]

        row[
            f"{role}_sku"
        ] = item["sku"]

    outfit_rows.append(
        row
    )


outfit_output = pd.DataFrame(
    outfit_rows
)

outfit_output.to_csv(
    "zyra_v1_outfits.csv",
    index=False
)


# ------------------------------------------------------------
# 29. FINAL SYSTEM SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ZYRA V1 COMPLETE")
print("=" * 70)

print(
    f"""
User:
    Gender       : {user_gender}
    Occasion     : {occasion}
    Style        : {primary_style}
    Fit          : {preferred_fit}
    Palette      : Deep Autumn

Catalog:
    Products     : {len(df)}
    Candidates   : {len(candidate_df)}

Recommendation:
    Products     : {len(final_products)}
    Outfits      : {len(unique_outfits)}

Pipeline:
    ✓ Product validation
    ✓ Metadata normalization
    ✓ Category extraction
    ✓ Color extraction
    ✓ Style extraction
    ✓ Occasion extraction
    ✓ Gender filtering
    ✓ Compatibility scoring
    ✓ Candidate ranking
    ✓ Diversity optimization
    ✓ Outfit composition
    ✓ Recommendation explanations

Saved:
    zyra_v1_product_recommendations.csv
    zyra_v1_outfits.csv
"""
)

print("\nNEXT STEP:")
print(
    "After verifying these recommendations, "
    "we will plug your EXISTING User Encoder "
    "and Product Encoder embeddings into the "
    "compatibility layer and build the actual "
    "training dataset/model."
)

ZYRA V1 INITIALIZATION


FileNotFoundError: 
Dataset not found: products.csv
Change DATASET_PATH at the top of this notebook.

In [2]:
DATASET_PATH = "products.csv"

In [4]:
import os
from pathlib import Path

print("Current notebook directory:")
print(os.getcwd())

print("\nSearching for CSV files...")

# Search current directory + subdirectories
csv_files = list(Path(".").rglob("*.csv"))

if not csv_files:
    print("\n❌ No CSV found.")
    print("\nFiles/folders in current directory:")
    for item in Path(".").iterdir():
        print(" -", item)
    
    raise FileNotFoundError(
        "\nYour dataset is not inside this notebook's accessible directory. "
        "You need to put/copy the CSV into this Jupyter environment."
    )

print(f"\n✅ Found {len(csv_files)} CSV file(s):")

for i, file in enumerate(csv_files):
    print(f"{i}: {file}")

# Show sizes so we can identify the 12,491-product dataset
print("\nCSV sizes:")

for file in csv_files:
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"{file} → {size_mb:.2f} MB")

Current notebook directory:
/Users/saketh/Desktop/Projects/weavly/core-model

Searching for CSV files...

✅ Found 40 CSV file(s):
0: .venv/lib/python3.13/site-packages/scipy/signal/tests/data/GLB.Ts+dSST.csv
1: .venv/lib/python3.13/site-packages/matplotlib/mpl-data/sample_data/msft.csv
2: .venv/lib/python3.13/site-packages/matplotlib/mpl-data/sample_data/data_x_x2_x3.csv
3: .venv/lib/python3.13/site-packages/matplotlib/mpl-data/sample_data/Stocks.csv
4: .venv/lib/python3.13/site-packages/sklearn/datasets/data/wine_data.csv
5: .venv/lib/python3.13/site-packages/sklearn/datasets/data/iris.csv
6: .venv/lib/python3.13/site-packages/sklearn/datasets/data/breast_cancer.csv
7: .venv/lib/python3.13/site-packages/sklearn/datasets/data/linnerud_physiological.csv
8: .venv/lib/python3.13/site-packages/sklearn/datasets/data/linnerud_exercise.csv
9: .venv/lib/python3.13/site-packages/tornado/test/csv_translations/fr_FR.csv
10: .venv/lib/python3.13/site-packages/numpy/random/tests/data/philox-testset

In [8]:
from pathlib import Path

search_locations = [
    Path.home() / "Desktop",
    Path.home() / "Documents",
    Path.home() / "Downloads",
]

csv_files = []

for location in search_locations:
    if location.exists():
        for file in location.rglob("*.csv"):
            if ".venv" not in file.parts:
                csv_files.append(file)

print(f"Found {len(csv_files)} CSV files:\n")

for i, file in enumerate(csv_files):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"[{i}] {file}")
    print(f"    Size: {size_mb:.2f} MB")

Found 35 CSV files:

[0] /Users/saketh/Desktop/Projects/weavly/data/raw/myntra-products-dataset/myntra202305041052.csv
    Size: 1365.61 MB
[1] /Users/saketh/Desktop/Projects/weavly/data/raw/myntra-fashion-products/Myntra_fashion_products.csv
    Size: 12.14 MB
[2] /Users/saketh/Desktop/Projects/Zera-Search/catalog_vector_service/venv/lib/python3.14/site-packages/matplotlib/mpl-data/sample_data/msft.csv
    Size: 0.00 MB
[3] /Users/saketh/Desktop/Projects/Zera-Search/catalog_vector_service/venv/lib/python3.14/site-packages/matplotlib/mpl-data/sample_data/data_x_x2_x3.csv
    Size: 0.00 MB
[4] /Users/saketh/Desktop/Projects/Zera-Search/catalog_vector_service/venv/lib/python3.14/site-packages/matplotlib/mpl-data/sample_data/Stocks.csv
    Size: 0.06 MB
[5] /Users/saketh/Desktop/Projects/Zera-Search/catalog_vector_service/venv/lib/python3.14/site-packages/numpy/random/tests/data/philox-testset-1.csv
    Size: 0.02 MB
[6] /Users/saketh/Desktop/Projects/Zera-Search/catalog_vector_service/ve

In [9]:
import pandas as pd
from pathlib import Path

matches = []

for file in csv_files:
    try:
        df_test = pd.read_csv(file)

        if df_test.shape == (12491, 10):
            matches.append(file)

    except Exception:
        pass

print("Datasets matching (12491, 10):")

for i, file in enumerate(matches):
    print(f"[{i}] {file}")

Datasets matching (12491, 10):
[0] /Users/saketh/Desktop/Projects/weavly/data/raw/myntra-fashion-products/Myntra_fashion_products.csv


In [10]:
DATASET_PATH = str(matches[0])

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully")
print("Path:", DATASET_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset loaded successfully
Path: /Users/saketh/Desktop/Projects/weavly/data/raw/myntra-fashion-products/Myntra_fashion_products.csv
Shape: (12491, 10)
Columns: ['name', 'sku', 'mpn', 'price', 'in_stock', 'currency', 'brand', 'description', 'images', 'gender']


In [11]:
# ============================================================
# ZYRA V1 — DATASET VALIDATION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 60)
print("ZYRA V1 — DATASET VALIDATION")
print("=" * 60)

# ------------------------------------------------------------
# Expected schema
# ------------------------------------------------------------

EXPECTED_COLUMNS = [
    "name",
    "sku",
    "mpn",
    "price",
    "in_stock",
    "currency",
    "brand",
    "description",
    "images",
    "gender"
]

# Check columns
missing_columns = [
    col for col in EXPECTED_COLUMNS
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in EXPECTED_COLUMNS
]

print("\nSchema check")

if missing_columns:
    print("❌ Missing columns:", missing_columns)
else:
    print("✅ All required columns present")

if extra_columns:
    print("⚠️ Extra columns:", extra_columns)
else:
    print("✅ No unexpected columns")

# ------------------------------------------------------------
# Shape
# ------------------------------------------------------------

print("\nDataset shape:")
print(df.shape)

assert df.shape[0] == 12491, \
    f"Expected 12491 products, got {df.shape[0]}"

# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

print("\nMissing values:")
print(df.isnull().sum())

assert df.isnull().sum().sum() == 0, \
    "Dataset contains missing values"

# ------------------------------------------------------------
# Duplicate products
# ------------------------------------------------------------

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate SKU:")
print(df["sku"].duplicated().sum())

print("\nDuplicate MPN:")
print(df["mpn"].duplicated().sum())

# ------------------------------------------------------------
# Gender
# ------------------------------------------------------------

print("\nGender distribution:")
print(df["gender"].value_counts())

# ------------------------------------------------------------
# Data types
# ------------------------------------------------------------

print("\nData types:")
print(df.dtypes)

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET VALIDATION COMPLETE")
print("=" * 60)

ZYRA V1 — DATASET VALIDATION

Schema check
✅ All required columns present
✅ No unexpected columns

Dataset shape:
(12491, 10)

Missing values:
name           0
sku            0
mpn            0
price          0
in_stock       0
currency       0
brand          0
description    0
images         0
gender         0
dtype: int64

Duplicate rows:
0

Duplicate SKU:
0

Duplicate MPN:
0

Gender distribution:
gender
Women          5126
Men            4591
Unisex         1188
Boys           1100
Girls           440
Unisex Kids      46
Name: count, dtype: int64

Data types:
name             str
sku            int64
mpn            int64
price          int64
in_stock        bool
currency         str
brand            str
description      str
images           str
gender           str
dtype: object

DATASET VALIDATION COMPLETE


# Product Feature Extraction


In [12]:
# ============================================================
# ZYRA V1 — PRODUCT FEATURE EXTRACTION
# ============================================================

import re
import numpy as np
import pandas as pd

products = df.copy()

# ------------------------------------------------------------
# 1. Normalize text
# ------------------------------------------------------------

def clean_text(value):
    value = str(value).lower()
    value = value.replace("&", " and ")
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


products["name_clean"] = products["name"].apply(clean_text)
products["description_clean"] = products["description"].apply(clean_text)
products["brand_clean"] = products["brand"].apply(clean_text)

# Combined semantic text
products["product_text"] = (
    products["name_clean"] + " " +
    products["name_clean"] + " " +
    products["description_clean"] + " " +
    products["brand_clean"]
).str.strip()


# ------------------------------------------------------------
# 2. Normalize gender
# ------------------------------------------------------------

gender_map = {
    "Men": "men",
    "Women": "women",
    "Unisex": "unisex",
    "Boys": "boys",
    "Girls": "girls",
    "Unisex Kids": "unisex_kids"
}

products["gender_normalized"] = (
    products["gender"]
    .map(gender_map)
    .fillna(products["gender"].str.lower().str.strip())
)


# ------------------------------------------------------------
# 3. Price normalization
# ------------------------------------------------------------

products["price"] = pd.to_numeric(
    products["price"],
    errors="coerce"
)

products["price_log"] = np.log1p(products["price"])


# ------------------------------------------------------------
# 4. Image URL processing
# ------------------------------------------------------------

def split_images(value):
    if pd.isna(value):
        return []

    return [
        url.strip()
        for url in str(value).split("~")
        if url.strip()
    ]


products["image_urls"] = products["images"].apply(split_images)

products["image_count"] = products["image_urls"].apply(len)


# ------------------------------------------------------------
# 5. Product identity
# ------------------------------------------------------------

products["product_id"] = products["sku"].astype(str)

products["mpn"] = products["mpn"].astype(str)


# ------------------------------------------------------------
# 6. Basic text statistics
# ------------------------------------------------------------

products["name_word_count"] = (
    products["name_clean"]
    .str.split()
    .str.len()
)

products["description_word_count"] = (
    products["description_clean"]
    .str.split()
    .str.len()
)


# ------------------------------------------------------------
# 7. Detect useful fashion attributes from text
# ------------------------------------------------------------

ATTRIBUTE_PATTERNS = {

    "black": r"\bblack\b",
    "white": r"\bwhite\b",
    "grey": r"\bgrey\b|\bgray\b",
    "navy": r"\bnavy\b",
    "blue": r"\bblue\b",
    "red": r"\bred\b",
    "green": r"\bgreen\b",
    "yellow": r"\byellow\b",
    "orange": r"\borange\b",
    "pink": r"\bpink\b",
    "purple": r"\bpurple\b",
    "brown": r"\bbrown\b",
    "beige": r"\bbeige\b",
    "maroon": r"\bmaroon\b",
    "burgundy": r"\bburgundy\b",
    "camel": r"\bcamel\b",

    "slim_fit": r"\bslim\s*fit\b|\bslim\b",
    "regular_fit": r"\bregular\s*fit\b",
    "oversized": r"\boversized\b",
    "skinny_fit": r"\bskinny\s*fit\b|\bskinny\b",
    "tapered_fit": r"\btapered\s*fit\b|\btapered\b",
    "relaxed_fit": r"\brelaxed\s*fit\b",

    "printed": r"\bprinted\b|\bprint\b",
    "solid": r"\bsolid\b",
    "checked": r"\bchecked\b|\bcheck\b",
    "striped": r"\bstriped\b|\bstripe\b",
    "floral": r"\bfloral\b",
    "embroidered": r"\bembroidered\b|\bembroidery\b",
    "self_design": r"\bself[\s-]?design\b",

    "formal": r"\bformal\b",
    "casual": r"\bcasual\b",
    "party": r"\bparty\b",
    "wedding": r"\bwedding\b",
    "sports": r"\bsports?\b",
    "running": r"\brunning\b",
    "travel": r"\btravel\b",

    "shirt": r"\bshirt\b",
    "tshirt": r"\bt[\s-]?shirt\b|\btee\b",
    "trousers": r"\btrouser\b|\btrousers\b",
    "jeans": r"\bjeans\b",
    "shorts": r"\bshorts\b",
    "blazer": r"\bblazer\b",
    "suit": r"\bsuit\b",
    "kurta": r"\bkurta\b",
    "dress": r"\bdress\b",
    "skirt": r"\bskirt\b",
    "jacket": r"\bjacket\b",
    "coat": r"\bcoat\b",
    "bag": r"\bbag\b",
    "shoes": r"\bshoe\b|\bshoes\b",
    "sandal": r"\bsandal\b",
    "watch": r"\bwatch\b",
    "belt": r"\bbelt\b",
    "wallet": r"\bwallet\b",
    "perfume": r"\bperfume\b|\beau de toilette\b"
}


# ------------------------------------------------------------
# Create binary fashion attributes
# ------------------------------------------------------------

for attribute, pattern in ATTRIBUTE_PATTERNS.items():

    products[f"attr_{attribute}"] = (
        products["product_text"]
        .str.contains(
            pattern,
            regex=True,
            case=False,
            na=False
        )
        .astype(np.int8)
    )


# ------------------------------------------------------------
# 8. Attribute summary
# ------------------------------------------------------------

attribute_columns = [
    col for col in products.columns
    if col.startswith("attr_")
]

attribute_summary = (
    products[attribute_columns]
    .sum()
    .sort_values(ascending=False)
)

print("=" * 60)
print("ZYRA V1 — PRODUCT FEATURE EXTRACTION")
print("=" * 60)

print("\nProducts:", len(products))

print("\nGender:")
print(products["gender_normalized"].value_counts())

print("\nImage count:")
print(products["image_count"].describe())

print("\nDetected attributes:")
print(attribute_summary)

print("\nSample product representation:")
print(
    products[
        [
            "product_id",
            "name",
            "brand",
            "gender_normalized",
            "price",
            "image_count"
        ]
        + attribute_columns[:10]
    ].head(5).to_string(index=False)
)

print("\nFeature columns created:", len(products.columns))

print("\n✅ PRODUCT FEATURE EXTRACTION COMPLETE")

ZYRA V1 — PRODUCT FEATURE EXTRACTION

Products: 12491

Gender:
gender_normalized
women          5126
men            4591
unisex         1188
boys           1100
girls           440
unisex_kids      46
Name: count, dtype: int64

Image count:
count    12491.000000
mean         4.913698
std          1.092333
min          1.000000
25%          5.000000
50%          5.000000
75%          5.000000
max         10.000000
Name: image_count, dtype: float64

Detected attributes:
attr_solid          3785
attr_blue           3680
attr_shirt          3203
attr_printed        3108
attr_black          2254
attr_white          1965
attr_slim_fit       1674
attr_navy           1500
attr_tshirt         1465
attr_casual         1396
attr_grey           1149
attr_jeans          1103
attr_green          1083
attr_regular_fit    1005
attr_red             959
attr_kurta           909
attr_belt            876
attr_checked         834
attr_pink            755
attr_brown           746
attr_striped         655
at

# Inspect Your Product Encoder

In [13]:
# ============================================================
# ZYRA V1 — PRODUCT ENCODER INSPECTION
# ============================================================

print("=" * 60)
print("ZYRA V1 — PRODUCT ENCODER INSPECTION")
print("=" * 60)

# Show variables currently available in notebook
print("\nAvailable variables:\n")

for name in sorted(globals().keys()):
    if not name.startswith("_"):
        obj = globals()[name]

        # Ignore huge dataframe dumps
        if isinstance(obj, pd.DataFrame):
            print(f"{name} → DataFrame {obj.shape}")
        elif isinstance(obj, np.ndarray):
            print(f"{name} → NumPy array {obj.shape}")
        elif isinstance(obj, list):
            print(f"{name} → list ({len(obj)} items)")
        elif isinstance(obj, dict):
            print(f"{name} → dict ({len(obj)} keys)")
        else:
            print(f"{name} → {type(obj).__name__}")

print("\n" + "=" * 60)

ZYRA V1 — PRODUCT ENCODER INSPECTION

Available variables:

ATTRIBUTE_PATTERNS → dict (55 keys)
Counter → type
DATASET_PATH → str
EXPECTED_COLUMNS → list (10 items)
FINAL_OUTFITS → int
FINAL_PRODUCTS → int
In → list (14 items)
Out → dict (0 keys)
Path → type
RANDOM_SEED → int
TOP_CANDIDATES → int
attribute → str
attribute_columns → list (55 items)
attribute_summary → Series
clean_text → function
csv_files → list (35 items)
df → DataFrame (12491, 10)
df_test → DataFrame (411, 4)
exit → ZMQExitAutocall
extra_columns → list (0 items)
file → PosixPath
gender_map → dict (6 keys)
get_ipython → method
glob → module
i → int
json → module
location → PosixPath
matches → list (1 items)
math → module
missing_columns → list (0 items)
np → module
open → function
os → module
p → PosixPath
pattern → str
pd → module
products → DataFrame (12491, 76)
project_root → PosixPath
quit → ZMQExitAutocall
re → module
search_locations → list (3 items)
size_mb → float
split_images → function
warnings → module



In [14]:
# ============================================================
# FIND EXISTING ZYRA / PRODUCT ENCODER FILES
# ============================================================

from pathlib import Path

ROOT = Path("/Users/saketh/Desktop/Projects/weavly")

print("=" * 70)
print("SEARCHING FOR EXISTING PRODUCT ENCODER")
print("=" * 70)

# Search likely model/code files
patterns = [
    "*.py",
    "*.ipynb",
    "*.pkl",
    "*.joblib",
    "*.pt",
    "*.pth",
    "*.onnx",
    "*.bin",
    "*.safetensors",
    "*.npy",
    "*.npz"
]

matches = []

for pattern in patterns:
    for p in ROOT.rglob(pattern):
        # Ignore virtual environments and caches
        if any(x in p.parts for x in [
            ".venv",
            "node_modules",
            "__pycache__",
            ".git"
        ]):
            continue

        matches.append(p)

# Remove duplicates
matches = sorted(set(matches))

keywords = [
    "product",
    "encoder",
    "embedding",
    "clip",
    "model",
    "zyra"
]

likely = []

for p in matches:
    name = p.name.lower()

    if any(k in name for k in keywords):
        likely.append(p)

print(f"\nFound {len(matches)} candidate files total.")
print(f"Likely relevant files: {len(likely)}\n")

for i, p in enumerate(likely):
    size_mb = p.stat().st_size / (1024 * 1024)

    print(f"[{i}] {p}")
    print(f"    Type : {p.suffix}")
    print(f"    Size : {size_mb:.2f} MB")
    print()

print("=" * 70)
print("SEARCH COMPLETE")
print("=" * 70)

SEARCHING FOR EXISTING PRODUCT ENCODER

Found 655 candidate files total.
Likely relevant files: 46

[0] /Users/saketh/Desktop/Projects/weavly/core-model/.env/lib/python3.13/site-packages/pip/_vendor/pygments/modeline.py
    Type : .py
    Size : 0.00 MB

[1] /Users/saketh/Desktop/Projects/weavly/core-model/.env/lib/python3.13/site-packages/pip/_vendor/requests/models.py
    Type : .py
    Size : 0.03 MB

[2] /Users/saketh/Desktop/Projects/weavly/core-model/.ipynb_checkpoints/Zyra-v1-checkpoint.ipynb
    Type : .ipynb
    Size : 0.00 MB

[3] /Users/saketh/Desktop/Projects/weavly/core-model/.ipynb_checkpoints/Zyrav2-checkpoint.ipynb
    Type : .ipynb
    Size : 0.04 MB

[4] /Users/saketh/Desktop/Projects/weavly/core-model/Zyra-v1.ipynb
    Type : .ipynb
    Size : 0.10 MB

[5] /Users/saketh/Desktop/Projects/weavly/core-model/Zyrav2.ipynb
    Type : .ipynb
    Size : 0.04 MB

[6] /Users/saketh/Desktop/Projects/weavly/core-model/scripts/download_models.py
    Type : .py
    Size : 0.00 MB


In [15]:
# ============================================================
# ZYRA V1 — SECTION 5
# PRODUCT CATALOG + EMBEDDING MATRIX
# ============================================================

import os
import json
import pickle
import numpy as np
import pandas as pd

print("=" * 70)
print("ZYRA V1 — PRODUCT CATALOG BUILD")
print("=" * 70)

# ------------------------------------------------------------
# 1. VERIFY INPUT
# ------------------------------------------------------------

assert "products" in globals(), \
    "❌ 'products' DataFrame not found. Run Product Feature Extraction first."

print(f"\nInput products: {len(products):,}")
print(f"Input features: {products.shape[1]}")

# ------------------------------------------------------------
# 2. IDENTIFY PRODUCT ID
# ------------------------------------------------------------

if "product_id" in products.columns:
    PRODUCT_ID_COLUMN = "product_id"
elif "sku" in products.columns:
    PRODUCT_ID_COLUMN = "sku"
else:
    raise ValueError("❌ No product_id or sku column found.")

print(f"Product ID column: {PRODUCT_ID_COLUMN}")

# ------------------------------------------------------------
# 3. CREATE RECOMMENDATION CATALOG
# ------------------------------------------------------------

catalog = products.copy()

# Standardize product ID
catalog["product_id"] = catalog[PRODUCT_ID_COLUMN].astype(str)

# ------------------------------------------------------------
# 4. STANDARDIZE GENDER
# ------------------------------------------------------------

if "gender_normalized" in catalog.columns:
    catalog["gender_normalized"] = (
        catalog["gender_normalized"]
        .astype(str)
        .str.lower()
        .str.strip()
    )
else:
    catalog["gender_normalized"] = (
        catalog["gender"]
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )

# ------------------------------------------------------------
# 5. BASIC RECOMMENDATION FEATURES
# ------------------------------------------------------------

# Keep useful original fields if available
required_base = [
    "product_id",
    "name",
    "brand",
    "price",
    "gender_normalized"
]

for col in required_base:
    if col not in catalog.columns:
        catalog[col] = None

# ------------------------------------------------------------
# 6. ATTRIBUTE FEATURE LIST
# ------------------------------------------------------------

attribute_columns = [
    col for col in catalog.columns
    if col.startswith("attr_")
]

print(f"\nAttribute features: {len(attribute_columns)}")

# ------------------------------------------------------------
# 7. CREATE COMPACT ATTRIBUTE REPRESENTATION
# ------------------------------------------------------------

def extract_attributes(row):
    attrs = []

    for col in attribute_columns:
        try:
            value = row[col]

            if value == 1 or value is True:
                attrs.append(col.replace("attr_", ""))
        except Exception:
            pass

    return attrs


catalog["attributes"] = catalog.apply(
    extract_attributes,
    axis=1
)

# ------------------------------------------------------------
# 8. IMAGE COUNT
# ------------------------------------------------------------

if "image_count" not in catalog.columns:

    if "images" in catalog.columns:

        def count_images(value):
            if pd.isna(value):
                return 0

            return len([
                x for x in str(value).split("~")
                if x.strip()
            ])

        catalog["image_count"] = catalog["images"].apply(
            count_images
        )

    else:
        catalog["image_count"] = 0

# ------------------------------------------------------------
# 9. TEXT REPRESENTATION
# ------------------------------------------------------------

text_columns = [
    col for col in [
        "name",
        "brand",
        "description"
    ]
    if col in catalog.columns
]

def build_text(row):

    parts = []

    for col in text_columns:

        value = row.get(col)

        if pd.notna(value):

            text = str(value).strip()

            if text:
                parts.append(text)

    # Add detected fashion attributes
    attrs = row.get("attributes", [])

    if attrs:
        parts.append(
            " ".join(attrs)
        )

    return " ".join(parts)


catalog["search_text"] = catalog.apply(
    build_text,
    axis=1
)

# ------------------------------------------------------------
# 10. REMOVE ACCIDENTAL DUPLICATE PRODUCT IDS
# ------------------------------------------------------------

duplicate_ids = catalog["product_id"].duplicated().sum()

print(f"\nDuplicate product IDs: {duplicate_ids}")

if duplicate_ids > 0:

    catalog = (
        catalog
        .drop_duplicates(
            subset=["product_id"],
            keep="first"
        )
        .reset_index(drop=True)
    )

# ------------------------------------------------------------
# 11. PRODUCT QUALITY SIGNAL
# ------------------------------------------------------------

# Simple V1 quality signal.
# This is NOT a learned score.

catalog["product_quality"] = (
    np.clip(
        catalog["image_count"].fillna(0) / 5.0,
        0,
        1
    )
)

# ------------------------------------------------------------
# 12. NORMALIZE PRICE
# ------------------------------------------------------------

catalog["price"] = pd.to_numeric(
    catalog["price"],
    errors="coerce"
).fillna(0)

# ------------------------------------------------------------
# 13. CREATE PRODUCT INDEX
# ------------------------------------------------------------

catalog = catalog.reset_index(drop=True)

catalog["catalog_index"] = np.arange(
    len(catalog)
)

# ------------------------------------------------------------
# 14. BUILD PRODUCT ID → INDEX MAP
# ------------------------------------------------------------

product_id_to_index = {
    product_id: index
    for index, product_id
    in enumerate(catalog["product_id"])
}

index_to_product_id = {
    index: product_id
    for product_id, index
    in product_id_to_index.items()
}

# ------------------------------------------------------------
# 15. CHECK EMBEDDING COLUMN
# ------------------------------------------------------------

embedding_columns = [
    col for col in catalog.columns
    if "embedding" in col.lower()
]

print("\nEmbedding-related columns:")

for col in embedding_columns:
    print("  ", col)

# ------------------------------------------------------------
# 16. FIND EXISTING EMBEDDING MATRIX
# ------------------------------------------------------------

embedding_matrix = None

# Case A: an existing numpy matrix already exists
possible_embedding_vars = [
    "product_embeddings",
    "product_embedding_matrix",
    "embeddings",
    "embedding_matrix"
]

for variable_name in possible_embedding_vars:

    if variable_name in globals():

        candidate = globals()[variable_name]

        if isinstance(candidate, np.ndarray):

            if candidate.ndim == 2:
                embedding_matrix = candidate
                print(
                    f"\n✅ Found existing embedding matrix: "
                    f"{variable_name}"
                )
                break

# ------------------------------------------------------------
# 17. CHECK EMBEDDING DIMENSION
# ------------------------------------------------------------

if embedding_matrix is not None:

    print(
        "Embedding shape:",
        embedding_matrix.shape
    )

    if embedding_matrix.shape[0] != len(catalog):

        print(
            "⚠️ Embedding rows do not match catalog rows."
        )

        embedding_matrix = None

# ------------------------------------------------------------
# 18. IF EMBEDDINGS ARE NOT YET AVAILABLE
# ------------------------------------------------------------

if embedding_matrix is None:

    print("\n⚠️ No aligned product embedding matrix found.")

    print(
        "This is expected if the Product Encoder has not yet "
        "generated the 662D vectors for all 12,491 products."
    )

    print(
        "\nThe catalog will still be created."
    )

else:

    # Convert to float32 for memory efficiency
    embedding_matrix = np.asarray(
        embedding_matrix,
        dtype=np.float32
    )

    # Normalize embeddings for cosine similarity
    norms = np.linalg.norm(
        embedding_matrix,
        axis=1,
        keepdims=True
    )

    norms = np.maximum(
        norms,
        1e-12
    )

    normalized_embedding_matrix = (
        embedding_matrix / norms
    )

    print(
        "Normalized embedding shape:",
        normalized_embedding_matrix.shape
    )

# ------------------------------------------------------------
# 19. FINAL CATALOG SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATALOG SUMMARY")
print("=" * 70)

print(
    f"Products:        {len(catalog):,}"
)

print(
    f"Attributes:      {len(attribute_columns)}"
)

print(
    f"Gender groups:   {catalog['gender_normalized'].nunique()}"
)

print(
    f"Average images:  {catalog['image_count'].mean():.2f}"
)

print(
    f"Price range:     ₹{catalog['price'].min():,.0f}"
    f" → ₹{catalog['price'].max():,.0f}"
)

print("\nGender distribution:")

print(
    catalog["gender_normalized"]
    .value_counts()
)

# ------------------------------------------------------------
# 20. DISPLAY SAMPLE
# ------------------------------------------------------------

display_columns = [
    "product_id",
    "name",
    "brand",
    "gender_normalized",
    "price",
    "image_count",
    "product_quality",
    "attributes"
]

display_columns = [
    col for col in display_columns
    if col in catalog.columns
]

print("\nSample products:")

display(
    catalog[display_columns].head(5)
)

# ------------------------------------------------------------
# 21. SAVE V1 CATALOG
# ------------------------------------------------------------

PROJECT_ROOT = "/Users/saketh/Desktop/Projects/weavly/core-model"

MODEL_DATA_DIR = os.path.join(
    PROJECT_ROOT,
    "data",
    "recommendation"
)

os.makedirs(
    MODEL_DATA_DIR,
    exist_ok=True
)

catalog_path = os.path.join(
    MODEL_DATA_DIR,
    "product_catalog.pkl"
)

index_path = os.path.join(
    MODEL_DATA_DIR,
    "product_index.json"
)

catalog.to_pickle(
    catalog_path
)

with open(index_path, "w") as f:
    json.dump(
        {
            "product_id_to_index":
                product_id_to_index,
            "index_to_product_id":
                {
                    str(k): v
                    for k, v
                    in index_to_product_id.items()
                }
        },
        f,
        indent=2
    )

# ------------------------------------------------------------
# 22. SAVE EMBEDDINGS IF AVAILABLE
# ------------------------------------------------------------

if embedding_matrix is not None:

    embedding_path = os.path.join(
        MODEL_DATA_DIR,
        "product_embeddings.npy"
    )

    normalized_path = os.path.join(
        MODEL_DATA_DIR,
        "product_embeddings_normalized.npy"
    )

    np.save(
        embedding_path,
        embedding_matrix
    )

    np.save(
        normalized_path,
        normalized_embedding_matrix
    )

    print(
        "\n✅ Embedding matrix saved:"
    )

    print(
        "   ",
        embedding_path
    )

# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ ZYRA V1 — PRODUCT CATALOG COMPLETE")
print("=" * 70)

print(
    "\nCatalog:",
    catalog_path
)

print(
    "Index:",
    index_path
)

if embedding_matrix is not None:

    print(
        "Embeddings:",
        embedding_matrix.shape
    )

else:

    print(
        "Embeddings: NOT YET GENERATED"
    )

print(
    "\nNext → Section 6: Product Embedding Retrieval"
)

ZYRA V1 — PRODUCT CATALOG BUILD

Input products: 12,491
Input features: 76
Product ID column: product_id

Attribute features: 55

Duplicate product IDs: 0

Embedding-related columns:

⚠️ No aligned product embedding matrix found.
This is expected if the Product Encoder has not yet generated the 662D vectors for all 12,491 products.

The catalog will still be created.

CATALOG SUMMARY
Products:        12,491
Attributes:      55
Gender groups:   6
Average images:  4.91
Price range:     ₹90 → ₹63,090

Gender distribution:
gender_normalized
women          5126
men            4591
unisex         1188
boys           1100
girls           440
unisex_kids      46
Name: count, dtype: int64

Sample products:


,product_id,name,brand,gender_normalized,price,image_count,product_quality,attributes
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,unisex,11745,7,1.0,"[black, grey, printed, bag]"
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,women,5810,7,1.0,"[grey, beige, printed, solid, kurta, jacket]"
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,women,899,7,1.0,"[pink, skinny_fit, jeans, belt]"
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,men,5599,5,1.0,"[blue, self_design, trousers, blazer, suit, belt]"
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,men,759,5,1.0,"[white, brown, slim_fit, printed, casual, shirt]"



✅ ZYRA V1 — PRODUCT CATALOG COMPLETE

Catalog: /Users/saketh/Desktop/Projects/weavly/core-model/data/recommendation/product_catalog.pkl
Index: /Users/saketh/Desktop/Projects/weavly/core-model/data/recommendation/product_index.json
Embeddings: NOT YET GENERATED

Next → Section 6: Product Embedding Retrieval


In [16]:
# ============================================================
# ZYRA V1 — SECTION 6A
# INSPECT EXISTING PRODUCT ENCODER
# ============================================================

import inspect
import importlib

print("=" * 70)
print("ZYRA V1 — EXISTING PRODUCT ENCODER INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# Encoder modules
# ------------------------------------------------------------

MODULES = {
    "fusion": "zyra.product_encoder.fusion.models",
    "insights": "zyra.product_encoder.insights.models",
    "text_encoder": "zyra.product_encoder.text_encoder.encoder",
    "image_encoder": "zyra.product_encoder.image_encoder.encoder",
    "attribute_encoder": "zyra.product_encoder.attribute_encoder.encoder",
}

for name, module_name in MODULES.items():

    print("\n" + "-" * 70)
    print(f"{name.upper()}")
    print("-" * 70)

    try:

        module = importlib.import_module(module_name)

        print("Module:", module_name)

        members = inspect.getmembers(module)

        public_members = [
            (member_name, member)
            for member_name, member in members
            if not member_name.startswith("_")
        ]

        for member_name, member in public_members:

            if inspect.isclass(member):

                print(f"\nCLASS: {member_name}")

                try:
                    print(
                        "Constructor:",
                        inspect.signature(member)
                    )
                except Exception:
                    pass

                try:
                    methods = inspect.getmembers(
                        member,
                        predicate=inspect.isfunction
                    )

                    for method_name, method in methods:

                        if not method_name.startswith("_"):

                            try:
                                sig = inspect.signature(method)
                            except Exception:
                                sig = "(signature unavailable)"

                            print(
                                f"  method: {method_name}{sig}"
                            )

                except Exception:
                    pass

            elif inspect.isfunction(member):

                try:
                    sig = inspect.signature(member)
                except Exception:
                    sig = "(signature unavailable)"

                print(
                    f"FUNCTION: {member_name}{sig}"
                )

    except Exception as e:

        print(
            f"❌ Could not import {module_name}"
        )

        print(
            "Error:",
            repr(e)
        )

# ------------------------------------------------------------
# Existing global variables
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CURRENT NOTEBOOK VARIABLES")
print("=" * 70)

interesting = [
    "catalog",
    "products",
    "df",
    "embedding_matrix",
    "normalized_embedding_matrix",
    "product_embeddings",
    "product_encoder",
    "encoder",
]

for name in interesting:

    if name in globals():

        value = globals()[name]

        print(
            f"{name}: "
            f"{type(value).__name__}"
        )

        if isinstance(value, np.ndarray):

            print(
                f"    shape={value.shape}, "
                f"dtype={value.dtype}"
            )

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — EXISTING PRODUCT ENCODER INSPECTION

----------------------------------------------------------------------
FUSION
----------------------------------------------------------------------
Module: zyra.product_encoder.fusion.models

CLASS: Any
Constructor: (*args, **kwargs)

CLASS: BaseModel
Constructor: (**data: 'Any') -> 'None'
  method: copy(self, *, include: 'AbstractSetIntStr | MappingIntStrAny | None' = None, exclude: 'AbstractSetIntStr | MappingIntStrAny | None' = None, update: 'Dict[str, Any] | None' = None, deep: 'bool' = False) -> 'Self'
  method: dict(self, *, include: 'IncEx | None' = None, exclude: 'IncEx | None' = None, by_alias: 'bool' = False, exclude_unset: 'bool' = False, exclude_defaults: 'bool' = False, exclude_none: 'bool' = False) -> 'Dict[str, Any]'
  method: json(self, *, include: 'IncEx | None' = None, exclude: 'IncEx | None' = None, by_alias: 'bool' = False, exclude_unset: 'bool' = False, exclude_defaults: 'bool' = False, exclude_none: 'bool' = False, e

In [17]:
# ============================================================
# ZYRA V1 — SECTION 6B
# SINGLE PRODUCT ENCODER TEST
# ============================================================

import inspect
import importlib

print("=" * 70)
print("ZYRA V1 — SINGLE PRODUCT ENCODER TEST")
print("=" * 70)

# ------------------------------------------------------------
# Inspect ingestion/router schemas
# ------------------------------------------------------------

router = importlib.import_module(
    "zyra.product_encoder.ingestion.router"
)

print("\nINGESTION / ROUTER CLASSES")
print("-" * 70)

for name, obj in inspect.getmembers(router):

    if inspect.isclass(obj) and (
        "Input" in name or
        "Product" in name
    ):
        try:
            print(f"{name}{inspect.signature(obj)}")
        except Exception:
            print(name)

# ------------------------------------------------------------
# Inspect fusion module source-level functions/classes
# ------------------------------------------------------------

fusion = importlib.import_module(
    "zyra.product_encoder.fusion.models"
)

print("\nFUSION PUBLIC MEMBERS")
print("-" * 70)

for name, obj in inspect.getmembers(fusion):

    if not name.startswith("_"):

        if inspect.isclass(obj):

            if obj.__module__ == fusion.__name__:
                print("CLASS:", name)

        elif inspect.isfunction(obj):

            print("FUNCTION:", name)

# ------------------------------------------------------------
# Inspect available Product Encoder package modules
# ------------------------------------------------------------

print("\nPRODUCT ENCODER PACKAGE")
print("-" * 70)

package = importlib.import_module(
    "zyra.product_encoder"
)

print("Package:", package)

# ------------------------------------------------------------
# Select one catalog product
# ------------------------------------------------------------

sample = catalog.iloc[0]

print("\nSAMPLE PRODUCT")
print("-" * 70)

for column in catalog.columns:

    value = sample[column]

    print(
        f"{column}: "
        f"{str(value)[:300]}"
    )

print("\n" + "=" * 70)
print("SINGLE PRODUCT INSPECTION READY")
print("=" * 70)

ZYRA V1 — SINGLE PRODUCT ENCODER TEST

INGESTION / ROUTER CLASSES
----------------------------------------------------------------------
ProductAttributeEncoderInput(*, productId: str, category: str, subcategory: Optional[str] = None, attributes: zyra.product_encoder.schemas.input_schemas.ProductAttributes = <factory>, sizeInfo: zyra.product_encoder.schemas.input_schemas.SizeInfo = <factory>, fitInformation: zyra.product_encoder.schemas.input_schemas.FitInformation = <factory>, occasions: List[str] = <factory>, styles: List[str] = <factory>, seasons: List[str] = <factory>, tags: List[str] = <factory>, rawAttributes: Dict[str, Any] = <factory>) -> None
ProductAttributes(*, color: Optional[str] = None, material: Optional[str] = None, fit: Optional[str] = None, silhouette: Optional[str] = None, pattern: Optional[str] = None, neckline: Optional[str] = None, sleeve: Optional[str] = None, length: Optional[str] = None, closure: Optional[str] = None, garmentDetails: List[str] = <factory>, care

In [18]:
# ============================================================
# ZYRA V1 — SECTION 6C
# REAL SINGLE-PRODUCT ENCODER TEST
# ============================================================

from zyra.product_encoder.ingestion.router import (
    ProductTextEncoderInput,
    ProductImageEncoderInput,
    ProductImageInput,
    ProductAttributeEncoderInput,
    ProductAttributes,
)

from zyra.product_encoder.text_encoder.encoder import ProductTextEncoder
from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder
from zyra.product_encoder.attribute_encoder.encoder import ProductAttributeEncoder

import traceback

print("=" * 70)
print("ZYRA V1 — REAL SINGLE PRODUCT ENCODER TEST")
print("=" * 70)

# ------------------------------------------------------------
# Select product
# ------------------------------------------------------------

p = catalog.iloc[0]

product_id = str(p["product_id"])
title = str(p["name"])
description = str(p["description"])
brand = str(p["brand"])

print("\nProduct:")
print(title)
print("Product ID:", product_id)
print("Brand:", brand)

# ------------------------------------------------------------
# Parse image URLs
# ------------------------------------------------------------

image_urls = p["image_urls"]

if isinstance(image_urls, str):
    image_urls = [x.strip() for x in image_urls.split("~") if x.strip()]

print("\nImages:", len(image_urls))

# ------------------------------------------------------------
# Build image inputs
# ------------------------------------------------------------

image_inputs = [
    ProductImageInput(
        imageId=f"{product_id}_{i}",
        imageUrl=url,
        viewType="front",
        sortOrder=i,
    )
    for i, url in enumerate(image_urls)
]

# ------------------------------------------------------------
# Build text input
# ------------------------------------------------------------

text_input = ProductTextEncoderInput(
    productId=product_id,
    title=title,
    description=description,
    brand=brand,
    category="unknown",
)

# ------------------------------------------------------------
# Build attribute input
# ------------------------------------------------------------

detected_attributes = p["attributes"]

attribute_dict = {}

for attr in detected_attributes:
    attribute_dict[attr] = True

attribute_input = ProductAttributeEncoderInput(
    productId=product_id,
    category="unknown",
    attributes=ProductAttributes(
        color=None,
        material=None,
        fit=None,
        silhouette=None,
        pattern=None,
    ),
    rawAttributes=attribute_dict,
    tags=list(detected_attributes),
)

# ============================================================
# TEXT ENCODER
# ============================================================

print("\n" + "=" * 70)
print("1. TEXT ENCODER")
print("=" * 70)

try:

    text_encoder = ProductTextEncoder()

    text_output = text_encoder.encode(text_input)

    print("✅ Text encoder completed")
    print("Embedding dimension:", text_output.embeddingDimension)
    print("Embedding length:",
          len(text_output.textEmbedding)
          if text_output.textEmbedding is not None
          else None)

    print("Confidence:", text_output.confidence)

except Exception as e:

    print("❌ TEXT ENCODER FAILED")
    print(type(e).__name__, ":", e)
    traceback.print_exc()

# ============================================================
# ATTRIBUTE ENCODER
# ============================================================

print("\n" + "=" * 70)
print("2. ATTRIBUTE ENCODER")
print("=" * 70)

try:

    attribute_encoder = ProductAttributeEncoder()

    attribute_output = attribute_encoder.encode(
        attribute_input
    )

    print("✅ Attribute encoder completed")
    print(
        "Embedding dimension:",
        attribute_output.embeddingDimension
    )

    print(
        "Embedding length:",
        len(attribute_output.attributeEmbedding)
        if attribute_output.attributeEmbedding is not None
        else None
    )

    print("Confidence:", attribute_output.confidence)

except Exception as e:

    print("❌ ATTRIBUTE ENCODER FAILED")
    print(type(e).__name__, ":", e)
    traceback.print_exc()

# ============================================================
# IMAGE ENCODER
# ============================================================

print("\n" + "=" * 70)
print("3. IMAGE ENCODER")
print("=" * 70)

try:

    image_input = ProductImageEncoderInput(
        productId=product_id,
        title=title,
        images=image_inputs,
    )

    image_encoder = ProductImageEncoder()

    image_output = image_encoder.encode(image_input)

    print("✅ Image encoder completed")

    print(
        "Embedding dimension:",
        image_output.embeddingDimension
    )

    print(
        "Aggregated embedding:",
        len(image_output.aggregatedEmbedding)
        if image_output.aggregatedEmbedding is not None
        else None
    )

    print(
        "Successful images:",
        image_output.successfulImageCount
    )

    print(
        "Failed images:",
        image_output.failedImageCount
    )

    print("Confidence:", image_output.confidence)

except Exception as e:

    print("❌ IMAGE ENCODER FAILED")
    print(type(e).__name__, ":", e)
    traceback.print_exc()

print("\n" + "=" * 70)
print("SINGLE MODALITY TEST COMPLETE")
print("=" * 70)

ZYRA V1 — REAL SINGLE PRODUCT ENCODER TEST

Product:
DKNY Unisex Black & Grey Printed Medium Trolley Bag
Product ID: 10017413
Brand: DKNY

Images: 7

1. TEXT ENCODER
✅ Text encoder completed
Embedding dimension: 512
Embedding length: 512
Confidence: 0.9

2. ATTRIBUTE ENCODER
✅ Attribute encoder completed
Embedding dimension: 128
Embedding length: 128
Confidence: 1.0

3. IMAGE ENCODER
✅ Image encoder completed
Embedding dimension: 512
Aggregated embedding: 512
Successful images: 7
Failed images: 0
Confidence: 1.0

SINGLE MODALITY TEST COMPLETE


In [19]:
# ============================================================
# ZYRA V1 — REAL PRODUCT FUSION TEST
# ============================================================

import numpy as np

print("=" * 70)
print("ZYRA V1 — PRODUCT FUSION TEST")
print("=" * 70)

# ------------------------------------------------------------
# Get the three modality embeddings from the previous test
# ------------------------------------------------------------

text_embedding = np.asarray(text_result.embedding, dtype=np.float32)
attribute_embedding = np.asarray(attribute_result.embedding, dtype=np.float32)
image_embedding = np.asarray(image_result.embedding, dtype=np.float32)

print("\nInput embeddings:")
print("Text       :", text_embedding.shape)
print("Attributes :", attribute_embedding.shape)
print("Image      :", image_embedding.shape)

# ------------------------------------------------------------
# Validate dimensions
# ------------------------------------------------------------

assert text_embedding.shape == (512,), \
    f"Unexpected text embedding shape: {text_embedding.shape}"

assert attribute_embedding.shape == (128,), \
    f"Unexpected attribute embedding shape: {attribute_embedding.shape}"

assert image_embedding.shape == (512,), \
    f"Unexpected image embedding shape: {image_embedding.shape}"

# ------------------------------------------------------------
# Inspect Fusion configuration
# ------------------------------------------------------------

print("\nFusion classes:")

print("FusionWeightsConfig")
print(FusionWeightsConfig)

print("\nModalityContribution")
print(ModalityContribution)

print("\nUnifiedProductRepresentation")
print(UnifiedProductRepresentation)

# ------------------------------------------------------------
# Try to construct the fusion representation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CREATING UNIFIED PRODUCT REPRESENTATION")
print("=" * 70)

fusion_success = False
unified = None

try:

    # Try the most direct construction based on the existing
    # Fusion model schema.

    unified = UnifiedProductRepresentation(
        productId=str(product["product_id"]),
        textEmbedding=text_embedding.tolist(),
        attributeEmbedding=attribute_embedding.tolist(),
        imageEmbedding=image_embedding.tolist()
    )

    fusion_success = True

except Exception as e:

    print("\n⚠️ Direct construction failed:")
    print(type(e).__name__, ":", e)

# ------------------------------------------------------------
# If schema requires another structure, inspect fields
# ------------------------------------------------------------

if not fusion_success:

    print("\nInspecting UnifiedProductRepresentation fields...")

    try:
        print(
            UnifiedProductRepresentation.model_fields
        )
    except Exception:
        try:
            print(
                UnifiedProductRepresentation.__fields__
            )
        except Exception as e:
            print("Could not inspect fields:", e)

# ------------------------------------------------------------
# Manual fallback fusion
# ------------------------------------------------------------

if not fusion_success:

    print("\nUsing deterministic V1 fusion fallback.")

    # Normalize each modality first
    def l2_normalize(x):
        norm = np.linalg.norm(x)

        if norm == 0:
            return x

        return x / norm

    text_norm = l2_normalize(text_embedding)
    attribute_norm = l2_normalize(attribute_embedding)
    image_norm = l2_normalize(image_embedding)

    # V1 modality weights
    TEXT_WEIGHT = 0.35
    ATTRIBUTE_WEIGHT = 0.15
    IMAGE_WEIGHT = 0.50

    fused = np.concatenate([
        text_norm * TEXT_WEIGHT,
        attribute_norm * ATTRIBUTE_WEIGHT,
        image_norm * IMAGE_WEIGHT
    ])

    fused = l2_normalize(fused)

    print("\nFusion weights:")
    print("Text       :", TEXT_WEIGHT)
    print("Attributes :", ATTRIBUTE_WEIGHT)
    print("Image      :", IMAGE_WEIGHT)

    print("\nUnified embedding:")
    print("Dimension :", len(fused))
    print("Norm      :", np.linalg.norm(fused))

    print("\nFirst 20 values:")
    print(fused[:20])

    print("\n" + "=" * 70)
    print("✅ PRODUCT FUSION TEST COMPLETE")
    print("=" * 70)

ZYRA V1 — PRODUCT FUSION TEST


NameError: name 'text_result' is not defined

In [20]:
# ============================================================
# ZYRA V1 — INSPECT ENCODER TEST VARIABLES
# ============================================================

print("Variables containing encoder/result/embedding:\n")

for name, value in globals().items():
    if any(x in name.lower() for x in [
        "text",
        "attribute",
        "image",
        "embedding",
        "result",
        "fusion"
    ]):
        if not name.startswith("_"):
            try:
                print(
                    f"{name:35} → "
                    f"{type(value).__name__}"
                )
            except:
                pass

Variables containing encoder/result/embedding:

clean_text                          → function
split_images                        → function
ATTRIBUTE_PATTERNS                  → dict
attribute                           → str
attribute_columns                   → list
attribute_summary                   → Series
extract_attributes                  → function
text_columns                        → list
build_text                          → function
embedding_columns                   → list
embedding_matrix                    → NoneType
possible_embedding_vars             → list
fusion                              → module
ProductTextEncoderInput             → ModelMetaclass
ProductImageEncoderInput            → ModelMetaclass
ProductImageInput                   → ModelMetaclass
ProductAttributeEncoderInput        → ModelMetaclass
ProductAttributes                   → ModelMetaclass
ProductTextEncoder                  → ABCMeta
ProductImageEncoder                 → ABCMeta
ProductAttrib

In [21]:
# ============================================================
# ZYRA V1 — INSPECT REAL ENCODER OUTPUTS
# ============================================================

print("=" * 70)
print("ZYRA V1 — REAL ENCODER OUTPUT INSPECTION")
print("=" * 70)

outputs = {
    "TEXT": text_output,
    "ATTRIBUTE": attribute_output,
    "IMAGE": image_output
}

for name, output in outputs.items():

    print("\n" + "-" * 70)
    print(f"{name} OUTPUT")
    print("-" * 70)

    print("Type:", type(output))
    print("\nObject:")

    try:
        print(output)
    except Exception as e:
        print("Could not print object:", e)

    print("\nAvailable fields:")

    try:
        if hasattr(output, "model_fields"):
            print(list(output.model_fields.keys()))
        elif hasattr(output, "__fields__"):
            print(list(output.__fields__.keys()))
        else:
            print([
                x for x in dir(output)
                if not x.startswith("_")
            ])
    except Exception as e:
        print("Could not inspect fields:", e)

    print("\nDictionary representation:")

    try:
        if hasattr(output, "model_dump"):
            print(output.model_dump())
        elif hasattr(output, "dict"):
            print(output.dict())
        elif hasattr(output, "__dict__"):
            print(output.__dict__)
    except Exception as e:
        print("Could not create dictionary:", e)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — REAL ENCODER OUTPUT INSPECTION

----------------------------------------------------------------------
TEXT OUTPUT
----------------------------------------------------------------------
Type: <class 'zyra.product_encoder.schemas.output_schemas.TextRepresentation'>

Object:
productId='10017413' textInsights=TextInsights(productMeaning='DKNY unknown', primaryStyle=None, secondaryStyles=[], intendedUse=[], extractedMaterials=[], fitDescriptor=None, targetSeasons=[], targetOccasions=[], semanticKeywords=['dkny', 'unisex', 'black', 'grey', 'printed', 'medium', 'trolley', 'bag', 'secured', 'tsa', 'lockone', 'handle', 'top', 'one', 'side'], detectedContradictions=[], fieldProvenance={'title': 'DKNY Unisex Black & Grey Printed Medium Trolley Bag', 'descriptionPresent': True, 'brand': 'DKNY', 'category': 'unknown', 'tagsCount': 0}) textEmbedding=[-0.021539820358157158, 0.014359880238771439, -0.046071283519268036, -0.014958208426833153, 0.05684119462966919, 0.0550462082028389, 0.043079

In [22]:
# ============================================================
# ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST
# ============================================================

import numpy as np
import pandas as pd
import traceback
from collections import Counter

print("=" * 70)
print("ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST")
print("=" * 70)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

TEST_SIZE = min(100, len(df))
RANDOM_SEED = 42

sample_df = df.sample(
    n=TEST_SIZE,
    random_state=RANDOM_SEED
).reset_index(drop=True)

print(f"\nTesting {TEST_SIZE} products")
print(f"Total catalog: {len(df)}")

# ------------------------------------------------------------
# STORAGE
# ------------------------------------------------------------

text_results = []
attribute_results = []
image_results = []

errors = []

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def get_embedding(output):
    """
    Try to extract an embedding from the encoder output.
    """

    possible_fields = [
        "embedding",
        "vector",
        "features",
        "representation",
        "values"
    ]

    for field in possible_fields:

        if hasattr(output, field):

            value = getattr(output, field)

            if value is not None:
                try:
                    arr = np.asarray(value, dtype=np.float32)

                    if arr.size > 0:
                        return arr.flatten()

                except Exception:
                    pass

    # Pydantic / dictionary fallback
    try:

        if hasattr(output, "model_dump"):
            data = output.model_dump()

        elif hasattr(output, "dict"):
            data = output.dict()

        elif hasattr(output, "__dict__"):
            data = output.__dict__

        else:
            data = {}

        for field in possible_fields:

            if field in data and data[field] is not None:

                try:
                    arr = np.asarray(
                        data[field],
                        dtype=np.float32
                    )

                    if arr.size > 0:
                        return arr.flatten()

                except Exception:
                    pass

    except Exception:
        pass

    return None


def build_text_input(row):

    return ProductTextEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        description=str(row["description"]),
        brand=str(row["brand"]),
        category="fashion",
        subcategory=None,
        styles=[],
        occasions=[],
        seasons=[],
        tags=[]
    )


def build_attribute_input(row):

    attrs = {}

    for column in attribute_columns:

        if column in row and row[column] == 1:

            attribute_name = column.replace(
                "attr_",
                ""
            )

            attrs[attribute_name] = attribute_name

    return ProductAttributeEncoderInput(
        productId=str(row["sku"]),
        category="fashion",
        subcategory=None,
        attributes=ProductAttributes(
            customAttributes=attrs
        ),
        occasions=[],
        styles=[],
        seasons=[],
        tags=[],
        rawAttributes=attrs
    )


def build_image_input(row):

    urls = split_images(row["images"])

    image_inputs = []

    for i, url in enumerate(urls):

        image_inputs.append(
            ProductImageInput(
                imageId=f"{row['sku']}_{i}",
                imageUrl=url,
                viewType="front",
                sortOrder=i
            )
        )

    return ProductImageEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        images=image_inputs
    )


# ------------------------------------------------------------
# ENCODER INITIALIZATION
# ------------------------------------------------------------

print("\nInitializing encoders...")

try:

    text_encoder = ProductTextEncoder()
    attribute_encoder = ProductAttributeEncoder()
    image_encoder = ProductImageEncoder()

    print("✅ Text encoder initialized")
    print("✅ Attribute encoder initialized")
    print("✅ Image encoder initialized")

except Exception as e:

    print("\n❌ Encoder initialization failed")
    print(e)
    traceback.print_exc()

    raise


# ------------------------------------------------------------
# PROCESS PRODUCTS
# ------------------------------------------------------------

print("\nProcessing products...")
print("-" * 70)

for i, row in sample_df.iterrows():

    product_id = str(row["sku"])

    try:

        # ====================================================
        # TEXT
        # ====================================================

        text_input = build_text_input(row)

        text_output = text_encoder.encode(text_input)

        text_embedding = get_embedding(text_output)

        text_results.append({
            "product_id": product_id,
            "embedding": text_embedding,
            "dimension": (
                len(text_embedding)
                if text_embedding is not None
                else None
            ),
            "norm": (
                float(np.linalg.norm(text_embedding))
                if text_embedding is not None
                else None
            )
        })


        # ====================================================
        # ATTRIBUTE
        # ====================================================

        attribute_input = build_attribute_input(row)

        attribute_output = attribute_encoder.encode(
            attribute_input
        )

        attribute_embedding = get_embedding(
            attribute_output
        )

        attribute_results.append({
            "product_id": product_id,
            "embedding": attribute_embedding,
            "dimension": (
                len(attribute_embedding)
                if attribute_embedding is not None
                else None
            ),
            "norm": (
                float(np.linalg.norm(attribute_embedding))
                if attribute_embedding is not None
                else None
            )
        })


        # ====================================================
        # IMAGE
        # ====================================================

        image_input = build_image_input(row)

        image_output = image_encoder.encode(image_input)

        image_embedding = get_embedding(image_output)

        image_results.append({
            "product_id": product_id,
            "embedding": image_embedding,
            "dimension": (
                len(image_embedding)
                if image_embedding is not None
                else None
            ),
            "norm": (
                float(np.linalg.norm(image_embedding))
                if image_embedding is not None
                else None
            )
        })


    except Exception as e:

        errors.append({
            "product_id": product_id,
            "error": str(e)
        })


    if (i + 1) % 10 == 0:

        print(
            f"Processed {i + 1}/{TEST_SIZE}"
        )


# ------------------------------------------------------------
# CONVERT RESULTS
# ------------------------------------------------------------

text_df = pd.DataFrame(text_results)
attribute_df = pd.DataFrame(attribute_results)
image_df = pd.DataFrame(image_results)

# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("BULK ENCODER RESULTS")
print("=" * 70)

print("\nTEXT ENCODER")
print("-" * 70)

print("Successful:", len(text_df))

if len(text_df):

    print(
        "Dimensions:",
        text_df["dimension"].value_counts().to_dict()
    )

    print(
        "Missing embeddings:",
        text_df["embedding"].isnull().sum()
    )

    print(
        "Zero vectors:",
        sum(
            x is not None and np.linalg.norm(x) == 0
            for x in text_df["embedding"]
        )
    )

    print(
        "Average norm:",
        text_df["norm"].mean()
    )


print("\nATTRIBUTE ENCODER")
print("-" * 70)

print("Successful:", len(attribute_df))

if len(attribute_df):

    print(
        "Dimensions:",
        attribute_df["dimension"].value_counts().to_dict()
    )

    print(
        "Missing embeddings:",
        attribute_df["embedding"].isnull().sum()
    )

    print(
        "Zero vectors:",
        sum(
            x is not None and np.linalg.norm(x) == 0
            for x in attribute_df["embedding"]
        )
    )

    print(
        "Average norm:",
        attribute_df["norm"].mean()
    )


print("\nIMAGE ENCODER")
print("-" * 70)

print("Successful:", len(image_df))

if len(image_df):

    print(
        "Dimensions:",
        image_df["dimension"].value_counts().to_dict()
    )

    print(
        "Missing embeddings:",
        image_df["embedding"].isnull().sum()
    )

    print(
        "Zero vectors:",
        sum(
            x is not None and np.linalg.norm(x) == 0
            for x in image_df["embedding"]
        )
    )

    print(
        "Average norm:",
        image_df["norm"].mean()
    )


# ------------------------------------------------------------
# ERRORS
# ------------------------------------------------------------

print("\nERRORS")
print("-" * 70)

print("Total errors:", len(errors))

if errors:

    for error in errors[:10]:

        print(
            error["product_id"],
            "→",
            error["error"]
        )


# ------------------------------------------------------------
# DIMENSION CONSISTENCY
# ------------------------------------------------------------

print("\nDIMENSION CONSISTENCY")
print("-" * 70)

for name, result_df in [
    ("TEXT", text_df),
    ("ATTRIBUTE", attribute_df),
    ("IMAGE", image_df)
]:

    if len(result_df):

        dimensions = result_df["dimension"].dropna().unique()

        print(
            f"{name}:",
            dimensions.tolist()
        )

        if len(dimensions) == 1:
            print("✅ Consistent")
        else:
            print("⚠️ INCONSISTENT")


# ------------------------------------------------------------
# FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BULK ENCODER TEST COMPLETE")
print("=" * 70)

print("""
Next we will analyze:

1. Embedding validity
2. Embedding distribution
3. Gender separation
4. Product similarity
5. Text ↔ Image alignment
6. Attribute ↔ Text alignment
7. Duplicate / near-duplicate representations
8. Whether the encoder is ready for recommendation training

DO NOT TRAIN THE RECOMMENDER YET.
""")

ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST

Testing 100 products
Total catalog: 12491

Initializing encoders...
✅ Text encoder initialized
✅ Attribute encoder initialized
✅ Image encoder initialized

Processing products...
----------------------------------------------------------------------
Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100


BULK ENCODER RESULTS

TEXT ENCODER
----------------------------------------------------------------------
Successful: 100
Dimensions: {}
Missing embeddings: 100
Zero vectors: 0
Average norm: nan

ATTRIBUTE ENCODER
----------------------------------------------------------------------
Successful: 100
Dimensions: {}
Missing embeddings: 100
Zero vectors: 0
Average norm: nan

IMAGE ENCODER
----------------------------------------------------------------------
Successful: 100
Dimensions: {}
Missing embeddings: 100
Zero vectors: 0
A

In [23]:
# ============================================================
# ZYRA V1 — FIND ACTUAL EMBEDDING FIELDS
# ============================================================

outputs = {
    "TEXT": text_output,
    "ATTRIBUTE": attribute_output,
    "IMAGE": image_output
}

for name, output in outputs.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("\nTYPE:")
    print(type(output))

    print("\nMODEL FIELDS:")

    if hasattr(output, "model_fields"):
        print(list(output.model_fields.keys()))

    elif hasattr(output, "__fields__"):
        print(list(output.__fields__.keys()))

    else:
        print("No model_fields")

    print("\nDICTIONARY:")

    try:
        if hasattr(output, "model_dump"):
            data = output.model_dump()

        elif hasattr(output, "dict"):
            data = output.dict()

        elif hasattr(output, "__dict__"):
            data = output.__dict__

        else:
            data = {}

        for key, value in data.items():

            print(
                f"{key}: "
                f"type={type(value)}"
            )

            if isinstance(value, (list, tuple, np.ndarray)):

                try:
                    print(
                        f"    length={len(value)}"
                    )
                except:
                    pass

    except Exception as e:

        print("Could not inspect:", e)

    print("\nPUBLIC ATTRIBUTES:")

    print([
        x for x in dir(output)
        if not x.startswith("_")
    ])


TEXT

TYPE:
<class 'zyra.product_encoder.schemas.output_schemas.TextRepresentation'>

MODEL FIELDS:
['productId', 'textInsights', 'textEmbedding', 'embeddingDimension', 'confidence', 'encoderVersion', 'generatedAt', 'processingMetadata']

DICTIONARY:
productId: type=<class 'str'>
textInsights: type=<class 'dict'>
textEmbedding: type=<class 'list'>
    length=512
embeddingDimension: type=<class 'int'>
confidence: type=<class 'float'>
encoderVersion: type=<class 'str'>
generatedAt: type=<class 'datetime.datetime'>
processingMetadata: type=<class 'dict'>

PUBLIC ATTRIBUTES:
['confidence', 'construct', 'copy', 'dict', 'embeddingDimension', 'encoderVersion', 'from_orm', 'generatedAt', 'json', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_str

In [24]:
text_embedding = np.asarray(
    text_output.textEmbedding,
    dtype=np.float32
)

attribute_embedding = np.asarray(
    attribute_output.attributeEmbedding,
    dtype=np.float32
)

image_embedding = np.asarray(
    image_output.visualEmbedding,
    dtype=np.float32
)

print("Text:", text_embedding.shape)
print("Attribute:", attribute_embedding.shape)
print("Image:", image_embedding.shape)

print("\nNorms:")
print("Text:", np.linalg.norm(text_embedding))
print("Attribute:", np.linalg.norm(attribute_embedding))
print("Image:", np.linalg.norm(image_embedding))

Text: (512,)
Attribute: (128,)
Image: (512,)

Norms:
Text: 1.0
Attribute: 1.0
Image: 0.99999994


In [25]:
# ============================================================
# ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST — FIXED
# ============================================================

import numpy as np
import pandas as pd
import traceback

print("=" * 70)
print("ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST")
print("=" * 70)

TEST_COUNT = 100

# ------------------------------------------------------------
# Use existing catalog
# ------------------------------------------------------------

test_df = df.head(TEST_COUNT).copy()

print(f"\nTesting {len(test_df)} products")
print(f"Total catalog: {len(df)}")

# ------------------------------------------------------------
# Initialize encoders
# ------------------------------------------------------------

print("\nInitializing encoders...")

text_encoder = ProductTextEncoder()
attribute_encoder = ProductAttributeEncoder()
image_encoder = ProductImageEncoder()

print("✅ Text encoder initialized")
print("✅ Attribute encoder initialized")
print("✅ Image encoder initialized")

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

text_embeddings = []
attribute_embeddings = []
image_embeddings = []

text_dimensions = []
attribute_dimensions = []
image_dimensions = []

text_norms = []
attribute_norms = []
image_norms = []

errors = []

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def to_vector(value):
    """
    Convert an embedding field into a float32 numpy vector.
    """
    if value is None:
        return None

    arr = np.asarray(value, dtype=np.float32)

    if arr.ndim != 1:
        arr = arr.reshape(-1)

    return arr


# ------------------------------------------------------------
# Process products
# ------------------------------------------------------------

print("\nProcessing products...")
print("-" * 70)

for i, row in test_df.iterrows():

    try:

        # ====================================================
        # TEXT INPUT
        # ====================================================

        text_input = ProductTextEncoderInput(
            productId=str(row["product_id"]),
            title=str(row["name"]),
            description=str(row["description"]),
            brand=str(row["brand"]),
            category="fashion",
            subcategory=None,
            styles=[],
            occasions=[],
            seasons=[],
            tags=[]
        )

        text_output = text_encoder.encode(text_input)

        text_vec = to_vector(text_output.textEmbedding)

        # ====================================================
        # ATTRIBUTE INPUT
        # ====================================================

        detected_attributes = row.get("attributes", [])

        if not isinstance(detected_attributes, list):
            detected_attributes = []

        attribute_dict = {
            str(attr): True
            for attr in detected_attributes
        }

        attribute_input = ProductAttributeEncoderInput(
            productId=str(row["product_id"]),
            category="fashion",
            subcategory=None,
            attributes=ProductAttributes(
                customAttributes=attribute_dict
            ),
            occasions=[],
            styles=[],
            seasons=[],
            tags=[],
            rawAttributes=attribute_dict
        )

        attribute_output = attribute_encoder.encode(attribute_input)

        attribute_vec = to_vector(
            attribute_output.attributeEmbedding
        )

        # ====================================================
        # IMAGE INPUT
        # ====================================================

        image_urls = row.get("image_urls", [])

        if not isinstance(image_urls, list):
            image_urls = []

        image_inputs = [
            ProductImageInput(
                imageUrl=str(url),
                viewType="front",
                sortOrder=j
            )
            for j, url in enumerate(image_urls)
        ]

        image_input = ProductImageEncoderInput(
            productId=str(row["product_id"]),
            title=str(row["name"]),
            images=image_inputs
        )

        image_output = image_encoder.encode(image_input)

        image_vec = to_vector(
            image_output.visualEmbedding
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        if text_vec is None:
            raise ValueError("Text embedding is None")

        if attribute_vec is None:
            raise ValueError("Attribute embedding is None")

        if image_vec is None:
            raise ValueError("Image embedding is None")

        # Dimensions
        text_dimensions.append(len(text_vec))
        attribute_dimensions.append(len(attribute_vec))
        image_dimensions.append(len(image_vec))

        # Norms
        text_norms.append(float(np.linalg.norm(text_vec)))
        attribute_norms.append(float(np.linalg.norm(attribute_vec)))
        image_norms.append(float(np.linalg.norm(image_vec)))

        # Store vectors
        text_embeddings.append(text_vec)
        attribute_embeddings.append(attribute_vec)
        image_embeddings.append(image_vec)

    except Exception as e:

        errors.append({
            "row": i,
            "product_id": row.get("product_id"),
            "error": str(e)
        })

    if len(text_embeddings) % 10 == 0 and len(text_embeddings) > 0:
        print(f"Processed {len(text_embeddings)}/{TEST_COUNT}")


# ============================================================
# RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("BULK ENCODER RESULTS")
print("=" * 70)

# ------------------------------------------------------------
# Text
# ------------------------------------------------------------

print("\nTEXT ENCODER")
print("-" * 70)

print("Successful:", len(text_embeddings))
print("Missing embeddings:", sum(x is None for x in text_embeddings))
print("Dimensions:", sorted(set(text_dimensions)))

if text_norms:
    print("Average norm:", np.mean(text_norms))
    print("Min norm:", np.min(text_norms))
    print("Max norm:", np.max(text_norms))

print(
    "Zero vectors:",
    sum(np.linalg.norm(x) == 0 for x in text_embeddings)
)

# ------------------------------------------------------------
# Attributes
# ------------------------------------------------------------

print("\nATTRIBUTE ENCODER")
print("-" * 70)

print("Successful:", len(attribute_embeddings))
print("Missing embeddings:", sum(x is None for x in attribute_embeddings))
print("Dimensions:", sorted(set(attribute_dimensions)))

if attribute_norms:
    print("Average norm:", np.mean(attribute_norms))
    print("Min norm:", np.min(attribute_norms))
    print("Max norm:", np.max(attribute_norms))

print(
    "Zero vectors:",
    sum(np.linalg.norm(x) == 0 for x in attribute_embeddings)
)

# ------------------------------------------------------------
# Image
# ------------------------------------------------------------

print("\nIMAGE ENCODER")
print("-" * 70)

print("Successful:", len(image_embeddings))
print("Missing embeddings:", sum(x is None for x in image_embeddings))
print("Dimensions:", sorted(set(image_dimensions)))

if image_norms:
    print("Average norm:", np.mean(image_norms))
    print("Min norm:", np.min(image_norms))
    print("Max norm:", np.max(image_norms))

print(
    "Zero vectors:",
    sum(np.linalg.norm(x) == 0 for x in image_embeddings)
)

# ============================================================
# NUMERICAL VALIDITY
# ============================================================

print("\nNUMERICAL VALIDITY")
print("-" * 70)

for name, vectors in [
    ("TEXT", text_embeddings),
    ("ATTRIBUTE", attribute_embeddings),
    ("IMAGE", image_embeddings)
]:

    if len(vectors) > 0:

        matrix = np.vstack(vectors)

        print(f"\n{name}")

        print("Shape:", matrix.shape)

        print("NaN values:", np.isnan(matrix).sum())

        print("Inf values:", np.isinf(matrix).sum())

        print(
            "Finite:",
            np.isfinite(matrix).all()
        )

# ============================================================
# ERRORS
# ============================================================

print("\nERRORS")
print("-" * 70)

print("Total errors:", len(errors))

if errors:

    for error in errors[:10]:
        print(error)

# ============================================================
# DIMENSION CONSISTENCY
# ============================================================

print("\nDIMENSION CONSISTENCY")
print("-" * 70)

print(
    "TEXT:",
    sorted(set(text_dimensions)),
    "→",
    "✅ CONSISTENT" if len(set(text_dimensions)) == 1 else "❌ INCONSISTENT"
)

print(
    "ATTRIBUTE:",
    sorted(set(attribute_dimensions)),
    "→",
    "✅ CONSISTENT" if len(set(attribute_dimensions)) == 1 else "❌ INCONSISTENT"
)

print(
    "IMAGE:",
    sorted(set(image_dimensions)),
    "→",
    "✅ CONSISTENT" if len(set(image_dimensions)) == 1 else "❌ INCONSISTENT"
)

# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)

if (
    len(errors) == 0
    and len(text_embeddings) == TEST_COUNT
    and len(attribute_embeddings) == TEST_COUNT
    and len(image_embeddings) == TEST_COUNT
    and sorted(set(text_dimensions)) == [512]
    and sorted(set(attribute_dimensions)) == [128]
    and sorted(set(image_dimensions)) == [512]
):

    print("✅ BULK PRODUCT ENCODER PASSED")

else:

    print("⚠️ BULK PRODUCT ENCODER NEEDS INVESTIGATION")

print("=" * 70)

ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST

Testing 100 products
Total catalog: 12491

Initializing encoders...
✅ Text encoder initialized
✅ Attribute encoder initialized
✅ Image encoder initialized

Processing products...
----------------------------------------------------------------------


BULK ENCODER RESULTS

TEXT ENCODER
----------------------------------------------------------------------
Successful: 0
Missing embeddings: 0
Dimensions: []
Zero vectors: 0

ATTRIBUTE ENCODER
----------------------------------------------------------------------
Successful: 0
Missing embeddings: 0
Dimensions: []
Zero vectors: 0

IMAGE ENCODER
----------------------------------------------------------------------
Successful: 0
Missing embeddings: 0
Dimensions: []
Zero vectors: 0

NUMERICAL VALIDITY
----------------------------------------------------------------------

ERRORS
----------------------------------------------------------------------
Total errors: 100
{'row': 0, 'product_id': None, '

In [26]:
# ============================================================
# ZYRA V1 — BULK PRODUCT ENCODER TEST — CORRECTED
# ============================================================

import numpy as np

TEST_COUNT = 100
test_df = df.head(TEST_COUNT).copy()

print("=" * 70)
print("ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST")
print("=" * 70)

print(f"\nTesting {len(test_df)} products")
print(f"Total catalog: {len(df)}")

# ------------------------------------------------------------
# Initialize
# ------------------------------------------------------------

print("\nInitializing encoders...")

text_encoder = ProductTextEncoder()
attribute_encoder = ProductAttributeEncoder()
image_encoder = ProductImageEncoder()

print("✅ Text encoder initialized")
print("✅ Attribute encoder initialized")
print("✅ Image encoder initialized")

# ------------------------------------------------------------
# Storage
# ------------------------------------------------------------

text_embeddings = []
attribute_embeddings = []
image_embeddings = []

text_dimensions = []
attribute_dimensions = []
image_dimensions = []

text_norms = []
attribute_norms = []
image_norms = []

errors = []

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def parse_images(value):

    if isinstance(value, list):
        return value

    if not isinstance(value, str):
        return []

    return [
        x.strip()
        for x in value.split("~")
        if x.strip()
    ]


def to_vector(value):

    if value is None:
        return None

    arr = np.asarray(value, dtype=np.float32)

    if arr.ndim != 1:
        arr = arr.reshape(-1)

    return arr


# ------------------------------------------------------------
# Processing
# ------------------------------------------------------------

print("\nProcessing products...")
print("-" * 70)

for count, (_, row) in enumerate(test_df.iterrows(), start=1):

    try:

        product_id = str(row["sku"])

        # ====================================================
        # TEXT
        # ====================================================

        text_input = ProductTextEncoderInput(
            productId=product_id,
            title=str(row["name"]),
            description=str(row["description"]),
            brand=str(row["brand"]),
            category="fashion",
            subcategory=None,
            styles=[],
            occasions=[],
            seasons=[],
            tags=[]
        )

        text_output = text_encoder.encode(text_input)

        text_vec = to_vector(
            text_output.textEmbedding
        )

        # ====================================================
        # ATTRIBUTES
        # ====================================================

        detected_attributes = []

        # Use the generated product feature table when available
        if "attributes" in products.columns:

            product_row = products[
                products["product_id"] == int(row["sku"])
            ]

            if len(product_row) > 0:

                attrs = product_row.iloc[0]["attributes"]

                if isinstance(attrs, list):
                    detected_attributes = attrs

        attribute_dict = {
            str(attr): True
            for attr in detected_attributes
        }

        attribute_input = ProductAttributeEncoderInput(
            productId=product_id,
            category="fashion",
            subcategory=None,
            attributes=ProductAttributes(
                customAttributes=attribute_dict
            ),
            occasions=[],
            styles=[],
            seasons=[],
            tags=[],
            rawAttributes=attribute_dict
        )

        attribute_output = attribute_encoder.encode(
            attribute_input
        )

        attribute_vec = to_vector(
            attribute_output.attributeEmbedding
        )

        # ====================================================
        # IMAGE
        # ====================================================

        image_urls = parse_images(row["images"])

        image_inputs = [
            ProductImageInput(
                imageUrl=url,
                viewType="front",
                sortOrder=j
            )
            for j, url in enumerate(image_urls)
        ]

        image_input = ProductImageEncoderInput(
            productId=product_id,
            title=str(row["name"]),
            images=image_inputs
        )

        image_output = image_encoder.encode(
            image_input
        )

        image_vec = to_vector(
            image_output.visualEmbedding
        )

        # ====================================================
        # VALIDATION
        # ====================================================

        if text_vec is None:
            raise ValueError("Text embedding is None")

        if attribute_vec is None:
            raise ValueError("Attribute embedding is None")

        if image_vec is None:
            raise ValueError("Image embedding is None")

        # Store
        text_embeddings.append(text_vec)
        attribute_embeddings.append(attribute_vec)
        image_embeddings.append(image_vec)

        # Dimensions
        text_dimensions.append(len(text_vec))
        attribute_dimensions.append(len(attribute_vec))
        image_dimensions.append(len(image_vec))

        # Norms
        text_norms.append(float(np.linalg.norm(text_vec)))
        attribute_norms.append(float(np.linalg.norm(attribute_vec)))
        image_norms.append(float(np.linalg.norm(image_vec)))

    except Exception as e:

        errors.append({
            "row": count,
            "product_id": str(row["sku"]),
            "error": str(e)
        })

    if count % 10 == 0:
        print(f"Processed {count}/{TEST_COUNT}")


# ============================================================
# RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("BULK ENCODER RESULTS")
print("=" * 70)

print("\nTEXT ENCODER")
print("-" * 70)

print("Successful:", len(text_embeddings))
print("Missing embeddings:", TEST_COUNT - len(text_embeddings))
print("Dimensions:", sorted(set(text_dimensions)))

if text_norms:
    print("Average norm:", np.mean(text_norms))
    print("Min norm:", np.min(text_norms))
    print("Max norm:", np.max(text_norms))

print(
    "Zero vectors:",
    sum(np.linalg.norm(x) == 0 for x in text_embeddings)
)


print("\nATTRIBUTE ENCODER")
print("-" * 70)

print("Successful:", len(attribute_embeddings))
print("Missing embeddings:", TEST_COUNT - len(attribute_embeddings))
print("Dimensions:", sorted(set(attribute_dimensions)))

if attribute_norms:
    print("Average norm:", np.mean(attribute_norms))
    print("Min norm:", np.min(attribute_norms))
    print("Max norm:", np.max(attribute_norms))

print(
    "Zero vectors:",
    sum(np.linalg.norm(x) == 0 for x in attribute_embeddings)
)


print("\nIMAGE ENCODER")
print("-" * 70)

print("Successful:", len(image_embeddings))
print("Missing embeddings:", TEST_COUNT - len(image_embeddings))
print("Dimensions:", sorted(set(image_dimensions)))

if image_norms:
    print("Average norm:", np.mean(image_norms))
    print("Min norm:", np.min(image_norms))
    print("Max norm:", np.max(image_norms))

print(
    "Zero vectors:",
    sum(np.linalg.norm(x) == 0 for x in image_embeddings)
)


# ============================================================
# NUMERICAL VALIDITY
# ============================================================

print("\nNUMERICAL VALIDITY")
print("-" * 70)

for name, vectors in [
    ("TEXT", text_embeddings),
    ("ATTRIBUTE", attribute_embeddings),
    ("IMAGE", image_embeddings)
]:

    if vectors:

        matrix = np.vstack(vectors)

        print(f"\n{name}")
        print("Shape:", matrix.shape)
        print("NaN values:", np.isnan(matrix).sum())
        print("Inf values:", np.isinf(matrix).sum())
        print("Finite:", np.isfinite(matrix).all())


# ============================================================
# ERRORS
# ============================================================

print("\nERRORS")
print("-" * 70)

print("Total errors:", len(errors))

for error in errors[:10]:
    print(error)


# ============================================================
# DIMENSION CONSISTENCY
# ============================================================

print("\nDIMENSION CONSISTENCY")
print("-" * 70)

print(
    "TEXT:",
    sorted(set(text_dimensions)),
    "→",
    "✅ CONSISTENT"
    if len(set(text_dimensions)) == 1
    else "❌ INCONSISTENT"
)

print(
    "ATTRIBUTE:",
    sorted(set(attribute_dimensions)),
    "→",
    "✅ CONSISTENT"
    if len(set(attribute_dimensions)) == 1
    else "❌ INCONSISTENT"
)

print(
    "IMAGE:",
    sorted(set(image_dimensions)),
    "→",
    "✅ CONSISTENT"
    if len(set(image_dimensions)) == 1
    else "❌ INCONSISTENT"
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)

if (
    len(errors) == 0
    and len(text_embeddings) == TEST_COUNT
    and len(attribute_embeddings) == TEST_COUNT
    and len(image_embeddings) == TEST_COUNT
    and sorted(set(text_dimensions)) == [512]
    and sorted(set(attribute_dimensions)) == [128]
    and sorted(set(image_dimensions)) == [512]
):

    print("✅ BULK PRODUCT ENCODER PASSED")

else:

    print("⚠️ BULK PRODUCT ENCODER NEEDS INVESTIGATION")

print("=" * 70)

ZYRA V1 — BULK PRODUCT ENCODER QUALITY TEST

Testing 100 products
Total catalog: 12491

Initializing encoders...
✅ Text encoder initialized
✅ Attribute encoder initialized
✅ Image encoder initialized

Processing products...
----------------------------------------------------------------------
Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100


BULK ENCODER RESULTS

TEXT ENCODER
----------------------------------------------------------------------
Successful: 100
Missing embeddings: 0
Dimensions: [512]
Average norm: 0.9999999833106995
Min norm: 0.9999999403953552
Max norm: 1.0
Zero vectors: 0

ATTRIBUTE ENCODER
----------------------------------------------------------------------
Successful: 100
Missing embeddings: 0
Dimensions: [128]
Average norm: 1.0
Min norm: 1.0
Max norm: 1.0
Zero vectors: 0

IMAGE ENCODER
-------------------------------------------------------

In [27]:
# ============================================================
# ZYRA V1 — INSPECT EXISTING PRODUCT FUSION
# ============================================================

import inspect
from zyra.product_encoder import fusion

print("=" * 70)
print("ZYRA V1 — PRODUCT FUSION INSPECTION")
print("=" * 70)

print("\nFUSION MODULE:")
print(fusion)

print("\nMODULE MEMBERS:")
for name in dir(fusion):
    if not name.startswith("_"):
        print(name)

# ------------------------------------------------------------
# Inspect the important classes
# ------------------------------------------------------------

for class_name in [
    "UnifiedProductRepresentation",
    "FusionWeightsConfig",
    "ModalityContribution"
]:

    print("\n" + "=" * 70)
    print(class_name)
    print("=" * 70)

    cls = getattr(fusion, class_name, None)

    if cls is None:
        print("❌ Class not found")
        continue

    print("Type:", cls)

    try:
        print("\nSignature:")
        print(inspect.signature(cls))
    except Exception as e:
        print("Signature unavailable:", e)

    print("\nFields:")

    try:
        if hasattr(cls, "model_fields"):
            for field, info in cls.model_fields.items():
                print(f"{field}: {info}")
        else:
            print("No Pydantic model fields")
    except Exception as e:
        print("Could not inspect fields:", e)

# ------------------------------------------------------------
# Search fusion source
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FUSION SOURCE")
print("=" * 70)

try:
    source = inspect.getsource(fusion)
    print(source)
except Exception as e:
    print("Could not retrieve fusion source:", e)

print("\n" + "=" * 70)
print("FUSION INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — PRODUCT FUSION INSPECTION

FUSION MODULE:
<module 'zyra.product_encoder.fusion' from '/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/fusion/__init__.py'>

MODULE MEMBERS:
DeterministicProjectionLayer
EmbeddingValidator
FusionWeightsConfig
ModalityContribution
PROJECTION_SEED
ProductFusionInterface
ProductFusionService
ProductFusionStrategy
UnifiedProductRepresentation
fusion_strategy
interface
models
projections
service
validator

UnifiedProductRepresentation
Type: <class 'zyra.product_encoder.fusion.models.UnifiedProductRepresentation'>

Signature:
(*, productId: str, unifiedProductProfile: zyra.product_encoder.insights.models.UnifiedProductProfile, unifiedEmbedding: List[float], embeddingDimension: int = 662, l2Norm: Annotated[float, Ge(ge=0)] = 1.0, modalities: Dict[str, zyra.product_encoder.fusion.models.ModalityContribution] = <factory>, confidence: Annotated[float, Ge(ge=0.0), Le(le=1.0)] = 1.0, provenance: List[str] = <factory>, metadata: Dict[st

In [28]:
# ============================================================
# ZYRA V1 — REAL PRODUCT FUSION TEST
# ============================================================

import inspect
import numpy as np
from zyra.product_encoder.fusion import (
    ProductFusionService,
    ProductFusionStrategy,
    FusionWeightsConfig
)

print("=" * 70)
print("ZYRA V1 — REAL PRODUCT FUSION TEST")
print("=" * 70)

# ------------------------------------------------------------
# Inspect service
# ------------------------------------------------------------

print("\nFUSION SERVICE")
print("-" * 70)

print(ProductFusionService)

try:
    print("Signature:")
    print(inspect.signature(ProductFusionService))
except Exception as e:
    print("Signature unavailable:", e)

print("\nMethods:")

for name in dir(ProductFusionService):
    if not name.startswith("_"):
        print(name)

# ------------------------------------------------------------
# Inspect strategy
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FUSION STRATEGY")
print("=" * 70)

print(ProductFusionStrategy)

for name in dir(ProductFusionStrategy):
    if not name.startswith("_"):
        print(name)

# ------------------------------------------------------------
# Inspect service source
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SERVICE SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(ProductFusionService))
except Exception as e:
    print("Could not retrieve source:", e)

print("\n" + "=" * 70)
print("STRATEGY SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(ProductFusionStrategy))
except Exception as e:
    print("Could not retrieve source:", e)

print("\n" + "=" * 70)
print("FUSION TEST INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — REAL PRODUCT FUSION TEST

FUSION SERVICE
----------------------------------------------------------------------
<class 'zyra.product_encoder.fusion.service.ProductFusionService'>
Signature:
(strategy: Optional[zyra.product_encoder.fusion.fusion_strategy.ProductFusionStrategy] = None, weights_config: Optional[zyra.product_encoder.fusion.models.FusionWeightsConfig] = None) -> None

Methods:
fuse
fuse_async
validate_inputs

FUSION STRATEGY
<class 'zyra.product_encoder.fusion.fusion_strategy.ProductFusionStrategy'>
fuse_embeddings

SERVICE SOURCE
class ProductFusionService(ProductFusionInterface):
    """
    Main Product Multimodal Fusion Service (Phase P6).
    Synthesizes 512-dim visual, 512-dim text, and 128-dim attribute embeddings with
    the P5 UnifiedProductProfile into a canonical 662-dimensional product representation.
    """

    def __init__(
        self,
        strategy: Optional[ProductFusionStrategy] = None,
        weights_config: Optional[FusionWeightsConfig]

In [29]:
# ============================================================
# ZYRA V1 — SINGLE PRODUCT REAL FUSION
# ============================================================

from zyra.product_encoder.fusion import (
    ProductFusionService,
    FusionWeightsConfig
)

print("=" * 70)
print("ZYRA V1 — SINGLE PRODUCT REAL FUSION")
print("=" * 70)

# ------------------------------------------------------------
# We already have these from the previous encoder test:
#
# text_output
# attribute_output
# image_output
#
# ------------------------------------------------------------

print("\nEncoder outputs:")
print("Text:", type(text_output))
print("Attribute:", type(attribute_output))
print("Image:", type(image_output))

# ------------------------------------------------------------
# Find the UnifiedProductProfile expected by fusion
# ------------------------------------------------------------

print("\nSearching for existing UnifiedProductProfile...")

profile = None

# Check variables already present in notebook
candidate_names = [
    "unified_profile",
    "product_profile",
    "profile",
    "unifiedProductProfile"
]

for name in candidate_names:
    if name in globals():
        candidate = globals()[name]

        if hasattr(candidate, "productId"):
            profile = candidate
            print(f"✅ Found profile variable: {name}")
            break

if profile is None:
    print("❌ UnifiedProductProfile was not found.")
    print()
    print("Available variables containing 'profile':")

    for name in globals():
        if "profile" in name.lower():
            print(" -", name)

    raise RuntimeError(
        "Need the existing UnifiedProductProfile before fusion."
    )

# ------------------------------------------------------------
# Verify IDs
# ------------------------------------------------------------

print("\nProduct IDs:")
print("Profile :", profile.productId)
print("Text    :", text_output.productId)
print("Attribute:", attribute_output.productId)
print("Image   :", image_output.productId)

assert profile.productId == text_output.productId
assert profile.productId == attribute_output.productId
assert profile.productId == image_output.productId

print("✅ Product IDs match")

# ------------------------------------------------------------
# Create fusion service
# ------------------------------------------------------------

fusion_service = ProductFusionService(
    weights_config=FusionWeightsConfig(
        visualWeight=0.45,
        textWeight=0.35,
        attributeWeight=0.20
    )
)

print("\n✅ Fusion service initialized")

# ------------------------------------------------------------
# Perform actual fusion
# ------------------------------------------------------------

fused_output = fusion_service.fuse(
    profile=profile,
    visual=image_output,
    text=text_output,
    attribute=attribute_output
)

# ------------------------------------------------------------
# Inspect result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FUSION RESULT")
print("=" * 70)

print("Product ID:", fused_output.productId)
print("Embedding dimension:", fused_output.embeddingDimension)
print("Embedding length:", len(fused_output.unifiedEmbedding))
print("L2 norm:", fused_output.l2Norm)
print("Confidence:", fused_output.confidence)

print("\nModalities:")

for modality, contribution in fused_output.modalities.items():
    print(
        modality,
        "→ available:", contribution.available,
        "| weight:", contribution.effectiveWeight,
        "| native dim:", contribution.nativeDimension,
        "| norm:", contribution.l2Norm
    )

print("\nProvenance:")
print(fused_output.provenance)

print("\nFirst 20 embedding values:")
print(fused_output.unifiedEmbedding[:20])

# ------------------------------------------------------------
# Numerical validation
# ------------------------------------------------------------

import numpy as np

embedding = np.asarray(
    fused_output.unifiedEmbedding,
    dtype=np.float32
)

print("\nNumerical validation:")
print("Shape:", embedding.shape)
print("NaN:", np.isnan(embedding).sum())
print("Inf:", np.isinf(embedding).sum())
print("Finite:", np.isfinite(embedding).all())
print("Actual norm:", np.linalg.norm(embedding))

assert embedding.shape == (662,)
assert np.isfinite(embedding).all()
assert not np.allclose(embedding, 0)

print("\n" + "=" * 70)
print("✅ SINGLE PRODUCT FUSION PASSED")
print("=" * 70)

ZYRA V1 — SINGLE PRODUCT REAL FUSION

Encoder outputs:
Text: <class 'zyra.product_encoder.schemas.output_schemas.TextRepresentation'>
Attribute: <class 'zyra.product_encoder.schemas.output_schemas.AttributeRepresentation'>
Image: <class 'zyra.product_encoder.schemas.output_schemas.ProductVisualRepresentation'>

Searching for existing UnifiedProductProfile...
❌ UnifiedProductProfile was not found.

Available variables containing 'profile':
 - profile


RuntimeError: Need the existing UnifiedProductProfile before fusion.

In [30]:
print("=" * 70)
print("PROFILE INSPECTION")
print("=" * 70)

print("Type:")
print(type(profile))

print("\nValue:")
print(profile)

print("\nAttributes:")
try:
    print([
        x for x in dir(profile)
        if not x.startswith("_")
    ])
except Exception as e:
    print(e)

print("\nDictionary:")
try:
    if hasattr(profile, "model_dump"):
        print(profile.model_dump())
    elif hasattr(profile, "__dict__"):
        print(profile.__dict__)
except Exception as e:
    print("Could not inspect:", e)

PROFILE INSPECTION
Type:
<class 'NoneType'>

Value:
None

Attributes:
[]

Dictionary:


In [31]:
# ============================================================
# ZYRA V1 — FIND UNIFIED PRODUCT PROFILE IMPLEMENTATION
# ============================================================

import inspect
import pkgutil
import importlib

print("=" * 70)
print("SEARCHING FOR UnifiedProductProfile")
print("=" * 70)

target = "UnifiedProductProfile"
found = []

# Search zyra package
import zyra

for module_info in pkgutil.walk_packages(
    zyra.__path__,
    zyra.__name__ + "."
):
    module_name = module_info.name

    try:
        module = importlib.import_module(module_name)

        if hasattr(module, target):
            obj = getattr(module, target)

            print("\nFOUND")
            print("-" * 70)
            print("Module:", module_name)
            print("Object:", obj)

            try:
                print("Signature:", inspect.signature(obj))
            except:
                pass

            found.append((module_name, obj))

    except Exception:
        pass

print("\n" + "=" * 70)

if not found:
    print("❌ UnifiedProductProfile implementation not found")
else:
    print(f"✅ Found {len(found)} implementation(s)")

print("=" * 70)

SEARCHING FOR UnifiedProductProfile

FOUND
----------------------------------------------------------------------
Module: zyra.product_encoder.api.routes
Object: <class 'zyra.product_encoder.insights.models.UnifiedProductProfile'>
Signature: (*, productId: str, identity: zyra.product_encoder.insights.models.ProductIdentityInsight = <factory>, color: zyra.product_encoder.insights.models.ColorInsightSummary = <factory>, material: zyra.product_encoder.insights.models.MaterialInsightSummary = <factory>, fit: zyra.product_encoder.insights.models.FitInsightSummary = <factory>, pattern: Optional[zyra.product_encoder.insights.models.ResolvedAttribute] = None, designDetails: zyra.product_encoder.insights.models.DesignDetailsSummary = <factory>, styleProfile: List[zyra.product_encoder.schemas.insight_schemas.ConfidenceAwareInsight] = <factory>, occasionProfile: List[zyra.product_encoder.schemas.insight_schemas.ConfidenceAwareInsight] = <factory>, seasonProfile: List[zyra.product_encoder.schemas.

In [32]:
# ============================================================
# ZYRA V1 — INSPECT EXISTING PRODUCT INSIGHT BUILDER
# ============================================================

import inspect

from zyra.product_encoder.insights import builder

print("=" * 70)
print("ZYRA V1 — PRODUCT INSIGHT BUILDER")
print("=" * 70)

print("\nMODULE:")
print(builder)

print("\nPUBLIC MEMBERS:")
for name in dir(builder):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("CLASSES / FUNCTIONS")
print("=" * 70)

for name in dir(builder):

    if name.startswith("_"):
        continue

    obj = getattr(builder, name)

    if inspect.isclass(obj) or inspect.isfunction(obj):

        print("\n" + "-" * 70)
        print(name)
        print("-" * 70)

        try:
            print("Signature:")
            print(inspect.signature(obj))
        except Exception:
            pass

print("\n" + "=" * 70)
print("BUILDER SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(builder))
except Exception as e:
    print("Could not retrieve source:", e)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — PRODUCT INSIGHT BUILDER

MODULE:
<module 'zyra.product_encoder.insights.builder' from '/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/insights/builder.py'>

PUBLIC MEMBERS:
Any
AttributeEvidence
AttributeRepresentation
ColorInsightSummary
ConfidenceAwareInsight
CrossModalConflict
DesignDetailsSummary
Dict
FitInsightSummary
List
MaterialInsightSummary
Optional
ProductConfidenceAggregator
ProductConflictDetector
ProductIdentityInsight
ProductProfileBuilder
ProductVisualRepresentation
ResolvedAttribute
SizeProfileSummary
TextRepresentation
UnifiedProductProfile
logger
logging

CLASSES / FUNCTIONS

----------------------------------------------------------------------
Any
----------------------------------------------------------------------
Signature:
(*args, **kwargs)

----------------------------------------------------------------------
AttributeEvidence
----------------------------------------------------------------------
Signature:
(*, attribute: str

In [35]:
# ============================================================
# ZYRA V1 — TRACE PRODUCT EVIDENCE PIPELINE
# ============================================================

import inspect

print("=" * 70)
print("ZYRA V1 — EVIDENCE PIPELINE INSPECTION")
print("=" * 70)

builder_obj = ProductProfileBuilder()

print("\nProductProfileBuilder methods:")
for name in dir(builder_obj):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("CONFLICT DETECTOR")
print("=" * 70)

print(ProductConflictDetector)

try:
    print(inspect.getsource(ProductConflictDetector))
except Exception as e:
    print("Source unavailable:", e)

print("\n" + "=" * 70)
print("CONFIDENCE AGGREGATOR")
print("=" * 70)

print(ProductConfidenceAggregator)

try:
    print(inspect.getsource(ProductConfidenceAggregator))
except Exception as e:
    print("Source unavailable:", e)

print("\n" + "=" * 70)
print("SEARCHING PROJECT FOR evidence_by_attr")
print("=" * 70)

import pathlib

project_root = pathlib.Path.cwd()

for path in project_root.rglob("*.py"):
    if ".venv" in str(path) or ".env" in str(path):
        continue

    try:
        text = path.read_text(errors="ignore")

        if "evidence_by_attr" in text:
            print("\nFOUND:", path)

            for i, line in enumerate(text.splitlines(), 1):
                if "evidence_by_attr" in line:
                    print(f"{i}: {line.strip()}")

    except Exception:
        pass

print("\n" + "=" * 70)
print("TRACE COMPLETE")
print("=" * 70)

ZYRA V1 — EVIDENCE PIPELINE INSPECTION

ProductProfileBuilder methods:
build_profile
confidence_aggregator
conflict_detector

CONFLICT DETECTOR
<class 'zyra.product_encoder.insights.conflict_detector.ProductConflictDetector'>
class ProductConflictDetector:
    """
    Detects cross-modal contradictions and distinguishes mutually exclusive
    conflicts from compatible multi-label differences.
    """

    def is_contradictory_fit(self, val1: str, val2: str) -> bool:
        v1, v2 = val1.lower(), val2.lower()
        if v1 == v2:
            return False
        return (v1, v2) in MUTUALLY_EXCLUSIVE_FITS or (v2, v1) in MUTUALLY_EXCLUSIVE_FITS

    def is_contradictory_pattern(self, val1: str, val2: str) -> bool:
        v1, v2 = val1.lower(), val2.lower()
        if v1 == v2:
            return False
        return (v1, v2) in MUTUALLY_EXCLUSIVE_PATTERNS or (v2, v1) in MUTUALLY_EXCLUSIVE_PATTERNS

    def is_contradictory_sleeve(self, val1: str, val2: str) -> bool:
        v1, v2 = val

In [34]:
from zyra.product_encoder.insights.builder import (
    ProductProfileBuilder,
    ProductConflictDetector,
    ProductConfidenceAggregator
)

print("✅ Builder imports loaded")

✅ Builder imports loaded


In [36]:
# ============================================================
# ZYRA V1 — REAL PRODUCT INSIGHT PIPELINE TEST
# ============================================================

import inspect

from zyra.product_encoder.insights.service import ProductInsightService

print("=" * 70)
print("ZYRA V1 — PRODUCT INSIGHT SERVICE INSPECTION")
print("=" * 70)

print("\nSERVICE:")
print(ProductInsightService)

print("\nSIGNATURE:")
try:
    print(inspect.signature(ProductInsightService))
except Exception as e:
    print("Unavailable:", e)

print("\nMETHODS:")
for name in dir(ProductInsightService):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(ProductInsightService))
except Exception as e:
    print("Could not retrieve source:", e)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ImportError: cannot import name 'ProductInsightService' from 'zyra.product_encoder.insights.service' (/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/insights/service.py)

In [37]:
# ============================================================
# ZYRA V1 — INSPECT ACTUAL INSIGHTS SERVICE
# ============================================================

import inspect
import zyra.product_encoder.insights.service as service

print("=" * 70)
print("ZYRA V1 — ACTUAL INSIGHTS SERVICE")
print("=" * 70)

print("\nMODULE:")
print(service)

print("\nPUBLIC MEMBERS:")
for name in dir(service):
    if not name.startswith("_"):
        obj = getattr(service, name)

        if inspect.isclass(obj) or inspect.isfunction(obj):
            print(f"\n{name}")
            print("Type:", type(obj))

            try:
                print("Signature:", inspect.signature(obj))
            except:
                pass

print("\n" + "=" * 70)
print("SERVICE SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(service))
except Exception as e:
    print("Could not retrieve source:", e)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — ACTUAL INSIGHTS SERVICE

MODULE:
<module 'zyra.product_encoder.insights.service' from '/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/insights/service.py'>

PUBLIC MEMBERS:

AttributeEvidenceCollector
Type: <class 'type'>
Signature: (aligner: Optional[zyra.product_encoder.insights.aligner.CrossModalAttributeAligner] = None) -> None

AttributeRepresentation
Type: <class 'pydantic._internal._model_construction.ModelMetaclass'>
Signature: (*, productId: str, structuredAttributes: Optional[zyra.product_encoder.schemas.insight_schemas.AttributeInsights] = None, attributeEmbedding: Optional[List[float]] = None, embeddingDimension: int = 128, confidence: Annotated[float, Ge(ge=0.0), Le(le=1.0)] = 1.0, encoderVersion: str = 'v0-foundation', generatedAt: datetime.datetime = <factory>, processingMetadata: Dict[str, Any] = <factory>) -> None

ProductInsightAggregationService
Type: <class 'abc.ABCMeta'>
Signature: (collector: Optional[zyra.product_encoder.insights.

In [38]:
# ============================================================
# ZYRA V1 — REAL P5 PRODUCT INSIGHT AGGREGATION TEST
# ============================================================

import numpy as np

from zyra.product_encoder.insights.service import (
    ProductInsightAggregationService
)

print("=" * 70)
print("ZYRA V1 — REAL P5 PRODUCT INSIGHT AGGREGATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify encoder outputs exist
# ------------------------------------------------------------

print("\nINPUT REPRESENTATIONS")
print("-" * 70)

print("TEXT:")
print("  Product ID:", text_output.productId)
print("  Embedding:", len(text_output.textEmbedding))
print("  Confidence:", text_output.confidence)

print("\nATTRIBUTE:")
print("  Product ID:", attribute_output.productId)
print("  Embedding:", len(attribute_output.attributeEmbedding))
print("  Confidence:", attribute_output.confidence)

print("\nIMAGE:")
print("  Product ID:", image_output.productId)
print("  Embedding:", len(image_output.aggregatedEmbedding))
print("  Successful images:", image_output.successfulImageCount)
print("  Failed images:", image_output.failedImageCount)
print("  Confidence:", image_output.confidence)

# ------------------------------------------------------------
# 2. Verify product IDs
# ------------------------------------------------------------

product_ids = {
    text_output.productId,
    attribute_output.productId,
    image_output.productId,
}

print("\nProduct IDs:", product_ids)

assert len(product_ids) == 1, (
    f"Product ID mismatch: {product_ids}"
)

print("✅ All modality product IDs match")

# ------------------------------------------------------------
# 3. Initialize P5 aggregation service
# ------------------------------------------------------------

insight_service = ProductInsightAggregationService()

print("\n✅ ProductInsightAggregationService initialized")

# ------------------------------------------------------------
# 4. Run REAL P5 aggregation
# ------------------------------------------------------------

print("\nRunning aggregation...")

profile = insight_service.aggregate(
    visual=image_output,
    text=text_output,
    attribute=attribute_output,
)

print("✅ Aggregation completed")

# ------------------------------------------------------------
# 5. Basic profile inspection
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UNIFIED PRODUCT PROFILE")
print("=" * 70)

print("\nType:")
print(type(profile))

print("\nProduct ID:")
print(profile.productId)

print("\nOverall confidence:")
print(profile.confidence)

print("\nIdentity:")
print(profile.identity)

print("\nColor:")
print(profile.color)

print("\nMaterial:")
print(profile.material)

print("\nFit:")
print(profile.fit)

print("\nPattern:")
print(profile.pattern)

print("\nDesign details:")
print(profile.designDetails)

print("\nStyle profile:")
print(profile.styleProfile)

print("\nOccasion profile:")
print(profile.occasionProfile)

print("\nSize profile:")
print(profile.sizeProfile)

print("\nConflicts:")
print(profile.conflicts)

print("\nMissing information:")
print(profile.missingInformation)

# ------------------------------------------------------------
# 6. Modality / provenance information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PROVENANCE / MODALITY SUMMARY")
print("=" * 70)

print("\nProvenance:")
print(profile.provenance)

print("\nModality summary:")
print(profile.modalitySummary)

print("\nEncoder versions:")
print(profile.encoderVersions)

# ------------------------------------------------------------
# 7. Pydantic validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PROFILE VALIDATION")
print("=" * 70)

profile_dict = profile.model_dump()

print("Fields generated:")
print(list(profile_dict.keys()))

assert profile.productId in product_ids
assert 0.0 <= profile.confidence <= 1.0

print("✅ UnifiedProductProfile is valid")

print("\n" + "=" * 70)
print("✅ P5 PRODUCT INSIGHT AGGREGATION PASSED")
print("=" * 70)

ZYRA V1 — REAL P5 PRODUCT INSIGHT AGGREGATION

INPUT REPRESENTATIONS
----------------------------------------------------------------------
TEXT:
  Product ID: 10003299
  Embedding: 512
  Confidence: 0.9

ATTRIBUTE:
  Product ID: 10003299
  Embedding: 128
  Confidence: 1.0

IMAGE:
  Product ID: 10003299
  Embedding: 512
  Successful images: 3
  Failed images: 0
  Confidence: 1.0

Product IDs: {'10003299'}
✅ All modality product IDs match
2026-08-27 23:14:14,268 | INFO     | zyra.product_encoder.insights.service | ProductInsightAggregationService initialized (P5)

✅ ProductInsightAggregationService initialized

Running aggregation...
2026-08-27 23:14:14,270 | INFO     | zyra.product_encoder.insights.service | Starting insight aggregation for productId=10003299
2026-08-27 23:14:14,271 | INFO     | zyra.product_encoder.insights.service | Insight aggregation complete for productId=10003299 in 2.00ms (collect=0.94ms, build=0.65ms, conflicts=2)
✅ Aggregation completed

UNIFIED PRODUCT PROFIL

In [39]:
# ============================================================
# ZYRA V1 — REAL 662D PRODUCT FUSION
# ============================================================

import numpy as np

from zyra.product_encoder.fusion import (
    ProductFusionService,
    FusionWeightsConfig
)

print("=" * 70)
print("ZYRA V1 — REAL 662D PRODUCT FUSION")
print("=" * 70)

# ------------------------------------------------------------
# Initialize existing fusion service
# ------------------------------------------------------------

fusion_service = ProductFusionService(
    weights_config=FusionWeightsConfig(
        visualWeight=0.45,
        textWeight=0.35,
        attributeWeight=0.20
    )
)

print("\n✅ Fusion service initialized")

# ------------------------------------------------------------
# Fuse the REAL encoder outputs + REAL P5 profile
# ------------------------------------------------------------

fused = fusion_service.fuse(
    profile=profile,
    visual=image_output,
    text=text_output,
    attribute=attribute_output
)

print("✅ Fusion completed")

# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FUSED PRODUCT REPRESENTATION")
print("=" * 70)

print("\nProduct ID:")
print(fused.productId)

print("\nEmbedding dimension:")
print(fused.embeddingDimension)

print("\nEmbedding length:")
print(len(fused.unifiedEmbedding))

print("\nStored L2 norm:")
print(fused.l2Norm)

print("\nConfidence:")
print(fused.confidence)

print("\nModalities:")

for name, modality in fused.modalities.items():
    print(
        f"{name}: "
        f"available={modality.available}, "
        f"weight={modality.effectiveWeight}, "
        f"dimension={modality.nativeDimension}, "
        f"norm={modality.l2Norm}"
    )

print("\nProvenance:")
print(fused.provenance)

# ------------------------------------------------------------
# Numerical validation
# ------------------------------------------------------------

embedding = np.asarray(
    fused.unifiedEmbedding,
    dtype=np.float32
)

print("\n" + "=" * 70)
print("NUMERICAL VALIDATION")
print("=" * 70)

print("Shape:", embedding.shape)
print("NaN:", np.isnan(embedding).sum())
print("Inf:", np.isinf(embedding).sum())
print("Finite:", np.isfinite(embedding).all())
print("Actual L2 norm:", np.linalg.norm(embedding))
print("Min:", embedding.min())
print("Max:", embedding.max())
print("Mean:", embedding.mean())
print("Std:", embedding.std())

assert embedding.shape == (662,)
assert np.isfinite(embedding).all()
assert np.linalg.norm(embedding) > 0

print("\n" + "=" * 70)
print("✅ REAL 662D PRODUCT FUSION PASSED")
print("=" * 70)

ZYRA V1 — REAL 662D PRODUCT FUSION
2026-08-27 23:15:15,239 | INFO     | zyra.product_encoder.fusion.projections | DeterministicProjectionLayer initialized with seed=42
2026-08-27 23:15:15,240 | INFO     | zyra.product_encoder.fusion.service | ProductFusionService initialized (P6, unified_dim=662)

✅ Fusion service initialized
2026-08-27 23:15:15,241 | INFO     | zyra.product_encoder.fusion.service | Starting multimodal fusion for productId=10003299
2026-08-27 23:15:15,243 | INFO     | zyra.product_encoder.fusion.service | Multimodal fusion complete for productId=10003299 in 2.14ms (dim=662, norm=1.0000, modalities=['visual', 'text', 'attribute'])
✅ Fusion completed

FUSED PRODUCT REPRESENTATION

Product ID:
10003299

Embedding dimension:
662

Embedding length:
662

Stored L2 norm:
1.0

Confidence:
0.63

Modalities:
visual: available=True, weight=0.45, dimension=512, norm=1.0
text: available=True, weight=0.35, dimension=512, norm=1.0
attribute: available=True, weight=0.2, dimension=128,

In [40]:
# ============================================================
# ZYRA V1 — P7 FULL PIPELINE BENCHMARK
# ============================================================

import time
import numpy as np
import pandas as pd

from zyra.product_encoder.text_encoder.encoder import ProductTextEncoder
from zyra.product_encoder.attribute_encoder.encoder import ProductAttributeEncoder
from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder
from zyra.product_encoder.insights.service import ProductInsightAggregationService
from zyra.product_encoder.fusion import (
    ProductFusionService,
    FusionWeightsConfig
)

print("=" * 70)
print("ZYRA V1 — P7 FULL PIPELINE BENCHMARK")
print("=" * 70)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

BENCHMARK_PRODUCTS = 10

benchmark_df = df.head(BENCHMARK_PRODUCTS).copy()

print("\nProducts:", len(benchmark_df))
print("Catalog:", len(df))

# ------------------------------------------------------------
# Initialize everything once
# ------------------------------------------------------------

print("\nInitializing pipeline...")

text_encoder = ProductTextEncoder()
attribute_encoder = ProductAttributeEncoder()
image_encoder = ProductImageEncoder()

insight_service = ProductInsightAggregationService()

fusion_service = ProductFusionService(
    weights_config=FusionWeightsConfig(
        visualWeight=0.45,
        textWeight=0.35,
        attributeWeight=0.20
    )
)

print("✅ Text encoder")
print("✅ Attribute encoder")
print("✅ Image encoder")
print("✅ P5 Insight service")
print("✅ P6 Fusion service")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def parse_images(value):
    if isinstance(value, list):
        return value

    if value is None:
        return []

    return [
        x.strip()
        for x in str(value).split("~")
        if x.strip()
    ]


def make_text_input(row):
    return ProductTextEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        description=str(row["description"]),
        brand=str(row["brand"]),
        category="fashion",
        styles=[],
        occasions=[],
        seasons=[],
        tags=[]
    )


def make_attribute_input(row):
    attrs = {}

    for col in attribute_columns:
        if col.startswith("attr_") and row.get(col, 0) == 1:
            attrs[col.replace("attr_", "")] = True

    return ProductAttributeEncoderInput(
        productId=str(row["sku"]),
        category="fashion",
        attributes=ProductAttributes(
            customAttributes=attrs
        ),
        rawAttributes=attrs
    )


def make_image_input(row):
    urls = parse_images(row["images"])

    images = [
        ProductImageInput(
            imageUrl=url,
            viewType="front",
            sortOrder=i
        )
        for i, url in enumerate(urls)
    ]

    return ProductImageEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        images=images
    )

# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

results = []
errors = []

pipeline_start = time.perf_counter()

print("\n" + "-" * 70)
print("PROCESSING")
print("-" * 70)

for idx, (_, row) in enumerate(benchmark_df.iterrows(), 1):

    product_id = str(row["sku"])

    start = time.perf_counter()

    try:

        # ----------------------------------------------------
        # TEXT
        # ----------------------------------------------------

        text_input = make_text_input(row)

        text_output = text_encoder.encode(
            text_input
        )

        # ----------------------------------------------------
        # ATTRIBUTE
        # ----------------------------------------------------

        attribute_input = make_attribute_input(row)

        attribute_output = attribute_encoder.encode(
            attribute_input
        )

        # ----------------------------------------------------
        # IMAGE
        # ----------------------------------------------------

        image_input = make_image_input(row)

        image_output = image_encoder.encode(
            image_input
        )

        # ----------------------------------------------------
        # P5 — INSIGHT AGGREGATION
        # ----------------------------------------------------

        profile = insight_service.aggregate(
            visual=image_output,
            text=text_output,
            attribute=attribute_output
        )

        # ----------------------------------------------------
        # P6 — FUSION
        # ----------------------------------------------------

        fused = fusion_service.fuse(
            profile=profile,
            visual=image_output,
            text=text_output,
            attribute=attribute_output
        )

        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        embedding = np.asarray(
            fused.unifiedEmbedding,
            dtype=np.float32
        )

        assert embedding.shape == (662,)
        assert np.isfinite(embedding).all()
        assert np.linalg.norm(embedding) > 0

        elapsed = time.perf_counter() - start

        results.append({
            "product_id": product_id,
            "text_dim": len(text_output.textEmbedding),
            "attribute_dim": len(attribute_output.attributeEmbedding),
            "image_dim": len(image_output.aggregatedEmbedding),
            "final_dim": len(fused.unifiedEmbedding),
            "profile_confidence": profile.confidence,
            "fusion_confidence": fused.confidence,
            "image_count": image_output.successfulImageCount,
            "elapsed_seconds": elapsed
        })

        print(
            f"Processed {idx}/{BENCHMARK_PRODUCTS} "
            f"| {product_id} "
            f"| {elapsed:.2f}s"
        )

    except Exception as e:

        elapsed = time.perf_counter() - start

        errors.append({
            "product_id": product_id,
            "error": str(e),
            "elapsed_seconds": elapsed
        })

        print(
            f"❌ {idx}/{BENCHMARK_PRODUCTS} "
            f"| {product_id} "
            f"| {e}"
        )

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

total_time = time.perf_counter() - pipeline_start

results_df = pd.DataFrame(results)
errors_df = pd.DataFrame(errors)

print("\n" + "=" * 70)
print("P7 BENCHMARK RESULTS")
print("=" * 70)

print("\nSuccessful:", len(results))
print("Failed:", len(errors))

print("\nTotal benchmark time:")
print(f"{total_time:.2f} seconds")

if len(results) > 0:

    avg_time = results_df["elapsed_seconds"].mean()

    print("\nAverage time/product:")
    print(f"{avg_time:.2f} seconds")

    estimated_seconds = avg_time * len(df)
    estimated_minutes = estimated_seconds / 60
    estimated_hours = estimated_minutes / 60

    print("\nEstimated full catalog processing:")
    print(f"{estimated_seconds:,.0f} seconds")
    print(f"{estimated_minutes:,.1f} minutes")
    print(f"{estimated_hours:,.2f} hours")

    print("\nEmbedding dimensions:")
    print(
        "Text:",
        results_df["text_dim"].unique()
    )
    print(
        "Attribute:",
        results_df["attribute_dim"].unique()
    )
    print(
        "Image:",
        results_df["image_dim"].unique()
    )
    print(
        "Final:",
        results_df["final_dim"].unique()
    )

    print("\nImage statistics:")
    print(
        results_df["image_count"].describe()
    )

    print("\nConfidence:")
    print(
        results_df[
            ["profile_confidence", "fusion_confidence"]
        ].describe()
    )

# ------------------------------------------------------------
# Errors
# ------------------------------------------------------------

if len(errors_df) > 0:

    print("\n" + "=" * 70)
    print("ERRORS")
    print("=" * 70)

    print(errors_df.to_string(index=False))

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if len(results) == BENCHMARK_PRODUCTS and len(errors) == 0:
    print("✅ P7 FULL PIPELINE BENCHMARK PASSED")
else:
    print("⚠️ P7 NEEDS INVESTIGATION")

print("=" * 70)

ZYRA V1 — P7 FULL PIPELINE BENCHMARK

Products: 10
Catalog: 12491

Initializing pipeline...
2026-08-27 23:16:00,668 | INFO     | zyra.product_encoder.text_encoder.model_manager | ProductTextModelManager initialized: device=mps, cache_dir=/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/models/cache, model=openai/clip-vit-base-patch32
2026-08-27 23:16:00,669 | INFO     | zyra.product_encoder.text_encoder.encoder | ProductTextEncoder initialized (version=v0-foundation)
2026-08-27 23:16:00,669 | INFO     | zyra.product_encoder.attribute_encoder.encoder | ProductAttributeEncoder initialized (version=v0-foundation)
2026-08-27 23:16:00,671 | INFO     | zyra.product_encoder.image_encoder.model_manager | ProductVisionModelManager initialized: device=mps, cache_dir=/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/models/cache, model=openai/clip-vit-base-patch32
2026-08-27 23:16:00,671 | INFO     | zyra.product_encoder.insights.service | ProductInsightAg

In [41]:
# ============================================================
# ZYRA V1 — CHECK EXISTING PRODUCT EMBEDDINGS
# ============================================================

from pathlib import Path

root = Path.cwd()

print("=" * 70)
print("SEARCHING FOR EXISTING PRODUCT EMBEDDINGS")
print("=" * 70)

patterns = [
    "*embedding*.npy",
    "*embeddings*.npy",
    "*embedding*.npz",
    "*embeddings*.npz",
    "*product*embedding*",
    "*product*embeddings*",
]

found = []

for pattern in patterns:
    for path in root.rglob(pattern):
        if ".venv" in str(path) or ".env" in str(path):
            continue

        if path.is_file() and path not in found:
            found.append(path)

if not found:
    print("\n❌ No existing product embedding files found.")
else:
    print(f"\n✅ Found {len(found)} candidate files:\n")

    for path in found:
        size_mb = path.stat().st_size / (1024 * 1024)

        print(
            f"{path}\n"
            f"  Size: {size_mb:.2f} MB\n"
        )

print("=" * 70)

SEARCHING FOR EXISTING PRODUCT EMBEDDINGS

❌ No existing product embedding files found.


In [43]:
# ============================================================
# CORRECT ZYRA PRODUCT INPUT IMPORTS
# ============================================================

from zyra.product_encoder.ingestion.router import (
    ProductTextEncoderInput,
    ProductAttributeEncoderInput,
    ProductAttributes,
    ProductImageEncoderInput,
    ProductImageInput,
)

print("✅ Product input classes imported")

✅ Product input classes imported


In [ ]:
print(ProductTextEncoderInput)
print(ProductAttributeEncoderInput)
print(ProductAttributes)
print(ProductImageEncoderInput)
print(ProductImageInput)

In [1]:
# ============================================================
# ZYRA V1 — M5 APPLE GPU CHECK
# ============================================================

import torch

print("=" * 70)
print("ZYRA V1 — M5 GPU / MPS CHECK")
print("=" * 70)

print("\nPyTorch:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("\n✅ M5 APPLE GPU AVAILABLE")
    print("Device:", DEVICE)
else:
    DEVICE = torch.device("cpu")
    print("\n⚠️ MPS NOT AVAILABLE")
    print("Device:", DEVICE)

print("=" * 70)

ZYRA V1 — M5 GPU / MPS CHECK

PyTorch: 2.13.0
MPS built: True
MPS available: True

✅ M5 APPLE GPU AVAILABLE
Device: mps


In [2]:
# ============================================================
# ZYRA V1 — CHECK IMAGE ENCODER DEVICE
# ============================================================

import inspect

from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder

print("=" * 70)
print("ZYRA V1 — IMAGE ENCODER DEVICE INSPECTION")
print("=" * 70)

print("\nProductImageEncoder:")
print(ProductImageEncoder)

print("\nConstructor:")
print(inspect.signature(ProductImageEncoder))

print("\nMethods:")
for name in dir(ProductImageEncoder):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(ProductImageEncoder))
except Exception as e:
    print("Could not inspect source:", e)

print("=" * 70)

ZYRA V1 — IMAGE ENCODER DEVICE INSPECTION

ProductImageEncoder:
<class 'zyra.product_encoder.image_encoder.encoder.ProductImageEncoder'>

Constructor:
(loader: Optional[zyra.product_encoder.image_encoder.retrieval.ProductImageLoader] = None, preprocessor: Optional[zyra.product_encoder.image_encoder.preprocessing.ProductImagePreprocessor] = None, backbone: Optional[zyra.product_encoder.image_encoder.vision_backbone.ProductVisionBackbone] = None, aggregator: Optional[zyra.product_encoder.image_encoder.aggregator.MultiImageVisualAggregator] = None) -> None

Methods:
encode
encode_async

SOURCE
class ProductImageEncoder(ProductImageEncoderInterface):
    """
    Main implementation of the Zyra Product Image Encoder (Phase P2).
    Processes product image collections, extracts deep visual representations and insights,
    and synthesizes view-weighted product visual embeddings.
    """

    def __init__(
        self,
        loader: Optional[ProductImageLoader] = None,
        preprocessor

In [3]:
# ============================================================
# ZYRA V1 — INSPECT VISION BACKBONE DEVICE
# ============================================================

import inspect

from zyra.product_encoder.image_encoder.vision_backbone import (
    ProductVisionBackbone
)

print("=" * 70)
print("ZYRA V1 — VISION BACKBONE DEVICE INSPECTION")
print("=" * 70)

print("\nCLASS:")
print(ProductVisionBackbone)

print("\nCONSTRUCTOR:")
print(inspect.signature(ProductVisionBackbone))

print("\nMETHODS:")
for name in dir(ProductVisionBackbone):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(ProductVisionBackbone))
except Exception as e:
    print("Could not retrieve source:", e)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — VISION BACKBONE DEVICE INSPECTION

CLASS:
<class 'zyra.product_encoder.image_encoder.vision_backbone.ProductVisionBackbone'>

CONSTRUCTOR:
(model_manager: Optional[zyra.product_encoder.image_encoder.model_manager.ProductVisionModelManager] = None, color_extractor: Optional[zyra.product_encoder.image_encoder.color_extractor.ProductColorExtractor] = None) -> None

METHODS:
extract_representation_and_insights

SOURCE
class ProductVisionBackbone:
    """
    Executes deep visual feature extraction using a pretrained vision-language backbone (CLIP).
    Extracts 512-dim dense visual vectors and evaluates zero-shot semantic similarities
    against canonical fashion taxonomies.
    """

    def __init__(
        self,
        model_manager: Optional[ProductVisionModelManager] = None,
        color_extractor: Optional[ProductColorExtractor] = None,
    ) -> None:
        self.model_manager = model_manager or ProductVisionModelManager()
        self.color_extractor = color_extractor 

In [4]:
# ============================================================
# ZYRA V1 — M5 / MPS VISION MODEL MANAGER INSPECTION
# ============================================================

import inspect

from zyra.product_encoder.image_encoder.model_manager import (
    ProductVisionModelManager
)

print("=" * 70)
print("ZYRA V1 — VISION MODEL MANAGER INSPECTION")
print("=" * 70)

print("\nCLASS:")
print(ProductVisionModelManager)

print("\nCONSTRUCTOR:")
print(inspect.signature(ProductVisionModelManager))

print("\nPUBLIC METHODS:")
for name in dir(ProductVisionModelManager):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("SOURCE")
print("=" * 70)

try:
    print(inspect.getsource(ProductVisionModelManager))
except Exception as e:
    print("Could not retrieve source:", e)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — VISION MODEL MANAGER INSPECTION

CLASS:
<class 'zyra.product_encoder.image_encoder.model_manager.ProductVisionModelManager'>

CONSTRUCTOR:
(models_dir: Optional[str] = None, device: Optional[str] = None, vision_model_name: str = 'openai/clip-vit-base-patch32') -> None

PUBLIC METHODS:
get_device
get_vision_model

SOURCE
class ProductVisionModelManager:
    """
    Manages pretrained vision model weights, local caching, and device placement.
    Supports CUDA, Apple Silicon MPS, and CPU execution with graceful offline fallbacks.
    """

    def __init__(
        self,
        models_dir: Optional[str] = None,
        device: Optional[str] = None,
        vision_model_name: str = "openai/clip-vit-base-patch32",
    ) -> None:
        settings = get_product_settings()
        self.models_dir = models_dir or os.path.join(settings.MODELS_DIR, "cache")
        os.makedirs(self.models_dir, exist_ok=True)

        self.vision_model_name = vision_model_name
        self._device_str =

In [7]:
# ============================================================
# ZYRA V1 — VERIFY REAL CLIP IS USING M5 GPU
# ============================================================

from zyra.product_encoder.image_encoder.model_manager import (
    ProductVisionModelManager
)

print("=" * 70)
print("ZYRA V1 — REAL M5 CLIP DEVICE TEST")
print("=" * 70)

manager = ProductVisionModelManager()

print("\nDetected device:")
print(manager.get_device())

model, processor = manager.get_vision_model()

print("\nModel loaded:")
print(model is not None)

print("Processor loaded:")
print(processor is not None)

if model is not None:

    actual_device = next(model.parameters()).device

    print("\nActual model device:")
    print(actual_device)

    print("\nModel class:")
    print(type(model))

    if actual_device.type == "mps":
        print("\n✅ CLIP IS RUNNING ON M5 GPU / MPS")
    else:
        print(
            f"\n⚠️ CLIP IS NOT ON MPS — running on {actual_device}"
        )

else:

    print("\n⚠️ CLIP MODEL WAS NOT LOADED")
    print("Zyra may be using deterministic fallback mode.")

print("=" * 70)

ZYRA V1 — REAL M5 CLIP DEVICE TEST

Detected device:
mps

Model loaded:
False
Processor loaded:
False

⚠️ CLIP MODEL WAS NOT LOADED
Zyra may be using deterministic fallback mode.


In [9]:
# ============================================================
# ZYRA V1 — CHECK ML ENCODING SETTING
# ============================================================

from zyra.product_encoder.config.settings import get_product_settings

settings = get_product_settings()

print("=" * 70)
print("ZYRA V1 — ML ENCODING CONFIGURATION")
print("=" * 70)

print("ENABLE_ML_ENCODING:")
print(getattr(settings, "ENABLE_ML_ENCODING", "NOT FOUND"))

print("\nSettings object:")
print(settings)

print("=" * 70)

ZYRA V1 — ML ENCODING CONFIGURATION
ENABLE_ML_ENCODING:
False

Settings object:
SERVICE_NAME='zyra-product-encoder' ENVIRONMENT='development' PORT=8000 HOST='0.0.0.0' LOG_LEVEL='INFO' PRODUCT_ENCODER_VERSION='v0-foundation' SCHEMA_VERSION='v1' SPRING_BOOT_BASE_URL='http://localhost:8081' SPRING_BOOT_TIMEOUT_SECONDS=10.0 MODELS_DIR='/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/models' POSTGRES_HOST='db.grduuzsxlugnmymgojky.supabase.co' POSTGRES_PORT=5432 POSTGRES_USER='postgres' POSTGRES_PASSWORD='Saketh@20056' POSTGRES_DB='postgres' POSTGRES_MIN_POOL_SIZE=1 POSTGRES_MAX_POOL_SIZE=10 POSTGRES_TIMEOUT_SECONDS=10.0 QDRANT_URL=None QDRANT_HOST='localhost' QDRANT_PORT=6333 QDRANT_API_KEY=None QDRANT_COLLECTION_NAME='zyra_product_embeddings' QDRANT_VECTOR_DIMENSION=662 QDRANT_USE_IN_MEMORY=False ENABLE_ML_ENCODING=False DEFAULT_VISUAL_WEIGHT=0.45 DEFAULT_TEXT_WEIGHT=0.35 DEFAULT_ATTRIBUTE_WEIGHT=0.2 FUSION_COMMON_DIMENSION=512 FUSION_FINAL_DIMENSION=662 FUSION_STRATE

In [10]:
# ============================================================
# ZYRA V1 — LOCATE ML ENCODING CONFIGURATION
# ============================================================

from pathlib import Path

root = Path.cwd()

print("=" * 70)
print("SEARCHING FOR ENABLE_ML_ENCODING")
print("=" * 70)

for path in root.rglob("*"):

    if not path.is_file():
        continue

    if ".venv" in str(path) or ".env" in str(path):
        continue

    if path.suffix not in [".py", ".env", ".yaml", ".yml", ".json"]:
        continue

    try:
        text = path.read_text(errors="ignore")

        if "ENABLE_ML_ENCODING" in text:

            print("\nFOUND:")
            print(path)

            for line_no, line in enumerate(text.splitlines(), 1):

                if "ENABLE_ML_ENCODING" in line:
                    print(f"{line_no}: {line}")

    except Exception:
        pass

print("\n" + "=" * 70)
print("SEARCH COMPLETE")
print("=" * 70)

SEARCHING FOR ENABLE_ML_ENCODING

FOUND:
/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/config/settings.py
56:     ENABLE_ML_ENCODING: bool = False

FOUND:
/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/tests/test_product_config.py
23:     assert settings.ENABLE_ML_ENCODING is False

FOUND:
/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/text_encoder/model_manager.py
57:         Returns (None, None) if transformers/weights are unavailable or ENABLE_ML_ENCODING is False.
63:         if not getattr(settings, "ENABLE_ML_ENCODING", False):

FOUND:
/Users/saketh/Desktop/Projects/weavly/core-model/zyra/product_encoder/image_encoder/model_manager.py
63:         if not getattr(settings, "ENABLE_ML_ENCODING", False):

SEARCH COMPLETE


In [1]:
# ============================================================
# ZYRA V1 — VERIFY REAL CLIP ON M5
# ============================================================

from zyra.product_encoder.image_encoder.model_manager import (
    ProductVisionModelManager
)

print("=" * 70)
print("ZYRA V1 — REAL CLIP / M5 VERIFICATION")
print("=" * 70)

manager = ProductVisionModelManager()

print("\nDetected device:")
print(manager.get_device())

model, processor = manager.get_vision_model()

print("\nModel loaded:")
print(model is not None)

print("Processor loaded:")
print(processor is not None)

if model is not None:

    actual_device = next(model.parameters()).device

    print("\nActual model device:")
    print(actual_device)

    print("\nModel:")
    print(type(model))

    if actual_device.type == "mps":
        print("\n✅ REAL CLIP IS RUNNING ON M5 GPU")
    else:
        print(
            f"\n⚠️ CLIP loaded but is running on {actual_device}"
        )

else:

    print("\n❌ CLIP DID NOT LOAD")
    print("Check the model-loading error/logs.")

print("=" * 70)

ZYRA V1 — REAL CLIP / M5 VERIFICATION

Detected device:
mps


/Users/saketh/Desktop/Projects/weavly/core-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 56318.38it/s]



Model loaded:
True
Processor loaded:
True

Actual model device:
mps:0

Model:
<class 'transformers.models.clip.modeling_clip.CLIPModel'>

✅ REAL CLIP IS RUNNING ON M5 GPU


In [2]:
# ============================================================
# ZYRA V1 — P8 M5 GPU BENCHMARK
# REAL CLIP + MPS
# ============================================================

import time
import numpy as np
import pandas as pd

from zyra.product_encoder.ingestion.router import (
    ProductTextEncoderInput,
    ProductAttributeEncoderInput,
    ProductAttributes,
    ProductImageEncoderInput,
    ProductImageInput,
)

from zyra.product_encoder.text_encoder.encoder import ProductTextEncoder
from zyra.product_encoder.attribute_encoder.encoder import ProductAttributeEncoder
from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder

from zyra.product_encoder.insights.service import (
    ProductInsightAggregationService
)

from zyra.product_encoder.fusion import (
    ProductFusionService,
    FusionWeightsConfig
)

print("=" * 70)
print("ZYRA V1 — P8 M5 GPU BENCHMARK")
print("=" * 70)

# ============================================================
# CONFIG
# ============================================================

BENCHMARK_SIZE = 100

benchmark_df = df.head(BENCHMARK_SIZE).copy()

print("\nProducts:", len(benchmark_df))
print("Total catalog:", len(df))

# ============================================================
# VERIFY MPS
# ============================================================

import torch

print("\nGPU:")
print("MPS available:", torch.backends.mps.is_available())

assert torch.backends.mps.is_available(), \
    "MPS is not available."

print("✅ M5 / MPS available")

# ============================================================
# INITIALIZE
# ============================================================

print("\nInitializing pipeline...")

text_encoder = ProductTextEncoder()
attribute_encoder = ProductAttributeEncoder()
image_encoder = ProductImageEncoder()

insight_service = ProductInsightAggregationService()

fusion_service = ProductFusionService(
    weights_config=FusionWeightsConfig(
        visualWeight=0.45,
        textWeight=0.35,
        attributeWeight=0.20
    )
)

# Force initialization of the actual CLIP model
model, processor = image_encoder.backbone.model_manager.get_vision_model()

assert model is not None, \
    "CLIP model failed to load."

actual_device = next(model.parameters()).device

print("CLIP device:", actual_device)

assert actual_device.type == "mps", \
    f"Expected MPS but got {actual_device}"

print("✅ REAL CLIP CONFIRMED ON M5 GPU")

# ============================================================
# HELPERS
# ============================================================

def parse_images(value):

    if isinstance(value, list):
        return value

    if value is None:
        return []

    return [
        x.strip()
        for x in str(value).split("~")
        if x.strip()
    ]


def make_text_input(row):

    return ProductTextEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        description=str(row["description"]),
        brand=str(row["brand"]),
        category="fashion",
        styles=[],
        occasions=[],
        seasons=[],
        tags=[]
    )


def make_attribute_input(row):

    attrs = {}

    for col in attribute_columns:

        if col.startswith("attr_") and row.get(col, 0) == 1:
            attrs[col.replace("attr_", "")] = True

    return ProductAttributeEncoderInput(
        productId=str(row["sku"]),
        category="fashion",
        attributes=ProductAttributes(
            customAttributes=attrs
        ),
        rawAttributes=attrs
    )


def make_image_input(row):

    urls = parse_images(row["images"])

    images = [
        ProductImageInput(
            imageUrl=url,
            viewType="front",
            sortOrder=i
        )
        for i, url in enumerate(urls)
    ]

    return ProductImageEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        images=images
    )


# ============================================================
# PROCESS
# ============================================================

results = []
errors = []

benchmark_start = time.perf_counter()

print("\n" + "-" * 70)
print("PROCESSING")
print("-" * 70)

for idx, (_, row) in enumerate(
    benchmark_df.iterrows(),
    start=1
):

    product_id = str(row["sku"])

    start = time.perf_counter()

    try:

        # ----------------------------------------------------
        # TEXT
        # ----------------------------------------------------

        text_output = text_encoder.encode(
            make_text_input(row)
        )

        # ----------------------------------------------------
        # ATTRIBUTE
        # ----------------------------------------------------

        attribute_output = attribute_encoder.encode(
            make_attribute_input(row)
        )

        # ----------------------------------------------------
        # IMAGE — REAL CLIP / MPS
        # ----------------------------------------------------

        image_output = image_encoder.encode(
            make_image_input(row)
        )

        # ----------------------------------------------------
        # P5
        # ----------------------------------------------------

        profile = insight_service.aggregate(
            visual=image_output,
            text=text_output,
            attribute=attribute_output
        )

        # ----------------------------------------------------
        # P6
        # ----------------------------------------------------

        fused = fusion_service.fuse(
            profile=profile,
            visual=image_output,
            text=text_output,
            attribute=attribute_output
        )

        # ----------------------------------------------------
        # VALIDATE
        # ----------------------------------------------------

        vector = np.asarray(
            fused.unifiedEmbedding,
            dtype=np.float32
        )

        assert vector.shape == (662,)
        assert np.isfinite(vector).all()
        assert np.linalg.norm(vector) > 0

        elapsed = time.perf_counter() - start

        results.append({
            "product_id": product_id,
            "embedding_dim": len(vector),
            "image_count": image_output.successfulImageCount,
            "confidence": fused.confidence,
            "elapsed_seconds": elapsed
        })

        if idx % 10 == 0:

            elapsed_total = time.perf_counter() - benchmark_start

            rate = idx / elapsed_total

            remaining = BENCHMARK_SIZE - idx

            eta = remaining / rate

            print(
                f"Processed {idx}/{BENCHMARK_SIZE} "
                f"| {rate:.2f} products/sec "
                f"| ETA {eta:.1f}s"
            )

    except Exception as e:

        errors.append({
            "product_id": product_id,
            "error": str(e)
        })

        print(
            f"❌ Product {idx}: {product_id} → {e}"
        )


# ============================================================
# RESULTS
# ============================================================

total_time = time.perf_counter() - benchmark_start

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("M5 GPU BENCHMARK RESULTS")
print("=" * 70)

print("\nSuccessful:", len(results))
print("Failed:", len(errors))

print(
    f"\nTotal time: {total_time:.2f} seconds"
)

if len(results) > 0:

    avg_time = results_df[
        "elapsed_seconds"
    ].mean()

    median_time = results_df[
        "elapsed_seconds"
    ].median()

    rate = len(results) / total_time

    print(
        f"Average/product: {avg_time:.2f} seconds"
    )

    print(
        f"Median/product: {median_time:.2f} seconds"
    )

    print(
        f"Throughput: {rate:.2f} products/sec"
    )

    # --------------------------------------------------------
    # FULL CATALOG ETA
    # --------------------------------------------------------

    remaining_products = len(df)

    estimated_seconds = (
        remaining_products / rate
    )

    print("\n" + "-" * 70)
    print("FULL CATALOG ESTIMATE")
    print("-" * 70)

    print(
        f"Products: {remaining_products:,}"
    )

    print(
        f"Estimated time: "
        f"{estimated_seconds / 60:.1f} minutes"
    )

    print(
        f"Estimated time: "
        f"{estimated_seconds / 3600:.2f} hours"
    )

    print("\nEmbedding dimension:")
    print(
        results_df["embedding_dim"].unique()
    )

    print("\nImages/product:")
    print(
        results_df["image_count"].describe()
    )

# ============================================================
# ERRORS
# ============================================================

if errors:

    print("\n" + "=" * 70)
    print("ERRORS")
    print("=" * 70)

    for error in errors[:20]:
        print(error)

# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)

if (
    len(results) == BENCHMARK_SIZE
    and len(errors) == 0
):

    print("✅ P8 M5 GPU BENCHMARK PASSED")

else:

    print("⚠️ P8 M5 BENCHMARK NEEDS INVESTIGATION")

print("=" * 70)

ZYRA V1 — P8 M5 GPU BENCHMARK


NameError: name 'df' is not defined

In [3]:
# ============================================================
# ZYRA V1 — P8 NOTEBOOK DATASET SETUP
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("ZYRA V1 — LOADING PRODUCT DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATASET_PATH = Path(
    "/Users/saketh/Desktop/Projects/weavly/data/raw/"
    "myntra-fashion-products/Myntra_fashion_products.csv"
)

df = pd.read_csv(DATASET_PATH)

print("\n✅ Dataset loaded")
print("Path:", DATASET_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# Product feature extraction
# ------------------------------------------------------------

# Load the already-generated catalog if available
CATALOG_PATH = Path(
    "/Users/saketh/Desktop/Projects/weavly/core-model/"
    "data/recommendation/product_catalog.pkl"
)

if CATALOG_PATH.exists():

    products = pd.read_pickle(CATALOG_PATH)

    print("\n✅ Product catalog loaded")
    print("Catalog shape:", products.shape)

else:

    print("\n⚠️ Product catalog not found.")
    print("We need to recreate the feature-extraction variables.")

# ------------------------------------------------------------
# Attribute columns
# ------------------------------------------------------------

if "products" in globals():

    attribute_columns = [
        col for col in products.columns
        if col.startswith("attr_")
    ]

    print(
        "\nAttribute columns:",
        len(attribute_columns)
    )

else:

    attribute_columns = []

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("P8 SETUP CHECK")
print("=" * 70)

print("df:", df.shape)

if "products" in globals():
    print("products:", products.shape)

print(
    "attribute_columns:",
    len(attribute_columns)
)

assert len(df) == 12491, \
    f"Expected 12491 products, got {len(df)}"

print("\n✅ P8 DATASET SETUP COMPLETE")

ZYRA V1 — LOADING PRODUCT DATASET

✅ Dataset loaded
Path: /Users/saketh/Desktop/Projects/weavly/data/raw/myntra-fashion-products/Myntra_fashion_products.csv
Shape: (12491, 10)
Columns: ['name', 'sku', 'mpn', 'price', 'in_stock', 'currency', 'brand', 'description', 'images', 'gender']

✅ Product catalog loaded
Catalog shape: (12491, 80)

Attribute columns: 55

P8 SETUP CHECK
df: (12491, 10)
products: (12491, 80)
attribute_columns: 55

✅ P8 DATASET SETUP COMPLETE


In [4]:
# ============================================================
# ZYRA V1 — P8 NOTEBOOK DATASET SETUP
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 70)
print("ZYRA V1 — LOADING PRODUCT DATASET")
print("=" * 70)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATASET_PATH = Path(
    "/Users/saketh/Desktop/Projects/weavly/data/raw/"
    "myntra-fashion-products/Myntra_fashion_products.csv"
)

df = pd.read_csv(DATASET_PATH)

print("\n✅ Dataset loaded")
print("Path:", DATASET_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# Product feature extraction
# ------------------------------------------------------------

# Load the already-generated catalog if available
CATALOG_PATH = Path(
    "/Users/saketh/Desktop/Projects/weavly/core-model/"
    "data/recommendation/product_catalog.pkl"
)

if CATALOG_PATH.exists():

    products = pd.read_pickle(CATALOG_PATH)

    print("\n✅ Product catalog loaded")
    print("Catalog shape:", products.shape)

else:

    print("\n⚠️ Product catalog not found.")
    print("We need to recreate the feature-extraction variables.")

# ------------------------------------------------------------
# Attribute columns
# ------------------------------------------------------------

if "products" in globals():

    attribute_columns = [
        col for col in products.columns
        if col.startswith("attr_")
    ]

    print(
        "\nAttribute columns:",
        len(attribute_columns)
    )

else:

    attribute_columns = []

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("P8 SETUP CHECK")
print("=" * 70)

print("df:", df.shape)

if "products" in globals():
    print("products:", products.shape)

print(
    "attribute_columns:",
    len(attribute_columns)
)

assert len(df) == 12491, \
    f"Expected 12491 products, got {len(df)}"

print("\n✅ P8 DATASET SETUP COMPLETE")

ZYRA V1 — LOADING PRODUCT DATASET

✅ Dataset loaded
Path: /Users/saketh/Desktop/Projects/weavly/data/raw/myntra-fashion-products/Myntra_fashion_products.csv
Shape: (12491, 10)
Columns: ['name', 'sku', 'mpn', 'price', 'in_stock', 'currency', 'brand', 'description', 'images', 'gender']

✅ Product catalog loaded
Catalog shape: (12491, 80)

Attribute columns: 55

P8 SETUP CHECK
df: (12491, 10)
products: (12491, 80)
attribute_columns: 55

✅ P8 DATASET SETUP COMPLETE


In [5]:
# ============================================================
# ZYRA V1 — P8 M5 GPU BENCHMARK
# REAL CLIP + APPLE MPS
# ============================================================

import time
import numpy as np
import pandas as pd
import torch

from zyra.product_encoder.ingestion.router import (
    ProductTextEncoderInput,
    ProductAttributeEncoderInput,
    ProductAttributes,
    ProductImageEncoderInput,
    ProductImageInput,
)

from zyra.product_encoder.text_encoder.encoder import ProductTextEncoder
from zyra.product_encoder.attribute_encoder.encoder import ProductAttributeEncoder
from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder

from zyra.product_encoder.insights.service import (
    ProductInsightAggregationService
)

from zyra.product_encoder.fusion import (
    ProductFusionService,
    FusionWeightsConfig
)

print("=" * 70)
print("ZYRA V1 — P8 M5 GPU BENCHMARK")
print("=" * 70)


# ============================================================
# CONFIGURATION
# ============================================================

BENCHMARK_SIZE = 100

benchmark_df = df.head(BENCHMARK_SIZE).copy()

print("\nProducts:", len(benchmark_df))
print("Total catalog:", len(df))


# ============================================================
# VERIFY M5 GPU
# ============================================================

print("\nGPU CHECK")
print("-" * 70)

print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

assert torch.backends.mps.is_available(), \
    "Apple MPS is not available."

print("✅ M5 / MPS available")


# ============================================================
# INITIALIZE ENCODERS
# ============================================================

print("\nInitializing pipeline...")

text_encoder = ProductTextEncoder()
attribute_encoder = ProductAttributeEncoder()
image_encoder = ProductImageEncoder()

insight_service = ProductInsightAggregationService()

fusion_service = ProductFusionService(
    weights_config=FusionWeightsConfig(
        visualWeight=0.45,
        textWeight=0.35,
        attributeWeight=0.20
    )
)

print("✅ Text encoder initialized")
print("✅ Attribute encoder initialized")
print("✅ Image encoder initialized")
print("✅ P5 Insight service initialized")
print("✅ P6 Fusion service initialized")


# ============================================================
# FORCE CLIP LOAD + VERIFY DEVICE
# ============================================================

print("\nLoading CLIP...")

model, processor = (
    image_encoder
    .backbone
    .model_manager
    .get_vision_model()
)

assert model is not None, \
    "CLIP model failed to load."

assert processor is not None, \
    "CLIP processor failed to load."

actual_device = next(model.parameters()).device

print("CLIP device:", actual_device)
print("CLIP model:", type(model))

assert actual_device.type == "mps", \
    f"Expected MPS but got {actual_device}"

print("✅ REAL CLIP CONFIRMED ON M5 GPU")


# ============================================================
# HELPERS
# ============================================================

def parse_images(value):

    if isinstance(value, list):
        return value

    if value is None:
        return []

    return [
        x.strip()
        for x in str(value).split("~")
        if x.strip()
    ]


def make_text_input(row):

    return ProductTextEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        description=str(row["description"]),
        brand=str(row["brand"]),
        category="fashion",
        styles=[],
        occasions=[],
        seasons=[],
        tags=[]
    )


def make_attribute_input(row):

    attrs = {}

    for col in attribute_columns:

        if col.startswith("attr_"):

            if row.get(col, 0) == 1:

                attrs[col.replace("attr_", "")] = True

    return ProductAttributeEncoderInput(
        productId=str(row["sku"]),
        category="fashion",
        attributes=ProductAttributes(
            customAttributes=attrs
        ),
        rawAttributes=attrs
    )


def make_image_input(row):

    urls = parse_images(row["images"])

    images = [
        ProductImageInput(
            imageUrl=url,
            viewType="front",
            sortOrder=i
        )
        for i, url in enumerate(urls)
    ]

    return ProductImageEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        images=images
    )


# ============================================================
# PROCESS 100 PRODUCTS
# ============================================================

results = []
errors = []

benchmark_start = time.perf_counter()

print("\n" + "-" * 70)
print("PROCESSING 100 PRODUCTS")
print("-" * 70)

for idx, (_, row) in enumerate(
    benchmark_df.iterrows(),
    start=1
):

    product_id = str(row["sku"])

    product_start = time.perf_counter()

    try:

        # ----------------------------------------------------
        # TEXT ENCODER
        # ----------------------------------------------------

        text_output = text_encoder.encode(
            make_text_input(row)
        )

        # ----------------------------------------------------
        # ATTRIBUTE ENCODER
        # ----------------------------------------------------

        attribute_output = attribute_encoder.encode(
            make_attribute_input(row)
        )

        # ----------------------------------------------------
        # IMAGE ENCODER
        # REAL CLIP / MPS
        # ----------------------------------------------------

        image_output = image_encoder.encode(
            make_image_input(row)
        )

        # ----------------------------------------------------
        # P5 — INSIGHT AGGREGATION
        # ----------------------------------------------------

        profile = insight_service.aggregate(
            visual=image_output,
            text=text_output,
            attribute=attribute_output
        )

        # ----------------------------------------------------
        # P6 — 662D FUSION
        # ----------------------------------------------------

        fused = fusion_service.fuse(
            profile=profile,
            visual=image_output,
            text=text_output,
            attribute=attribute_output
        )

        # ----------------------------------------------------
        # VALIDATE FINAL VECTOR
        # ----------------------------------------------------

        vector = np.asarray(
            fused.unifiedEmbedding,
            dtype=np.float32
        )

        assert vector.shape == (662,), \
            f"Invalid shape: {vector.shape}"

        assert np.isfinite(vector).all(), \
            "Embedding contains NaN/Inf"

        norm = np.linalg.norm(vector)

        assert norm > 0, \
            "Zero embedding"

        elapsed = time.perf_counter() - product_start

        results.append({
            "product_id": product_id,
            "embedding_dim": len(vector),
            "image_count": image_output.successfulImageCount,
            "confidence": fused.confidence,
            "elapsed_seconds": elapsed
        })

        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if idx % 10 == 0:

            elapsed_total = (
                time.perf_counter()
                - benchmark_start
            )

            rate = idx / elapsed_total

            remaining = BENCHMARK_SIZE - idx

            eta_seconds = (
                remaining / rate
                if rate > 0
                else 0
            )

            print(
                f"Processed {idx}/{BENCHMARK_SIZE} "
                f"| {rate:.2f} products/sec "
                f"| ETA {eta_seconds:.1f}s"
            )

    except Exception as e:

        errors.append({
            "product_id": product_id,
            "error": str(e)
        })

        print(
            f"❌ Product {idx}/{BENCHMARK_SIZE} "
            f"| {product_id} "
            f"| {e}"
        )


# ============================================================
# RESULTS
# ============================================================

total_time = (
    time.perf_counter()
    - benchmark_start
)

results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("M5 GPU BENCHMARK RESULTS")
print("=" * 70)

print("\nSuccessful:", len(results))
print("Failed:", len(errors))

print(
    f"\nTotal time: {total_time:.2f} seconds"
)


if len(results) > 0:

    avg_time = results_df[
        "elapsed_seconds"
    ].mean()

    median_time = results_df[
        "elapsed_seconds"
    ].median()

    throughput = (
        len(results)
        / total_time
    )

    print(
        f"Average/product: {avg_time:.2f} seconds"
    )

    print(
        f"Median/product: {median_time:.2f} seconds"
    )

    print(
        f"Throughput: {throughput:.2f} products/sec"
    )


    # ========================================================
    # FULL CATALOG ETA
    # ========================================================

    estimated_seconds = (
        len(df)
        / throughput
    )

    print("\n" + "-" * 70)
    print("FULL CATALOG ESTIMATE")
    print("-" * 70)

    print(
        f"Products: {len(df):,}"
    )

    print(
        f"Estimated time: "
        f"{estimated_seconds / 60:.1f} minutes"
    )

    print(
        f"Estimated time: "
        f"{estimated_seconds / 3600:.2f} hours"
    )

    print("\nEmbedding dimensions:")

    print(
        results_df[
            "embedding_dim"
        ].unique()
    )

    print("\nImages/product:")

    print(
        results_df[
            "image_count"
        ].describe()
    )

    print("\nConfidence:")

    print(
        results_df[
            "confidence"
        ].describe()
    )


# ============================================================
# ERRORS
# ============================================================

if errors:

    print("\n" + "=" * 70)
    print("ERRORS")
    print("=" * 70)

    for error in errors[:20]:
        print(error)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)

if (
    len(results) == BENCHMARK_SIZE
    and len(errors) == 0
):

    print("✅ P8 M5 GPU BENCHMARK PASSED")

else:

    print("⚠️ P8 M5 GPU BENCHMARK NEEDS INVESTIGATION")

print("=" * 70)

ZYRA V1 — P8 M5 GPU BENCHMARK

Products: 100
Total catalog: 12491

GPU CHECK
----------------------------------------------------------------------
MPS built: True
MPS available: True
✅ M5 / MPS available

Initializing pipeline...
✅ Text encoder initialized
✅ Attribute encoder initialized
✅ Image encoder initialized
✅ P5 Insight service initialized
✅ P6 Fusion service initialized

Loading CLIP...


Loading weights: 100%|████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 47273.82it/s]


CLIP device: mps:0
CLIP model: <class 'transformers.models.clip.modeling_clip.CLIPModel'>
✅ REAL CLIP CONFIRMED ON M5 GPU

----------------------------------------------------------------------
PROCESSING 100 PRODUCTS
----------------------------------------------------------------------


Loading weights: 100%|████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 42965.93it/s]
[transformers] CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias   

Processed 10/100 | 0.59 products/sec | ETA 151.6s
Processed 20/100 | 0.59 products/sec | ETA 135.4s
Processed 30/100 | 0.67 products/sec | ETA 104.7s
Processed 40/100 | 0.71 products/sec | ETA 84.2s
Processed 50/100 | 0.74 products/sec | ETA 67.5s
Processed 60/100 | 0.77 products/sec | ETA 51.7s
Processed 70/100 | 0.79 products/sec | ETA 38.0s
Processed 80/100 | 0.81 products/sec | ETA 24.6s
Processed 90/100 | 0.80 products/sec | ETA 12.5s
Processed 100/100 | 0.76 products/sec | ETA 0.0s

M5 GPU BENCHMARK RESULTS

Successful: 100
Failed: 0

Total time: 131.11 seconds
Average/product: 1.31 seconds
Median/product: 0.91 seconds
Throughput: 0.76 products/sec

----------------------------------------------------------------------
FULL CATALOG ESTIMATE
----------------------------------------------------------------------
Products: 12,491
Estimated time: 273.0 minutes
Estimated time: 4.55 hours

Embedding dimensions:
[662]

Images/product:
count    100.000000
mean       4.760000
std        1

In [6]:
# ============================================================
# ZYRA V1 — P8 OPTIMIZATION
# PHASE 1: INSPECT CLIP BATCHING PATH
# ============================================================

import inspect

from zyra.product_encoder.image_encoder.vision_backbone import (
    ProductVisionBackbone
)

print("=" * 70)
print("ZYRA V1 — CLIP BATCHING INSPECTION")
print("=" * 70)

print("\nMethods:")
for name in dir(ProductVisionBackbone):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("EXTRACT REPRESENTATION SOURCE")
print("=" * 70)

print(
    inspect.getsource(
        ProductVisionBackbone.extract_representation_and_insights
    )
)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — CLIP BATCHING INSPECTION

Methods:
extract_representation_and_insights

EXTRACT REPRESENTATION SOURCE
    def extract_representation_and_insights(
        self,
        image: Image.Image,
        view_type: str = "front",
    ) -> Tuple[List[float], VisualInsights, Dict[str, Any]]:
        """
        Extract 512-dim visual embedding and fashion insights from a single product image.
        Returns (embedding, visual_insights, metadata).
        """
        model, processor = self.model_manager.get_vision_model()
        colors = self.color_extractor.extract_colors(image)

        if model is not None and processor is not None:
            return self._run_clip_inference(image, view_type, model, processor, colors)
        else:
            return self._run_deterministic_heuristic_inference(image, view_type, colors)


INSPECTION COMPLETE


In [7]:
# ============================================================
# ZYRA V1 — INSPECT SINGLE IMAGE CLIP INFERENCE
# ============================================================

import inspect

from zyra.product_encoder.image_encoder.vision_backbone import (
    ProductVisionBackbone
)

print("=" * 70)
print("ZYRA V1 — SINGLE IMAGE CLIP INFERENCE INSPECTION")
print("=" * 70)

print("\nCLIP INFERENCE SOURCE")
print("-" * 70)

print(
    inspect.getsource(
        ProductVisionBackbone._run_clip_inference
    )
)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — SINGLE IMAGE CLIP INFERENCE INSPECTION

CLIP INFERENCE SOURCE
----------------------------------------------------------------------
    def _run_clip_inference(
        self,
        image: Image.Image,
        view_type: str,
        model: Any,
        processor: Any,
        colors: List[ConfidenceScore],
    ) -> Tuple[List[float], VisualInsights, Dict[str, Any]]:
        """Inference using loaded CLIP vision model."""
        device = self.model_manager.get_device()

        # Build prompt catalog
        garment_prompts = [f"a product photo of a {g.lower()}" for g in TAXONOMY_GARMENTS]
        pattern_prompts = [f"a photo of {p.lower()} patterned clothing" for p in TAXONOMY_PATTERNS]
        fit_prompts = [f"a photo of {f.lower()} fit apparel" for f in TAXONOMY_FITS]
        neckline_prompts = [f"a photo of clothing with {n.lower()}" for n in TAXONOMY_NECKLINES]
        sleeve_prompts = [f"a photo of {s.lower()} clothing" for s in TAXONOMY_SLEEVES]
        length_promp

In [10]:
# ============================================================
# ZYRA V1 — P8 OPTIMIZATION
# BATCHED CLIP + CACHED TEXT PROMPTS
# M5 / MPS
#
# BENCHMARK ONLY
# ============================================================

import time
import numpy as np
import torch
import torch.nn.functional as F

from zyra.product_encoder.image_encoder.model_manager import (
    ProductVisionModelManager
)

from zyra.product_encoder.image_encoder.vision_backbone import (
    TAXONOMY_GARMENTS,
    TAXONOMY_PATTERNS,
    TAXONOMY_FITS,
    TAXONOMY_NECKLINES,
    TAXONOMY_SLEEVES,
    TAXONOMY_LENGTHS,
    TAXONOMY_DETAILS,
)

from zyra.product_encoder.image_encoder.encoder import (
    ProductImageEncoder
)

print("=" * 70)
print("ZYRA V1 — BATCHED CLIP BENCHMARK")
print("=" * 70)


# ============================================================
# CONFIG
# ============================================================

BATCH_SIZE = 16

print("\nBatch size:", BATCH_SIZE)


# ============================================================
# LOAD CLIP
# ============================================================

manager = ProductVisionModelManager()

model, processor = manager.get_vision_model()

assert model is not None
assert processor is not None

device = manager.get_device()

print("\nDevice:", device)
print("Model:", type(model))

assert device.type == "mps"

print("✅ REAL CLIP RUNNING ON M5 MPS")


# ============================================================
# BUILD EXACT SAME PROMPTS
# ============================================================

garment_prompts = [
    f"a product photo of a {g.lower()}"
    for g in TAXONOMY_GARMENTS
]

pattern_prompts = [
    f"a photo of {p.lower()} patterned clothing"
    for p in TAXONOMY_PATTERNS
]

fit_prompts = [
    f"a photo of {f.lower()} fit apparel"
    for f in TAXONOMY_FITS
]

neckline_prompts = [
    f"a photo of clothing with {n.lower()}"
    for n in TAXONOMY_NECKLINES
]

sleeve_prompts = [
    f"a photo of {s.lower()} clothing"
    for s in TAXONOMY_SLEEVES
]

length_prompts = [
    f"a photo of {l.lower()} clothing"
    for l in TAXONOMY_LENGTHS
]

detail_prompts = [
    f"a photo of clothing featuring {d.lower()}"
    for d in TAXONOMY_DETAILS
]

all_prompts = (
    garment_prompts
    + pattern_prompts
    + fit_prompts
    + neckline_prompts
    + sleeve_prompts
    + length_prompts
    + detail_prompts
)

print("\nTotal prompts:", len(all_prompts))


# ============================================================
# CACHE TEXT EMBEDDINGS
#
# IMPORTANT:
# Do NOT call model(**text_only_inputs).
# New Transformers CLIP forward expects image inputs too.
#
# We therefore explicitly run the text tower.
# ============================================================

print("\nComputing cached text embeddings...")

prompt_inputs = processor(
    text=all_prompts,
    return_tensors="pt",
    padding=True,
)

input_ids = prompt_inputs["input_ids"].to(device)
attention_mask = prompt_inputs["attention_mask"].to(device)

with torch.no_grad():

    text_outputs = model.text_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        return_dict=True,
    )

    pooled_text = text_outputs.pooler_output

    text_embeds = model.text_projection(
        pooled_text
    )

    text_embeds = F.normalize(
        text_embeds,
        p=2,
        dim=-1
    )

if device.type == "mps":
    torch.mps.synchronize()

print(
    "Cached text embedding shape:",
    tuple(text_embeds.shape)
)

assert text_embeds.ndim == 2
assert text_embeds.shape[1] == 512

print("✅ TEXT PROMPTS CACHED ONCE")


# ============================================================
# INITIALIZE IMAGE ENCODER
# ============================================================

image_encoder = ProductImageEncoder()


# ============================================================
# LOAD 100 PRODUCTS
# ============================================================

benchmark_df = df.head(100).copy()

image_records = []

print("\nLoading images...")

for _, row in benchmark_df.iterrows():

    urls = parse_images(row["images"])

    for image_url in urls:

        try:

            pil_img, error = (
                image_encoder
                .loader
                .load_image_sync(image_url)
            )

            if pil_img is not None and error is None:

                image_records.append({
                    "product_id": str(row["sku"]),
                    "image_url": image_url,
                    "image": pil_img
                })

        except Exception:
            pass


print(
    "Successfully loaded images:",
    len(image_records)
)

assert len(image_records) > 0


# ============================================================
# BATCH IMAGE INFERENCE
# ============================================================

print("\n" + "-" * 70)
print("RUNNING BATCHED IMAGE INFERENCE")
print("-" * 70)

image_embeddings = []

batch_times = []

total_start = time.perf_counter()


for start_idx in range(
    0,
    len(image_records),
    BATCH_SIZE
):

    batch_records = image_records[
        start_idx:start_idx + BATCH_SIZE
    ]

    images = [
        record["image"]
        for record in batch_records
    ]

    batch_start = time.perf_counter()


    # --------------------------------------------------------
    # IMAGE PREPROCESSING
    # --------------------------------------------------------

    image_inputs = processor(
        images=images,
        return_tensors="pt"
    )

    pixel_values = image_inputs[
        "pixel_values"
    ].to(device)


    # --------------------------------------------------------
    # BATCH CLIP IMAGE EMBEDDINGS
    # --------------------------------------------------------

    with torch.no_grad():

        vision_outputs = model.vision_model(
            pixel_values=pixel_values,
            return_dict=True,
        )

        pooled_image = (
            vision_outputs.pooler_output
        )

        embeds = model.visual_projection(
            pooled_image
        )

        embeds = F.normalize(
            embeds,
            p=2,
            dim=-1
        )


        # ----------------------------------------------------
        # SAME COSINE SIMILARITY LOGIC
        # ----------------------------------------------------

        logits = torch.matmul(
            embeds,
            text_embeds.T
        )


    # --------------------------------------------------------
    # SYNCHRONIZE MPS FOR ACCURATE TIMING
    # --------------------------------------------------------

    if device.type == "mps":
        torch.mps.synchronize()


    batch_elapsed = (
        time.perf_counter()
        - batch_start
    )

    batch_times.append(
        batch_elapsed
    )


    # --------------------------------------------------------
    # MOVE EMBEDDINGS TO CPU
    # --------------------------------------------------------

    batch_numpy = (
        embeds
        .detach()
        .cpu()
        .numpy()
    )

    image_embeddings.append(
        batch_numpy
    )


    processed = min(
        start_idx + BATCH_SIZE,
        len(image_records)
    )

    print(
        f"Processed images: "
        f"{processed}/{len(image_records)} "
        f"| batch: {batch_elapsed:.3f}s"
    )


# ============================================================
# COMBINE
# ============================================================

total_elapsed = (
    time.perf_counter()
    - total_start
)

image_embeddings = np.vstack(
    image_embeddings
)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BATCHED CLIP RESULTS")
print("=" * 70)

print(
    "\nImage embeddings shape:",
    image_embeddings.shape
)

print(
    "Total images:",
    len(image_records)
)

print(
    "Total inference time:",
    f"{total_elapsed:.2f}s"
)

print(
    "Images/sec:",
    f"{len(image_records) / total_elapsed:.2f}"
)

print(
    "Average batch time:",
    f"{np.mean(batch_times):.3f}s"
)

print(
    "Min batch time:",
    f"{np.min(batch_times):.3f}s"
)

print(
    "Max batch time:",
    f"{np.max(batch_times):.3f}s"
)


# ============================================================
# NUMERICAL VALIDATION
# ============================================================

print("\n" + "-" * 70)
print("NUMERICAL VALIDATION")
print("-" * 70)

nan_count = np.isnan(
    image_embeddings
).sum()

inf_count = np.isinf(
    image_embeddings
).sum()

norms = np.linalg.norm(
    image_embeddings,
    axis=1
)

print("NaN:", nan_count)
print("Inf:", inf_count)

print(
    "Finite:",
    np.isfinite(image_embeddings).all()
)

print(
    "Mean L2 norm:",
    norms.mean()
)

print(
    "Min L2 norm:",
    norms.min()
)

print(
    "Max L2 norm:",
    norms.max()
)

print(
    "Zero vectors:",
    np.sum(norms == 0)
)


# ============================================================
# FINAL
# ============================================================

assert image_embeddings.shape[1] == 512
assert nan_count == 0
assert inf_count == 0
assert np.isfinite(image_embeddings).all()
assert np.all(norms > 0)

print("\n" + "=" * 70)
print("✅ BATCHED CLIP BENCHMARK PASSED")
print("=" * 70)

ZYRA V1 — BATCHED CLIP BENCHMARK

Batch size: 16


Loading weights: 100%|████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 43182.08it/s]



Device: mps
Model: <class 'transformers.models.clip.modeling_clip.CLIPModel'>
✅ REAL CLIP RUNNING ON M5 MPS

Total prompts: 50

Computing cached text embeddings...
Cached text embedding shape: (50, 512)
✅ TEXT PROMPTS CACHED ONCE

Loading images...
Successfully loaded images: 476

----------------------------------------------------------------------
RUNNING BATCHED IMAGE INFERENCE
----------------------------------------------------------------------
Processed images: 16/476 | batch: 1.067s
Processed images: 32/476 | batch: 0.196s
Processed images: 48/476 | batch: 0.167s
Processed images: 64/476 | batch: 0.164s
Processed images: 80/476 | batch: 0.151s
Processed images: 96/476 | batch: 0.156s
Processed images: 112/476 | batch: 0.169s
Processed images: 128/476 | batch: 0.158s
Processed images: 144/476 | batch: 0.144s
Processed images: 160/476 | batch: 0.155s
Processed images: 176/476 | batch: 0.135s
Processed images: 192/476 | batch: 0.159s
Processed images: 208/476 | batch: 0.184s
Pro

In [11]:
# ============================================================
# ZYRA V1 — P8 OPTIMIZATION
# INSPECT IMAGE AGGREGATION INTERFACE
# ============================================================

import inspect

from zyra.product_encoder.image_encoder.aggregator import (
    MultiImageVisualAggregator
)

from zyra.product_encoder.image_encoder.encoder import (
    ProductImageEncoder
)

print("=" * 70)
print("ZYRA V1 — P8 OPTIMIZATION INSPECTION")
print("=" * 70)

print("\nIMAGE ENCODER")
print("-" * 70)

print(inspect.getsource(ProductImageEncoder))

print("\n" + "=" * 70)
print("MULTI-IMAGE AGGREGATOR")
print("=" * 70)

print(inspect.signature(
    MultiImageVisualAggregator.aggregate
))

print(
    inspect.getsource(
        MultiImageVisualAggregator.aggregate
    )
)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — P8 OPTIMIZATION INSPECTION

IMAGE ENCODER
----------------------------------------------------------------------
class ProductImageEncoder(ProductImageEncoderInterface):
    """
    Main implementation of the Zyra Product Image Encoder (Phase P2).
    Processes product image collections, extracts deep visual representations and insights,
    and synthesizes view-weighted product visual embeddings.
    """

    def __init__(
        self,
        loader: Optional[ProductImageLoader] = None,
        preprocessor: Optional[ProductImagePreprocessor] = None,
        backbone: Optional[ProductVisionBackbone] = None,
        aggregator: Optional[MultiImageVisualAggregator] = None,
    ) -> None:
        self.loader = loader or ProductImageLoader()
        self.preprocessor = preprocessor or ProductImagePreprocessor()
        self.backbone = backbone or ProductVisionBackbone()
        self.aggregator = aggregator or MultiImageVisualAggregator()

    async def encode_async(self, input

In [12]:
# ============================================================
# ZYRA V1 — P8 OPTIMIZATION
# FINAL BACKBONE DEPENDENCY INSPECTION
# ============================================================

import inspect

from zyra.product_encoder.image_encoder.vision_backbone import (
    ProductVisionBackbone
)

print("=" * 70)
print("ZYRA V1 — BACKBONE DEPENDENCY INSPECTION")
print("=" * 70)

source = inspect.getsource(ProductVisionBackbone)

print("\nIMPORTS / DEPENDENCIES USED BY BACKBONE")
print("-" * 70)

for line in source.splitlines()[:80]:
    print(line)

print("\n" + "=" * 70)
print("FULL BACKBONE PUBLIC METHODS")
print("=" * 70)

for name in dir(ProductVisionBackbone):
    if not name.startswith("_"):
        print(name)

print("\n" + "=" * 70)
print("FINAL INSPECTION COMPLETE")
print("=" * 70)

ZYRA V1 — BACKBONE DEPENDENCY INSPECTION

IMPORTS / DEPENDENCIES USED BY BACKBONE
----------------------------------------------------------------------
class ProductVisionBackbone:
    """
    Executes deep visual feature extraction using a pretrained vision-language backbone (CLIP).
    Extracts 512-dim dense visual vectors and evaluates zero-shot semantic similarities
    against canonical fashion taxonomies.
    """

    def __init__(
        self,
        model_manager: Optional[ProductVisionModelManager] = None,
        color_extractor: Optional[ProductColorExtractor] = None,
    ) -> None:
        self.model_manager = model_manager or ProductVisionModelManager()
        self.color_extractor = color_extractor or ProductColorExtractor()

    def extract_representation_and_insights(
        self,
        image: Image.Image,
        view_type: str = "front",
    ) -> Tuple[List[float], VisualInsights, Dict[str, Any]]:
        """
        Extract 512-dim visual embedding and fashion 

In [13]:
# ============================================================
# ZYRA V1 — P8 FULL RUN ENVIRONMENT CHECK
# ============================================================

import torch
import pandas as pd

print("=" * 70)
print("ZYRA V1 — P8 FULL CATALOG RUN")
print("=" * 70)

print("\nPyTorch:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

assert torch.backends.mps.is_available(), \
    "MPS is not available. Stop before running P8."

device = torch.device("mps")

print("Device:", device)
print("✅ Apple M5 MPS confirmed")

# Dataset
DATASET_PATH = (
    "/Users/saketh/Desktop/Projects/weavly/"
    "data/raw/myntra-fashion-products/"
    "Myntra_fashion_products.csv"
)

df = pd.read_csv(DATASET_PATH)

print("\nDataset:")
print("Products:", len(df))
print("Columns:", list(df.columns))

assert len(df) == 12491

print("✅ Full 12,491-product catalog loaded")

print("\n" + "=" * 70)
print("READY FOR FULL P8")
print("=" * 70)

ZYRA V1 — P8 FULL CATALOG RUN

PyTorch: 2.13.0
MPS built: True
MPS available: True
Device: mps
✅ Apple M5 MPS confirmed

Dataset:
Products: 12491
Columns: ['name', 'sku', 'mpn', 'price', 'in_stock', 'currency', 'brand', 'description', 'images', 'gender']
✅ Full 12,491-product catalog loaded

READY FOR FULL P8


In [1]:
# ======================================================================
# ZYRA V1 — P8 SINGLE JUPYTER BOOTSTRAP CELL
# Restores all environment state, dataset, helpers, encoders & M5 GPU
# ======================================================================

import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# 1. Environment & Path Resolution
CORE_MODEL_ROOT = Path("/Users/saketh/Desktop/Projects/weavly/core-model")
if str(CORE_MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(CORE_MODEL_ROOT))

# 2. Configuration & Live ML Activation
from zyra.product_encoder.config.settings import get_product_settings

settings = get_product_settings()
settings.ENABLE_ML_ENCODING = True

# 3. Canonical Schema & Router Imports
from zyra.product_encoder.ingestion.router import (
    ProductTextEncoderInput,
    ProductAttributeEncoderInput,
    ProductAttributes,
    ProductImageEncoderInput,
    ProductImageInput,
)

# 4. Multimodal Pipeline Components
from zyra.product_encoder.text_encoder.encoder import ProductTextEncoder
from zyra.product_encoder.attribute_encoder.encoder import ProductAttributeEncoder
from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder
from zyra.product_encoder.insights.service import ProductInsightAggregationService
from zyra.product_encoder.fusion import ProductFusionService, FusionWeightsConfig

# 5. Dataset & Catalog Ingestion
DATASET_PATH = Path("/Users/saketh/Desktop/Projects/weavly/data/raw/myntra-fashion-products/Myntra_fashion_products.csv")
df = pd.read_csv(DATASET_PATH)

CATALOG_PATH = CORE_MODEL_ROOT / "data/recommendation/product_catalog.pkl"
if CATALOG_PATH.exists():
    products = pd.read_pickle(CATALOG_PATH)
    attribute_columns = [col for col in products.columns if col.startswith("attr_")]
    if "sku" in products.columns and "sku" in df.columns:
        cols_to_merge = ["sku"] + [c for c in attribute_columns if c not in df.columns]
        df = df.merge(products[cols_to_merge], on="sku", how="left")
else:
    products = None
    attribute_columns = [col for col in df.columns if col.startswith("attr_")]

# 6. Pipeline Encoders & Services Initialization (with Optimized Batch CLIP)
text_encoder = ProductTextEncoder()
attribute_encoder = ProductAttributeEncoder()
image_encoder = ProductImageEncoder(use_batch_inference=True)
insight_service = ProductInsightAggregationService()
fusion_service = ProductFusionService(
    weights_config=FusionWeightsConfig(
        visualWeight=0.45,
        textWeight=0.35,
        attributeWeight=0.20,
    )
)

# 7. Hardware & Model Acceleration Check
model, processor = image_encoder.backbone.model_manager.get_vision_model()
actual_device = next(model.parameters()).device

# 8. Data Ingestion & Builder Helper Functions
def parse_images(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return [x.strip() for x in str(value).split("~") if x.strip()]

def build_text_encoder_input(row):
    return ProductTextEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        description=str(row.get("description", "")),
        brand=str(row.get("brand", "")),
        category="fashion",
        styles=[],
        occasions=[],
        seasons=[],
        tags=[],
    )
make_text_input = build_text_encoder_input

def build_attribute_encoder_input(row, attr_cols=None):
    cols = attr_cols if attr_cols is not None else attribute_columns
    attrs = {}
    for col in cols:
        if col.startswith("attr_"):
            val = row.get(col, 0)
            if val == 1 or val is True or val == "1":
                attrs[col.replace("attr_", "")] = True
        else:
            val = row.get(col)
            if val is not None and not (isinstance(val, float) and np.isnan(val)):
                attrs[col] = val
    return ProductAttributeEncoderInput(
        productId=str(row["sku"]),
        category="fashion",
        attributes=ProductAttributes(customAttributes=attrs),
        rawAttributes=attrs,
    )
make_attribute_input = build_attribute_encoder_input

def build_image_encoder_input(row):
    urls = parse_images(row.get("images"))
    images = [
        ProductImageInput(
            imageUrl=url,
            viewType="front" if i == 0 else "detail" if i > 2 else "on_model",
            sortOrder=i,
        )
        for i, url in enumerate(urls)
    ]
    return ProductImageEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        images=images,
    )
make_image_input = build_image_encoder_input

# 9. Verification Summary
print("=" * 70)
print("ZYRA V1 — P8 BOOTSTRAP READY")
print("=" * 70)
print()
print(f"Dataset: {len(df)} products")
print(f"Device: {actual_device.type}")
print(f"ML encoding: {settings.ENABLE_ML_ENCODING}")
print("Text encoder: READY")
print("Attribute encoder: READY")
print("Image encoder: READY")
print("P5 service: READY")
print("P6 service: READY")
print("Optimized batch CLIP: READY")
print()
print("=" * 70)
print("READY TO RUN P8")
print("=" * 70)


/Users/saketh/Desktop/Projects/weavly/core-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 66122.67it/s]


ZYRA V1 — P8 BOOTSTRAP READY

Dataset: 12491 products
Device: mps
ML encoding: True
Text encoder: READY
Attribute encoder: READY
Image encoder: READY
P5 service: READY
P6 service: READY
Optimized batch CLIP: READY

READY TO RUN P8


In [2]:
# ============================================================
# ZYRA V1 — P8 FULL CATALOG GENERATION
# CHECKPOINT + RESUME
# APPLE M5 / MPS
# ============================================================

import os
import json
import time
import numpy as np
import pandas as pd
import torch

from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder
from zyra.product_encoder.insights.service import (
    ProductInsightAggregationService
)
from zyra.product_encoder.fusion.service import ProductFusionService

print("=" * 70)
print("ZYRA V1 — P8 FULL CATALOG GENERATION")
print("=" * 70)

# ============================================================
# CONFIG
# ============================================================

OUTPUT_DIR = (
    "/Users/saketh/Desktop/Projects/weavly/"
    "core-model/p8_output"
)

CHECKPOINT_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_checkpoint.json"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_product_embeddings.jsonl"
)

ERRORS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_errors.jsonl"
)

CHECKPOINT_EVERY = 25

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nCheckpoint every:")
print(CHECKPOINT_EVERY, "products")


# ============================================================
# DEVICE
# ============================================================

assert torch.backends.mps.is_available()

device = torch.device("mps")

print("\nDevice:", device)
print("✅ M5 MPS confirmed")


# ============================================================
# INITIALIZE PIPELINE
# ============================================================

print("\nInitializing Zyra pipeline...")

image_encoder = ProductImageEncoder(
    use_batch_inference=True
)

insight_service = ProductInsightAggregationService()

fusion_service = ProductFusionService()

print("✅ Image encoder initialized")
print("✅ P5 insight service initialized")
print("✅ P6 fusion service initialized")


# ============================================================
# LOAD EXISTING CHECKPOINT
# ============================================================

processed_ids = set()

if os.path.exists(CHECKPOINT_FILE):

    print("\nExisting checkpoint found.")

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        checkpoint = json.load(f)

    processed_ids = set(
        checkpoint.get(
            "successful_product_ids",
            []
        )
    )

    print(
        "Previously completed:",
        len(processed_ids)
    )

else:

    checkpoint = {
        "successful_product_ids": [],
        "failed_product_ids": [],
        "started_at": time.time()
    }

    print("\nNo existing checkpoint.")
    print("Starting from product 1.")


# ============================================================
# LOAD EXISTING OUTPUT IDS
# ============================================================

if os.path.exists(RESULTS_FILE):

    print("\nReading existing output...")

    with open(
        RESULTS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(line)

                pid = str(
                    record["productId"]
                )

                processed_ids.add(pid)

            except Exception:
                pass

print(
    "Products already completed:",
    len(processed_ids)
)


# ============================================================
# HELPERS
# ============================================================

def save_checkpoint():
    """Persist P8 progress safely."""

    checkpoint["successful_product_ids"] = sorted(
        processed_ids
    )

    checkpoint["updated_at"] = time.time()

    tmp_file = CHECKPOINT_FILE + ".tmp"

    with open(
        tmp_file,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f
        )

    os.replace(
        tmp_file,
        CHECKPOINT_FILE
    )


def append_jsonl(path, record):

    with open(
        path,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            )
            + "\n"
        )


def safe_float(value):

    try:
        return float(value)
    except Exception:
        return None


# ============================================================
# START
# ============================================================

total_products = len(df)

remaining = total_products - len(
    processed_ids
)

print("\n" + "=" * 70)
print("P8 RUN")
print("=" * 70)

print(
    "Total products:",
    total_products
)

print(
    "Already completed:",
    len(processed_ids)
)

print(
    "Remaining:",
    remaining
)

print("\nStarting...")


# ============================================================
# STATISTICS
# ============================================================

successful = len(processed_ids)
failed = 0
fallbacks = 0

start_time = time.perf_counter()

last_checkpoint = len(
    processed_ids
)


# ============================================================
# PRODUCT LOOP
# ============================================================

for row_idx, row in df.iterrows():

    product_id = str(
        row["sku"]
    )

    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    if product_id in processed_ids:
        continue

    product_start = time.perf_counter()

    try:

        # ====================================================
        # BUILD PRODUCT INPUTS
        # ====================================================

        # ----------------------------------------------------
        # IMAGE INPUT
        # ----------------------------------------------------

        image_urls = parse_images(
            row["images"]
        )

        image_inputs = []

        for idx, url in enumerate(
            image_urls
        ):

            image_inputs.append(
                {
                    "imageId": (
                        f"img-{product_id}-{idx}"
                    ),
                    "imageUrl": url,
                    "viewType": "front"
                }
            )

        # ----------------------------------------------------
        # NOTE:
        # Use the existing project input builders here.
        # ----------------------------------------------------

        text_input = build_text_encoder_input(
            row
        )

        attribute_input = (
            build_attribute_encoder_input(
                row
            )
        )

        image_input = (
            build_image_encoder_input(
                product_id,
                image_inputs
            )
        )

        # ====================================================
        # P3 TEXT
        # ====================================================

        text_rep = text_encoder.encode(
            text_input
        )

        # ====================================================
        # P4 ATTRIBUTE
        # ====================================================

        attribute_rep = (
            attribute_encoder.encode(
                attribute_input
            )
        )

        # ====================================================
        # P2 IMAGE / OPTIMIZED CLIP
        # ====================================================

        visual_rep = (
            image_encoder.encode(
                image_input
            )
        )

        # ====================================================
        # FALLBACK TRACKING
        # ====================================================

        metadata = (
            visual_rep.processingMetadata
            or {}
        )

        if metadata.get(
            "inferenceMode"
        ) not in (
            "CLIP",
            "CLIP_BATCH"
        ):

            fallbacks += 1

        # ====================================================
        # P5
        # ====================================================

        profile = (
            insight_service.aggregate(
                visual=visual_rep,
                text=text_rep,
                attribute=attribute_rep
            )
        )

        # ====================================================
        # P6
        # ====================================================

        fused = fusion_service.fuse(
            visual=visual_rep,
            text=text_rep,
            attribute=attribute_rep,
            unified_product_profile=profile
        )

        # ====================================================
        # VALIDATE 662D
        # ====================================================

        embedding = np.asarray(
            fused.unifiedEmbedding,
            dtype=np.float32
        )

        if embedding.shape != (662,):

            raise ValueError(
                f"Invalid embedding shape: "
                f"{embedding.shape}"
            )

        if not np.isfinite(
            embedding
        ).all():

            raise ValueError(
                "Embedding contains NaN/Inf"
            )

        norm = float(
            np.linalg.norm(
                embedding
            )
        )

        if not np.isfinite(norm):

            raise ValueError(
                "Invalid L2 norm"
            )

        if norm == 0:

            raise ValueError(
                "Zero embedding"
            )

        # ====================================================
        # SAVE
        # ====================================================

        result = {
            "productId": product_id,
            "unifiedEmbedding": (
                embedding.tolist()
            ),
            "embeddingDimension": 662,
            "l2Norm": norm,
            "confidence": safe_float(
                fused.confidence
            ),
            "modalities": {
                k: v.model_dump()
                if hasattr(v, "model_dump")
                else v
                for k, v in (
                    fused.modalities or {}
                ).items()
            },
            "provenance": fused.provenance,
            "encoderVersions": (
                profile.encoderVersions
            ),
            "rowIndex": int(row_idx)
        }

        append_jsonl(
            RESULTS_FILE,
            result
        )

        processed_ids.add(
            product_id
        )

        successful += 1

    except Exception as exc:

        failed += 1

        error_record = {
            "rowIndex": int(row_idx),
            "productId": product_id,
            "error": str(exc)
        }

        append_jsonl(
            ERRORS_FILE,
            error_record
        )

    # ========================================================
    # PROGRESS
    # ========================================================

    completed = (
        successful
        + failed
    )

    elapsed = (
        time.perf_counter()
        - start_time
    )

    if completed > 0:

        rate = (
            completed
            / elapsed
        )

        remaining_count = (
            total_products
            - len(processed_ids)
        )

        eta_seconds = (
            remaining_count / rate
            if rate > 0
            else 0
        )

    else:

        rate = 0
        eta_seconds = 0

    if (
        completed % 10 == 0
        or completed == total_products
    ):

        print(
            f"Processed "
            f"{completed}/{total_products} "
            f"| Success: {successful} "
            f"| Failed: {failed} "
            f"| Fallbacks: {fallbacks} "
            f"| Rate: {rate:.2f} prod/s "
            f"| ETA: "
            f"{eta_seconds / 60:.1f} min"
        )

    # ========================================================
    # CHECKPOINT
    # ========================================================

    if (
        len(processed_ids)
        - last_checkpoint
        >= CHECKPOINT_EVERY
    ):

        save_checkpoint()

        last_checkpoint = len(
            processed_ids
        )

        print(
            "  💾 Checkpoint saved:"
            f" {len(processed_ids)} products"
        )


# ============================================================
# FINAL CHECKPOINT
# ============================================================

save_checkpoint()


# ============================================================
# FINAL VALIDATION
# ============================================================

total_time = (
    time.perf_counter()
    - start_time
)

accounted = (
    len(processed_ids)
    + failed
)

print("\n" + "=" * 70)
print("P8 FINAL RESULTS")
print("=" * 70)

print(
    "\nTotal catalog:",
    total_products
)

print(
    "Successful:",
    len(processed_ids)
)

print(
    "Failed:",
    failed
)

print(
    "Accounted:",
    accounted
)

print(
    "Success rate:",
    f"{len(processed_ids) / total_products * 100:.2f}%"
)

print(
    "CLIP fallbacks:",
    fallbacks
)

print(
    "Total runtime:",
    f"{total_time / 60:.2f} minutes"
)

if total_time > 0:

    print(
        "Throughput:",
        f"{(successful + failed) / total_time:.2f}",
        "products/sec"
    )


# ============================================================
# VERIFY OUTPUT
# ============================================================

print("\n" + "-" * 70)
print("OUTPUT VALIDATION")
print("-" * 70)

embedding_count = 0
invalid_count = 0
duplicate_ids = set()
seen_ids = set()

if os.path.exists(
    RESULTS_FILE
):

    with open(
        RESULTS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            record = json.loads(line)

            pid = str(
                record["productId"]
            )

            if pid in seen_ids:

                duplicate_ids.add(pid)

            seen_ids.add(pid)

            embedding = np.asarray(
                record[
                    "unifiedEmbedding"
                ],
                dtype=np.float32
            )

            if (
                embedding.shape != (662,)
                or not np.isfinite(
                    embedding
                ).all()
                or np.linalg.norm(
                    embedding
                ) == 0
            ):

                invalid_count += 1

            else:

                embedding_count += 1


print(
    "Embeddings:",
    embedding_count
)

print(
    "Invalid embeddings:",
    invalid_count
)

print(
    "Duplicate product IDs:",
    len(duplicate_ids)
)

print(
    "Output file:",
    RESULTS_FILE
)

print(
    "Checkpoint:",
    CHECKPOINT_FILE
)

print("\n" + "=" * 70)

if (
    accounted == total_products
    and invalid_count == 0
    and len(duplicate_ids) == 0
):

    print(
        "✅ P8 FULL CATALOG GENERATION COMPLETE"
    )

else:

    print(
        "⚠️ P8 COMPLETED WITH VALIDATION ISSUES"
    )

print("=" * 70)

ZYRA V1 — P8 FULL CATALOG GENERATION

Output directory:
/Users/saketh/Desktop/Projects/weavly/core-model/p8_output

Checkpoint every:
25 products

Device: mps
✅ M5 MPS confirmed

Initializing Zyra pipeline...
✅ Image encoder initialized
✅ P5 insight service initialized
✅ P6 fusion service initialized

No existing checkpoint.
Starting from product 1.
Products already completed: 0

P8 RUN
Total products: 12491
Already completed: 0
Remaining: 12491

Starting...
Processed 10/12491 | Success: 0 | Failed: 10 | Fallbacks: 0 | Rate: 424.69 prod/s | ETA: 0.5 min
Processed 20/12491 | Success: 0 | Failed: 20 | Fallbacks: 0 | Rate: 806.02 prod/s | ETA: 0.3 min
Processed 30/12491 | Success: 0 | Failed: 30 | Fallbacks: 0 | Rate: 1152.94 prod/s | ETA: 0.2 min
Processed 40/12491 | Success: 0 | Failed: 40 | Fallbacks: 0 | Rate: 1478.70 prod/s | ETA: 0.1 min
Processed 50/12491 | Success: 0 | Failed: 50 | Fallbacks: 0 | Rate: 1786.64 prod/s | ETA: 0.1 min
Processed 60/12491 | Success: 0 | Failed: 60 | Fa

In [6]:
# ======================================================================
# ZYRA V1 — P8 FULL CATALOG GENERATION
# Optimized CLIP + M5 MPS + Checkpoint/Resume
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("ZYRA V1 — P8 FULL CATALOG GENERATION")
print("=" * 70)

# ======================================================================
# CONFIG
# ======================================================================

OUTPUT_DIR = (
    "/Users/saketh/Desktop/Projects/weavly/"
    "core-model/p8_output"
)

CHECKPOINT_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_checkpoint.json"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_product_embeddings.jsonl"
)

ERRORS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_errors.jsonl"
)

CHECKPOINT_EVERY = 25

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ======================================================================
# DEVICE
# ======================================================================

if not torch.backends.mps.is_available():
    raise RuntimeError(
        "Apple MPS is not available."
    )

device = torch.device("mps")

print("\nDevice:", device)
print(
    "ML encoding:",
    settings.ENABLE_ML_ENCODING
)
print(
    "Batch inference:",
    image_encoder.use_batch_inference
)

# ======================================================================
# VERIFY CLIP
# ======================================================================

model, processor = (
    image_encoder
    .backbone
    .model_manager
    .get_vision_model()
)

if model is None or processor is None:
    raise RuntimeError(
        "CLIP model or processor failed to load."
    )

actual_device = next(
    model.parameters()
).device

print(
    "CLIP model:",
    type(model)
)

print(
    "Actual model device:",
    actual_device
)

print(
    "Vision model:",
    image_encoder
    .backbone
    .model_manager
    .vision_model_name
)

print(
    "✅ REAL CLIP AVAILABLE"
)

# ======================================================================
# HELPER FUNCTIONS
# ======================================================================

def append_jsonl(path, record):

    with open(
        path,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=str
            )
            + "\n"
        )


def save_checkpoint(
    successful_ids,
    failed_ids
):

    checkpoint = {
        "successful_product_ids":
            sorted(successful_ids),

        "failed_product_ids":
            sorted(failed_ids),

        "updated_at":
            time.time()
    }

    tmp_path = (
        CHECKPOINT_FILE
        + ".tmp"
    )

    with open(
        tmp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=2
        )

    os.replace(
        tmp_path,
        CHECKPOINT_FILE
    )


def get_fused_embedding(fused):

    # Actual P6 output may expose the embedding
    # under one of these supported names.

    if hasattr(
        fused,
        "unifiedEmbedding"
    ):
        return fused.unifiedEmbedding

    if hasattr(
        fused,
        "embedding"
    ):
        return fused.embedding

    if hasattr(
        fused,
        "fusedEmbedding"
    ):
        return fused.fusedEmbedding

    if isinstance(
        fused,
        dict
    ):

        for key in (
            "unifiedEmbedding",
            "embedding",
            "fusedEmbedding"
        ):

            if key in fused:
                return fused[key]

    raise AttributeError(
        "Could not locate unified 662D "
        "embedding in ProductFusionService output."
    )


def get_object_value(
    obj,
    name,
    default=None
):

    if hasattr(
        obj,
        name
    ):

        return getattr(
            obj,
            name
        )

    if isinstance(
        obj,
        dict
    ):

        return obj.get(
            name,
            default
        )

    return default


# ======================================================================
# PRODUCT PROCESSING FUNCTION
# ======================================================================

def process_product(row):

    product_id = str(
        row["sku"]
    )

    # ==============================================================
    # BUILD INPUTS
    # ==============================================================

    text_input = (
        build_text_encoder_input(
            row
        )
    )

    attribute_input = (
        build_attribute_encoder_input(
            row
        )
    )

    image_input = (
        build_image_encoder_input(
            row
        )
    )

    # ==============================================================
    # P3 — TEXT
    # ==============================================================

    text_rep = (
        text_encoder.encode(
            text_input
        )
    )

    # ==============================================================
    # P4 — ATTRIBUTE
    # ==============================================================

    attribute_rep = (
        attribute_encoder.encode(
            attribute_input
        )
    )

    # ==============================================================
    # P2 — OPTIMIZED BATCH CLIP
    # ==============================================================

    visual_rep = (
        image_encoder.encode(
            image_input
        )
    )

    # ==============================================================
    # VERIFY REAL CLIP
    #
    # IMPORTANT:
    # inferenceMode is stored inside each
    # PerImageVisualRepresentation.processingMetadata,
    # NOT in ProductVisualRepresentation.processingMetadata.
    # ==============================================================

    per_image_reps = (
        visual_rep.perImageRepresentations
    )

    if not per_image_reps:

        raise RuntimeError(
            "No successful image representations generated."
        )

    inference_modes = set()

    for rep in per_image_reps:

        metadata = (
            rep.processingMetadata
            or {}
        )

        mode = metadata.get(
            "inferenceMode"
        )

        inference_modes.add(
            mode
        )

    allowed_modes = {
        "CLIP-Batch",
        "CLIP"
    }

    if not inference_modes.issubset(
        allowed_modes
    ):

        raise RuntimeError(
            "Non-CLIP inference detected: "
            f"{inference_modes}"
        )

    inference_mode = "CLIP-Batch"

    # ==============================================================
    # P5 — INSIGHT AGGREGATION
    # ==============================================================

    profile = (
        insight_service.aggregate(
            visual=visual_rep,
            text=text_rep,
            attribute=attribute_rep
        )
    )

    # ==============================================================
    # P6 — MULTIMODAL FUSION
    # ==============================================================

    fused = (
        fusion_service.fuse(
            visual=visual_rep,
            text=text_rep,
            attribute=attribute_rep,
            unified_product_profile=profile
        )
    )

    # ==============================================================
    # EXTRACT 662D EMBEDDING
    # ==============================================================

    embedding = np.asarray(
        get_fused_embedding(
            fused
        ),
        dtype=np.float32
    )

    # ==============================================================
    # NUMERICAL VALIDATION
    # ==============================================================

    if embedding.shape != (
        662,
    ):

        raise ValueError(
            "Invalid embedding shape: "
            f"{embedding.shape}"
        )

    if not np.isfinite(
        embedding
    ).all():

        raise ValueError(
            "Embedding contains NaN or Inf."
        )

    norm = float(
        np.linalg.norm(
            embedding
        )
    )

    if not np.isfinite(
        norm
    ):

        raise ValueError(
            "Embedding norm is not finite."
        )

    if norm == 0:

        raise ValueError(
            "Embedding is a zero vector."
        )

    # ==============================================================
    # CONFIDENCE
    # ==============================================================

    confidence = get_object_value(
        fused,
        "confidence",
        None
    )

    if confidence is not None:

        try:
            confidence = float(
                confidence
            )
        except Exception:
            confidence = None

    # ==============================================================
    # PROVENANCE
    # ==============================================================

    provenance = get_object_value(
        fused,
        "provenance",
        None
    )

    # ==============================================================
    # OUTPUT RECORD
    # ==============================================================

    result = {

        "productId":
            product_id,

        "unifiedEmbedding":
            embedding.tolist(),

        "embeddingDimension":
            662,

        "l2Norm":
            norm,

        "confidence":
            confidence,

        "provenance":
            provenance,

        "inferenceMode":
            inference_mode,

        "encoderVersions":
            profile.encoderVersions,

        "rowIndex":
            int(row.name),

        "generatedAt":
            time.time()
    }

    return result


# ======================================================================
# 10-PRODUCT SMOKE TEST
# ======================================================================

print("\n" + "=" * 70)
print("P8 — 10 PRODUCT SMOKE TEST")
print("=" * 70)

smoke_start = time.perf_counter()

smoke_success = 0

for i in range(
    min(10, len(df))
):

    row = df.iloc[i]

    product_id = str(
        row["sku"]
    )

    try:

        result = process_product(
            row
        )

        smoke_success += 1

        print(
            f"✅ {i + 1}/10 "
            f"| Product {product_id} "
            f"| dim={result['embeddingDimension']} "
            f"| norm={result['l2Norm']:.6f} "
            f"| mode={result['inferenceMode']}"
        )

    except Exception as exc:

        print("\n❌ SMOKE TEST FAILED")

        print(
            "Product:",
            product_id
        )

        print(
            "Error:",
            repr(exc)
        )

        raise

smoke_time = (
    time.perf_counter()
    - smoke_start
)

print(
    "\nSmoke test time:",
    f"{smoke_time:.2f}s"
)

print(
    "Smoke throughput:",
    f"{smoke_success / smoke_time:.2f}",
    "products/sec"
)

print(
    "✅ 10/10 PRODUCTS PASSED"
)

# ======================================================================
# ASK FOR EXPLICIT CONFIRMATION BEFORE FULL RUN
# ======================================================================

print("\n" + "=" * 70)
print("SMOKE TEST COMPLETE")
print("=" * 70)

print(
    "\nThe 10-product validation passed."
)

print(
    "The full 12,491-product run is NOT started yet."
)

print(
    "\nTo start the production run, execute the next cell."
)

print("=" * 70)

ZYRA V1 — P8 FULL CATALOG GENERATION

Device: mps
ML encoding: True
Batch inference: True
CLIP model: <class 'transformers.models.clip.modeling_clip.CLIPModel'>
Actual model device: mps:0
Vision model: openai/clip-vit-base-patch32
✅ REAL CLIP AVAILABLE

P8 — 10 PRODUCT SMOKE TEST

❌ SMOKE TEST FAILED
Product: 10017413
Error: TypeError("ProductFusionService.fuse() got an unexpected keyword argument 'unified_product_profile'")


TypeError: ProductFusionService.fuse() got an unexpected keyword argument 'unified_product_profile'

In [2]:
# ======================================================================
# ZYRA V1 — P8 FULL CATALOG GENERATION
# Optimized CLIP + M5 MPS + Checkpoint/Resume
# ======================================================================

import os
import json
import time
import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("ZYRA V1 — P8 FULL CATALOG GENERATION")
print("=" * 70)

# ======================================================================
# CONFIG
# ======================================================================

OUTPUT_DIR = (
    "/Users/saketh/Desktop/Projects/weavly/"
    "core-model/p8_output"
)

CHECKPOINT_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_checkpoint.json"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_product_embeddings.jsonl"
)

ERRORS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_errors.jsonl"
)

CHECKPOINT_EVERY = 25

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# ======================================================================
# DEVICE
# ======================================================================

if not torch.backends.mps.is_available():
    raise RuntimeError(
        "Apple MPS is not available."
    )

device = torch.device("mps")

print("\nDevice:", device)
print(
    "ML encoding:",
    settings.ENABLE_ML_ENCODING
)
print(
    "Batch inference:",
    image_encoder.use_batch_inference
)

# ======================================================================
# VERIFY CLIP
# ======================================================================

model, processor = (
    image_encoder
    .backbone
    .model_manager
    .get_vision_model()
)

if model is None or processor is None:
    raise RuntimeError(
        "CLIP model or processor failed to load."
    )

actual_device = next(
    model.parameters()
).device

print(
    "CLIP model:",
    type(model)
)

print(
    "Actual model device:",
    actual_device
)

print(
    "Vision model:",
    image_encoder
    .backbone
    .model_manager
    .vision_model_name
)

print(
    "✅ REAL CLIP AVAILABLE"
)

# ======================================================================
# HELPER FUNCTIONS
# ======================================================================

def append_jsonl(path, record):

    with open(
        path,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=str
            )
            + "\n"
        )


def save_checkpoint(
    successful_ids,
    failed_ids
):

    checkpoint = {
        "successful_product_ids":
            sorted(successful_ids),

        "failed_product_ids":
            sorted(failed_ids),

        "updated_at":
            time.time()
    }

    tmp_path = (
        CHECKPOINT_FILE
        + ".tmp"
    )

    with open(
        tmp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=2
        )

    os.replace(
        tmp_path,
        CHECKPOINT_FILE
    )


def get_fused_embedding(fused):

    # Actual P6 output may expose the embedding
    # under one of these supported names.

    if hasattr(
        fused,
        "unifiedEmbedding"
    ):
        return fused.unifiedEmbedding

    if hasattr(
        fused,
        "embedding"
    ):
        return fused.embedding

    if hasattr(
        fused,
        "fusedEmbedding"
    ):
        return fused.fusedEmbedding

    if isinstance(
        fused,
        dict
    ):

        for key in (
            "unifiedEmbedding",
            "embedding",
            "fusedEmbedding"
        ):

            if key in fused:
                return fused[key]

    raise AttributeError(
        "Could not locate unified 662D "
        "embedding in ProductFusionService output."
    )


def get_object_value(
    obj,
    name,
    default=None
):

    if hasattr(
        obj,
        name
    ):

        return getattr(
            obj,
            name
        )

    if isinstance(
        obj,
        dict
    ):

        return obj.get(
            name,
            default
        )

    return default


# ======================================================================
# PRODUCT PROCESSING FUNCTION
# ======================================================================

def process_product(row):

    product_id = str(
        row["sku"]
    )

    # ==============================================================
    # BUILD INPUTS
    # ==============================================================

    text_input = (
        build_text_encoder_input(
            row
        )
    )

    attribute_input = (
        build_attribute_encoder_input(
            row
        )
    )

    image_input = (
        build_image_encoder_input(
            row
        )
    )

    # ==============================================================
    # P3 — TEXT
    # ==============================================================

    text_rep = (
        text_encoder.encode(
            text_input
        )
    )

    # ==============================================================
    # P4 — ATTRIBUTE
    # ==============================================================

    attribute_rep = (
        attribute_encoder.encode(
            attribute_input
        )
    )

    # ==============================================================
    # P2 — OPTIMIZED BATCH CLIP
    # ==============================================================

    visual_rep = (
        image_encoder.encode(
            image_input
        )
    )

    # ==============================================================
    # VERIFY REAL CLIP
    #
    # IMPORTANT:
    # inferenceMode is stored inside each
    # PerImageVisualRepresentation.processingMetadata,
    # NOT in ProductVisualRepresentation.processingMetadata.
    # ==============================================================

    per_image_reps = (
        visual_rep.perImageRepresentations
    )

    if not per_image_reps:

        raise RuntimeError(
            "No successful image representations generated."
        )

    inference_modes = set()

    for rep in per_image_reps:

        metadata = (
            rep.processingMetadata
            or {}
        )

        mode = metadata.get(
            "inferenceMode"
        )

        inference_modes.add(
            mode
        )

    allowed_modes = {
        "CLIP-Batch",
        "CLIP"
    }

    if not inference_modes.issubset(
        allowed_modes
    ):

        raise RuntimeError(
            "Non-CLIP inference detected: "
            f"{inference_modes}"
        )

    inference_mode = "CLIP-Batch"

    # ==============================================================
    # P5 — INSIGHT AGGREGATION
    # ==============================================================

    profile = (
        insight_service.aggregate(
            visual=visual_rep,
            text=text_rep,
            attribute=attribute_rep
        )
    )

    # ==============================================================
    # P6 — MULTIMODAL FUSION
    # ==============================================================

    fused = fusion_service.fuse(
        profile=profile,
        visual=visual_rep,
        text=text_rep,
        attribute=attribute_rep
    )
            
    # ==============================================================
    # EXTRACT 662D EMBEDDING
    # ==============================================================

    embedding = np.asarray(
        get_fused_embedding(
            fused
        ),
        dtype=np.float32
    )

    # ==============================================================
    # NUMERICAL VALIDATION
    # ==============================================================

    if embedding.shape != (
        662,
    ):

        raise ValueError(
            "Invalid embedding shape: "
            f"{embedding.shape}"
        )

    if not np.isfinite(
        embedding
    ).all():

        raise ValueError(
            "Embedding contains NaN or Inf."
        )

    norm = float(
        np.linalg.norm(
            embedding
        )
    )

    if not np.isfinite(
        norm
    ):

        raise ValueError(
            "Embedding norm is not finite."
        )

    if norm == 0:

        raise ValueError(
            "Embedding is a zero vector."
        )

    # ==============================================================
    # CONFIDENCE
    # ==============================================================

    confidence = get_object_value(
        fused,
        "confidence",
        None
    )

    if confidence is not None:

        try:
            confidence = float(
                confidence
            )
        except Exception:
            confidence = None

    # ==============================================================
    # PROVENANCE
    # ==============================================================

    provenance = get_object_value(
        fused,
        "provenance",
        None
    )

    # ==============================================================
    # OUTPUT RECORD
    # ==============================================================

    result = {

        "productId":
            product_id,

        "unifiedEmbedding":
            embedding.tolist(),

        "embeddingDimension":
            662,

        "l2Norm":
            norm,

        "confidence":
            confidence,

        "provenance":
            provenance,

        "inferenceMode":
            inference_mode,

        "encoderVersions":
            profile.encoderVersions,

        "rowIndex":
            int(row.name),

        "generatedAt":
            time.time()
    }

    return result


# ======================================================================
# 10-PRODUCT SMOKE TEST
# ======================================================================

print("\n" + "=" * 70)
print("P8 — 10 PRODUCT SMOKE TEST")
print("=" * 70)

smoke_start = time.perf_counter()

smoke_success = 0

for i in range(
    min(10, len(df))
):

    row = df.iloc[i]

    product_id = str(
        row["sku"]
    )

    try:

        result = process_product(
            row
        )

        smoke_success += 1

        print(
            f"✅ {i + 1}/10 "
            f"| Product {product_id} "
            f"| dim={result['embeddingDimension']} "
            f"| norm={result['l2Norm']:.6f} "
            f"| mode={result['inferenceMode']}"
        )

    except Exception as exc:

        print("\n❌ SMOKE TEST FAILED")

        print(
            "Product:",
            product_id
        )

        print(
            "Error:",
            repr(exc)
        )

        raise

smoke_time = (
    time.perf_counter()
    - smoke_start
)

print(
    "\nSmoke test time:",
    f"{smoke_time:.2f}s"
)

print(
    "Smoke throughput:",
    f"{smoke_success / smoke_time:.2f}",
    "products/sec"
)

print(
    "✅ 10/10 PRODUCTS PASSED"
)

# ======================================================================
# ASK FOR EXPLICIT CONFIRMATION BEFORE FULL RUN
# ======================================================================

print("\n" + "=" * 70)
print("SMOKE TEST COMPLETE")
print("=" * 70)

print(
    "\nThe 10-product validation passed."
)

print(
    "The full 12,491-product run is NOT started yet."
)

print(
    "\nTo start the production run, execute the next cell."
)

print("=" * 70)

ZYRA V1 — P8 FULL CATALOG GENERATION

Device: mps


NameError: name 'settings' is not defined

In [3]:
# ======================================================================
# ZYRA V1 — P8 SINGLE JUPYTER BOOTSTRAP CELL
# Restores all environment state, dataset, helpers, encoders & M5 GPU
# ======================================================================

import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

# 1. Environment & Path Resolution
CORE_MODEL_ROOT = Path("/Users/saketh/Desktop/Projects/weavly/core-model")
if str(CORE_MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(CORE_MODEL_ROOT))

# 2. Configuration & Live ML Activation
from zyra.product_encoder.config.settings import get_product_settings

settings = get_product_settings()
settings.ENABLE_ML_ENCODING = True

# 3. Canonical Schema & Router Imports
from zyra.product_encoder.ingestion.router import (
    ProductTextEncoderInput,
    ProductAttributeEncoderInput,
    ProductAttributes,
    ProductImageEncoderInput,
    ProductImageInput,
)

# 4. Multimodal Pipeline Components
from zyra.product_encoder.text_encoder.encoder import ProductTextEncoder
from zyra.product_encoder.attribute_encoder.encoder import ProductAttributeEncoder
from zyra.product_encoder.image_encoder.encoder import ProductImageEncoder
from zyra.product_encoder.insights.service import ProductInsightAggregationService
from zyra.product_encoder.fusion import ProductFusionService, FusionWeightsConfig

# 5. Dataset & Catalog Ingestion
DATASET_PATH = Path("/Users/saketh/Desktop/Projects/weavly/data/raw/myntra-fashion-products/Myntra_fashion_products.csv")
df = pd.read_csv(DATASET_PATH)

CATALOG_PATH = CORE_MODEL_ROOT / "data/recommendation/product_catalog.pkl"
if CATALOG_PATH.exists():
    products = pd.read_pickle(CATALOG_PATH)
    attribute_columns = [col for col in products.columns if col.startswith("attr_")]
    if "sku" in products.columns and "sku" in df.columns:
        cols_to_merge = ["sku"] + [c for c in attribute_columns if c not in df.columns]
        df = df.merge(products[cols_to_merge], on="sku", how="left")
else:
    products = None
    attribute_columns = [col for col in df.columns if col.startswith("attr_")]

# 6. Pipeline Encoders & Services Initialization (with Optimized Batch CLIP)
text_encoder = ProductTextEncoder()
attribute_encoder = ProductAttributeEncoder()
image_encoder = ProductImageEncoder(use_batch_inference=True)
insight_service = ProductInsightAggregationService()
fusion_service = ProductFusionService(
    weights_config=FusionWeightsConfig(
        visualWeight=0.45,
        textWeight=0.35,
        attributeWeight=0.20,
    )
)

# 7. Hardware & Model Acceleration Check
model, processor = image_encoder.backbone.model_manager.get_vision_model()
actual_device = next(model.parameters()).device

# 8. Data Ingestion & Builder Helper Functions
def parse_images(value):
    if isinstance(value, list):
        return value
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return [x.strip() for x in str(value).split("~") if x.strip()]

def build_text_encoder_input(row):
    return ProductTextEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        description=str(row.get("description", "")),
        brand=str(row.get("brand", "")),
        category="fashion",
        styles=[],
        occasions=[],
        seasons=[],
        tags=[],
    )
make_text_input = build_text_encoder_input

def build_attribute_encoder_input(row, attr_cols=None):
    cols = attr_cols if attr_cols is not None else attribute_columns
    attrs = {}
    for col in cols:
        if col.startswith("attr_"):
            val = row.get(col, 0)
            if val == 1 or val is True or val == "1":
                attrs[col.replace("attr_", "")] = True
        else:
            val = row.get(col)
            if val is not None and not (isinstance(val, float) and np.isnan(val)):
                attrs[col] = val
    return ProductAttributeEncoderInput(
        productId=str(row["sku"]),
        category="fashion",
        attributes=ProductAttributes(customAttributes=attrs),
        rawAttributes=attrs,
    )
make_attribute_input = build_attribute_encoder_input

def build_image_encoder_input(row):
    urls = parse_images(row.get("images"))
    images = [
        ProductImageInput(
            imageUrl=url,
            viewType="front" if i == 0 else "detail" if i > 2 else "on_model",
            sortOrder=i,
        )
        for i, url in enumerate(urls)
    ]
    return ProductImageEncoderInput(
        productId=str(row["sku"]),
        title=str(row["name"]),
        images=images,
    )
make_image_input = build_image_encoder_input

# 9. Verification Summary
print("=" * 70)
print("ZYRA V1 — P8 BOOTSTRAP READY")
print("=" * 70)
print()
print(f"Dataset: {len(df)} products")
print(f"Device: {actual_device.type}")
print(f"ML encoding: {settings.ENABLE_ML_ENCODING}")
print("Text encoder: READY")
print("Attribute encoder: READY")
print("Image encoder: READY")
print("P5 service: READY")
print("P6 service: READY")
print("Optimized batch CLIP: READY")
print()
print("=" * 70)
print("READY TO RUN P8")
print("=" * 70)


/Users/saketh/Desktop/Projects/weavly/core-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████████████| 398/398 [00:00<00:00, 71907.52it/s]


ZYRA V1 — P8 BOOTSTRAP READY

Dataset: 12491 products
Device: mps
ML encoding: True
Text encoder: READY
Attribute encoder: READY
Image encoder: READY
P5 service: READY
P6 service: READY
Optimized batch CLIP: READY

READY TO RUN P8


In [ ]:
# ======================================================================
# ZYRA V1 — P8 FULL 12,491 PRODUCT PRODUCTION RUN
# Apple M5 / MPS + Optimized CLIP-Batch + Checkpoint/Resume
# ======================================================================

import os
import json
import time
import numpy as np
import torch

print("=" * 70)
print("ZYRA V1 — P8 FULL CATALOG PRODUCTION RUN")
print("=" * 70)

# ======================================================================
# CONFIG
# ======================================================================

OUTPUT_DIR = (
    "/Users/saketh/Desktop/Projects/weavly/"
    "core-model/p8_output"
)

CHECKPOINT_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_checkpoint.json"
)

RESULTS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_product_embeddings.jsonl"
)

ERRORS_FILE = os.path.join(
    OUTPUT_DIR,
    "p8_errors.jsonl"
)

CHECKPOINT_EVERY = 25

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================================================================
# DEVICE / MODEL VALIDATION
# ======================================================================

if not torch.backends.mps.is_available():
    raise RuntimeError("Apple MPS is not available.")

device = torch.device("mps")

print("\nDevice:", device)
print("ML encoding:", settings.ENABLE_ML_ENCODING)
print(
    "Batch inference:",
    image_encoder.use_batch_inference
)

model, processor = (
    image_encoder
    .backbone
    .model_manager
    .get_vision_model()
)

if model is None or processor is None:
    raise RuntimeError(
        "REAL CLIP model/processor is not loaded."
    )

actual_device = next(
    model.parameters()
).device

print(
    "CLIP model:",
    type(model)
)

print(
    "Actual model device:",
    actual_device
)

print(
    "Model:",
    image_encoder
    .backbone
    .model_manager
    .vision_model_name
)

print("✅ REAL CLIP + M5 MPS READY")

# ======================================================================
# LOAD PREVIOUS PROGRESS
# ======================================================================

successful_ids = set()
failed_ids = set()

if os.path.exists(CHECKPOINT_FILE):

    print("\nExisting checkpoint found.")

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        checkpoint = json.load(f)

    successful_ids = set(
        str(x)
        for x in checkpoint.get(
            "successful_product_ids",
            []
        )
    )

    failed_ids = set(
        str(x)
        for x in checkpoint.get(
            "failed_product_ids",
            []
        )
    )

    print(
        "Checkpoint successful:",
        len(successful_ids)
    )

    print(
        "Checkpoint failed:",
        len(failed_ids)
    )

else:

    print(
        "\nNo checkpoint found."
    )

# ======================================================================
# RECOVER IDS FROM OUTPUT FILE
# ======================================================================

if os.path.exists(RESULTS_FILE):

    print(
        "\nScanning existing embedding output..."
    )

    with open(
        RESULTS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(line)

                pid = str(
                    record["productId"]
                )

                successful_ids.add(pid)

            except Exception:
                pass

if os.path.exists(ERRORS_FILE):

    print(
        "Scanning existing error output..."
    )

    with open(
        ERRORS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(line)

                pid = str(
                    record["productId"]
                )

                failed_ids.add(pid)

            except Exception:
                pass

# ======================================================================
# SAFETY
# ======================================================================

# A product successfully embedded must never be treated as failed.

failed_ids -= successful_ids

total_products = len(df)

completed_ids = (
    successful_ids | failed_ids
)

remaining = (
    total_products
    - len(completed_ids)
)

print("\n" + "=" * 70)
print("P8 PRODUCTION STATUS")
print("=" * 70)

print(
    "Total products:",
    total_products
)

print(
    "Already successful:",
    len(successful_ids)
)

print(
    "Already failed:",
    len(failed_ids)
)

print(
    "Remaining:",
    remaining
)

# ======================================================================
# HELPERS
# ======================================================================

def append_jsonl(path, record):

    with open(
        path,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
                default=str
            )
            + "\n"
        )


def save_checkpoint():

    checkpoint = {
        "successful_product_ids":
            sorted(successful_ids),

        "failed_product_ids":
            sorted(failed_ids),

        "total_products":
            total_products,

        "updated_at":
            time.time()
    }

    tmp_path = (
        CHECKPOINT_FILE
        + ".tmp"
    )

    with open(
        tmp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            indent=2
        )

    os.replace(
        tmp_path,
        CHECKPOINT_FILE
    )


def get_fused_embedding(fused):

    if hasattr(
        fused,
        "unifiedEmbedding"
    ):
        return fused.unifiedEmbedding

    if hasattr(
        fused,
        "embedding"
    ):
        return fused.embedding

    if hasattr(
        fused,
        "fusedEmbedding"
    ):
        return fused.fusedEmbedding

    if isinstance(
        fused,
        dict
    ):

        for key in (
            "unifiedEmbedding",
            "embedding",
            "fusedEmbedding"
        ):

            if key in fused:
                return fused[key]

    raise AttributeError(
        "Could not find 662D embedding "
        "in P6 output."
    )


def get_value(
    obj,
    name,
    default=None
):

    if hasattr(
        obj,
        name
    ):

        return getattr(
            obj,
            name
        )

    if isinstance(
        obj,
        dict
    ):

        return obj.get(
            name,
            default
        )

    return default


# ======================================================================
# PRODUCT PROCESSING
# ======================================================================

def process_product(row):

    product_id = str(
        row["sku"]
    )

    # --------------------------------------------------
    # P3 — TEXT INPUT
    # --------------------------------------------------

    text_input = (
        build_text_encoder_input(
            row
        )
    )

    # --------------------------------------------------
    # P4 — ATTRIBUTE INPUT
    # --------------------------------------------------

    attribute_input = (
        build_attribute_encoder_input(
            row
        )
    )

    # --------------------------------------------------
    # P2 — IMAGE INPUT
    # --------------------------------------------------

    image_input = (
        build_image_encoder_input(
            row
        )
    )

    # --------------------------------------------------
    # P3 — TEXT ENCODER
    # --------------------------------------------------

    text_rep = (
        text_encoder.encode(
            text_input
        )
    )

    # --------------------------------------------------
    # P4 — ATTRIBUTE ENCODER
    # --------------------------------------------------

    attribute_rep = (
        attribute_encoder.encode(
            attribute_input
        )
    )

    # --------------------------------------------------
    # P2 — OPTIMIZED CLIP-BATCH
    # --------------------------------------------------

    visual_rep = (
        image_encoder.encode(
            image_input
        )
    )

    # --------------------------------------------------
    # VERIFY CLIP-BATCH
    # --------------------------------------------------

    per_image_reps = (
        visual_rep.perImageRepresentations
    )

    if not per_image_reps:

        raise RuntimeError(
            "No successful visual representations."
        )

    inference_modes = set()

    for rep in per_image_reps:

        metadata = (
            rep.processingMetadata
            or {}
        )

        inference_modes.add(
            metadata.get(
                "inferenceMode"
            )
        )

    allowed_modes = {
        "CLIP-Batch",
        "CLIP"
    }

    if not inference_modes.issubset(
        allowed_modes
    ):

        raise RuntimeError(
            "Non-CLIP inference detected: "
            f"{inference_modes}"
        )

    # --------------------------------------------------
    # P5 — INSIGHT AGGREGATION
    # --------------------------------------------------

    profile = (
        insight_service.aggregate(
            visual=visual_rep,
            text=text_rep,
            attribute=attribute_rep
        )
    )

    # --------------------------------------------------
    # P6 — MULTIMODAL FUSION
    #
    # IMPORTANT:
    # Actual API requires profile=profile.
    # --------------------------------------------------

    fused = (
        fusion_service.fuse(
            profile=profile,
            visual=visual_rep,
            text=text_rep,
            attribute=attribute_rep
        )
    )

    # --------------------------------------------------
    # 662D EMBEDDING
    # --------------------------------------------------

    embedding = np.asarray(
        get_fused_embedding(
            fused
        ),
        dtype=np.float32
    )

    # --------------------------------------------------
    # VALIDATION
    # --------------------------------------------------

    if embedding.shape != (
        662,
    ):

        raise ValueError(
            f"Expected (662,), "
            f"got {embedding.shape}"
        )

    if not np.isfinite(
        embedding
    ).all():

        raise ValueError(
            "Embedding contains NaN/Inf."
        )

    norm = float(
        np.linalg.norm(
            embedding
        )
    )

    if not np.isfinite(norm):

        raise ValueError(
            "Embedding norm is invalid."
        )

    if norm == 0:

        raise ValueError(
            "Embedding is zero vector."
        )

    # --------------------------------------------------
    # CONFIDENCE
    # --------------------------------------------------

    confidence = get_value(
        fused,
        "confidence",
        None
    )

    if confidence is not None:

        try:
            confidence = float(
                confidence
            )
        except Exception:
            confidence = None

    # --------------------------------------------------
    # PROVENANCE
    # --------------------------------------------------

    provenance = get_value(
        fused,
        "provenance",
        None
    )

    # --------------------------------------------------
    # RESULT
    # --------------------------------------------------

    return {

        "productId":
            product_id,

        "unifiedEmbedding":
            embedding.tolist(),

        "embeddingDimension":
            662,

        "l2Norm":
            norm,

        "confidence":
            confidence,

        "provenance":
            provenance,

        "inferenceMode":
            "CLIP-Batch",

        "encoderVersions":
            profile.encoderVersions,

        "successfulImageCount":
            visual_rep.successfulImageCount,

        "failedImageCount":
            visual_rep.failedImageCount,

        "rowIndex":
            int(row.name),

        "generatedAt":
            time.time()
    }


# ======================================================================
# START PRODUCTION RUN
# ======================================================================

print("\n" + "=" * 70)
print("STARTING FULL P8 RUN")
print("=" * 70)

run_start = time.perf_counter()

last_checkpoint_count = (
    len(successful_ids)
    + len(failed_ids)
)

processed_since_start = 0

for row_idx, row in df.iterrows():

    product_id = str(
        row["sku"]
    )

    # --------------------------------------------------
    # RESUME / SKIP
    # --------------------------------------------------

    if product_id in successful_ids:
        continue

    if product_id in failed_ids:
        continue

    product_start = time.perf_counter()

    try:

        result = process_product(
            row
        )

        append_jsonl(
            RESULTS_FILE,
            result
        )

        successful_ids.add(
            product_id
        )

    except Exception as exc:

        failed_ids.add(
            product_id
        )

        append_jsonl(
            ERRORS_FILE,
            {
                "rowIndex":
                    int(row_idx),

                "productId":
                    product_id,

                "error":
                    repr(exc),

                "timestamp":
                    time.time()
            }
        )

    processed_since_start += 1

    # --------------------------------------------------
    # PROGRESS
    # --------------------------------------------------

    completed = (
        len(successful_ids)
        + len(failed_ids)
    )

    elapsed = (
        time.perf_counter()
        - run_start
    )

    rate = (
        processed_since_start / elapsed
        if elapsed > 0
        else 0
    )

    remaining_count = (
        total_products
        - completed
    )

    eta_seconds = (
        remaining_count / rate
        if rate > 0
        else 0
    )

    if (
        processed_since_start % 10 == 0
        or completed == total_products
    ):

        print(
            f"Processed {completed}/{total_products}"
            f" | Success: {len(successful_ids)}"
            f" | Failed: {len(failed_ids)}"
            f" | Rate: {rate:.2f} prod/s"
            f" | ETA: {eta_seconds / 60:.1f} min"
        )

    # --------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------

    checkpoint_count = (
        len(successful_ids)
        + len(failed_ids)
    )

    if (
        checkpoint_count
        - last_checkpoint_count
        >= CHECKPOINT_EVERY
    ):

        save_checkpoint()

        last_checkpoint_count = (
            checkpoint_count
        )

        print(
            f"💾 Checkpoint saved "
            f"at {checkpoint_count} products"
        )


# ======================================================================
# FINAL CHECKPOINT
# ======================================================================

save_checkpoint()

total_time = (
    time.perf_counter()
    - run_start
)

# ======================================================================
# FINAL OUTPUT VALIDATION
# ======================================================================

print("\n" + "=" * 70)
print("P8 FINAL VALIDATION")
print("=" * 70)

embedding_count = 0
invalid_count = 0

seen_ids = set()
duplicate_ids = set()

norms = []
confidences = []

fallback_count = 0

if os.path.exists(
    RESULTS_FILE
):

    with open(
        RESULTS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            record = json.loads(
                line
            )

            pid = str(
                record["productId"]
            )

            if pid in seen_ids:

                duplicate_ids.add(
                    pid
                )

            seen_ids.add(
                pid
            )

            embedding = np.asarray(
                record[
                    "unifiedEmbedding"
                ],
                dtype=np.float32
            )

            norm = float(
                np.linalg.norm(
                    embedding
                )
            )

            if (
                embedding.shape != (662,)
                or not np.isfinite(
                    embedding
                ).all()
                or norm == 0
            ):

                invalid_count += 1

            else:

                embedding_count += 1

                norms.append(
                    norm
                )

            if (
                record.get(
                    "inferenceMode"
                )
                not in (
                    "CLIP-Batch",
                    "CLIP"
                )
            ):

                fallback_count += 1

            confidence = (
                record.get(
                    "confidence"
                )
            )

            if confidence is not None:

                try:
                    confidences.append(
                        float(confidence)
                    )
                except Exception:
                    pass


# ======================================================================
# FINAL NUMBERS
# ======================================================================

accounted = (
    len(successful_ids)
    + len(failed_ids)
)

print(
    "\nTotal catalog:",
    total_products
)

print(
    "Successful:",
    embedding_count
)

print(
    "Failed:",
    len(failed_ids)
)

print(
    "Accounted:",
    accounted
)

print(
    "Success rate:",
    f"{embedding_count / total_products * 100:.2f}%"
)

print(
    "CLIP fallback records:",
    fallback_count
)

print(
    "Invalid embeddings:",
    invalid_count
)

print(
    "Duplicate product IDs:",
    len(duplicate_ids)
)

print(
    "Embedding dimension:",
    662
)

if norms:

    print(
        "Mean L2 norm:",
        f"{np.mean(norms):.8f}"
    )

    print(
        "Min L2 norm:",
        f"{np.min(norms):.8f}"
    )

    print(
        "Max L2 norm:",
        f"{np.max(norms):.8f}"
    )

if confidences:

    print(
        "Mean confidence:",
        f"{np.mean(confidences):.4f}"
    )

print(
    "Total runtime:",
    f"{total_time / 60:.2f} minutes"
)

if total_time > 0:

    print(
        "Throughput:",
        f"{(embedding_count + len(failed_ids)) / total_time:.3f}",
        "products/sec"
    )

print(
    "\nResults:",
    RESULTS_FILE
)

print(
    "Errors:",
    ERRORS_FILE
)

print(
    "Checkpoint:",
    CHECKPOINT_FILE
)

# ======================================================================
# FINAL STATUS
# ======================================================================

print("\n" + "=" * 70)

if (
    accounted == total_products
    and embedding_count == total_products
    and len(failed_ids) == 0
    and invalid_count == 0
    and len(duplicate_ids) == 0
    and fallback_count == 0
):

    print(
        "✅ P8 FULL CATALOG GENERATION COMPLETE"
    )

else:

    print(
        "⚠️ P8 FINISHED WITH VALIDATION ISSUES"
    )

print("=" * 70)

ZYRA V1 — P8 FULL CATALOG PRODUCTION RUN

Device: mps
ML encoding: True
Batch inference: True
CLIP model: <class 'transformers.models.clip.modeling_clip.CLIPModel'>
Actual model device: mps:0
Model: openai/clip-vit-base-patch32
✅ REAL CLIP + M5 MPS READY

No checkpoint found.

P8 PRODUCTION STATUS
Total products: 12491
Already successful: 0
Already failed: 0
Remaining: 12491

STARTING FULL P8 RUN


Loading weights: 100%|████████████████████████████████████████████████████████████| 197/197 [00:00<00:00, 55694.11it/s]
[transformers] CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight 

Processed 10/12491 | Success: 10 | Failed: 0 | Rate: 0.92 prod/s | ETA: 227.1 min
Processed 20/12491 | Success: 20 | Failed: 0 | Rate: 1.11 prod/s | ETA: 186.9 min
💾 Checkpoint saved at 25 products
Processed 30/12491 | Success: 30 | Failed: 0 | Rate: 1.14 prod/s | ETA: 181.8 min
Processed 40/12491 | Success: 40 | Failed: 0 | Rate: 1.23 prod/s | ETA: 169.2 min
Processed 50/12491 | Success: 50 | Failed: 0 | Rate: 1.24 prod/s | ETA: 166.6 min
💾 Checkpoint saved at 50 products
Processed 60/12491 | Success: 60 | Failed: 0 | Rate: 1.30 prod/s | ETA: 159.8 min
Processed 70/12491 | Success: 70 | Failed: 0 | Rate: 1.33 prod/s | ETA: 156.1 min
💾 Checkpoint saved at 75 products
Processed 80/12491 | Success: 80 | Failed: 0 | Rate: 1.37 prod/s | ETA: 151.0 min
Processed 90/12491 | Success: 90 | Failed: 0 | Rate: 1.35 prod/s | ETA: 152.8 min
Processed 100/12491 | Success: 100 | Failed: 0 | Rate: 1.36 prod/s | ETA: 151.5 min
💾 Checkpoint saved at 100 products
Processed 110/12491 | Success: 110 | Fail